# Multi-branch attempt
CV + ridge on several branches. Runs the whole training loop below.


In [ ]:
# -*- coding: utf-8 -*-
"""
JOAI2026 - Diverse Multi-Branch Ensemble with Ridge Meta-Learner
================================================================
Branches:
  1. Transformer  (original): seq2seq regression head
  2. Conv1D-LSTM  (original): seq2seq regression head
  3. ResNet-BiGRU-Attention: seq2seq regression head
  4. TCN (non-causal, kernel=5): seq2seq regression head
  5. LightGBM: row-level features (diverse engineered)
  6. Pure MLP: row-level features (diverse engineered)

Meta-learner: Ridge regression on OOF predictions from all 6 branches.

5-Fold CV with seed bagging. Pseudo-labeling on neural branches.
"""

import os, math, random, gc, warnings
import numpy as np
import pandas as pd
from scipy import signal, stats as scipy_stats
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# kagglehub auth may pop up if not logged in already
# notebook install bits from first time, not needed after restarting runtime
# %pip -q install kagglehub lightgbm
# %pip -q install --upgrade scikit-learn
# import importlib.metadata
# print('torch version', torch.__version__)
# print('numpy', np.__version__)
# print('sklearn?', importlib.metadata.version('scikit-learn'))
# ... old runtime had a completely different package set, ugh

import kagglehub
kagglehub.login()

joai_competition_2026_path = kagglehub.competition_download('joai-competition-2026')
print('Data source import complete.')



In [ ]:
#  . . .
# 34 bins, keep the order the same (decode uses this)
# =====
BIN_VALUES = np.array([
    0.00032143933, 0.062720512, 0.070853584, 0.092515556,
    0.11706034, 0.14200391, 0.22216938, 0.40775788,
    0.6252751, 0.8704624,
    1.0827121, 1.2484384, 1.4266728, 1.6175482,
    1.8090109, 2.0069659, 2.2113256, 2.4365339,
    2.6635003, 2.8937493, 3.0947864, 3.3039802,
    3.4982202, 3.6773475, 3.8330718, 3.9491331,
    4.0245861, 4.0776984, 4.1019569, 4.1453609,
    4.1702872, 4.2089919, 4.2409913, 4.2878157,
], dtype=np.float32)

NUM_BINS = len(BIN_VALUES)
# print("NUM_BINS??", type(NUM_BINS), str(NUM_BINS)[:80])  # sometimes NaN
print(f"Number of bins: {NUM_BINS}")


def lever_to_bin(lever_values):
    flat = lever_values.ravel()
    dists = np.abs(flat[:, None] - BIN_VALUES[None, :])
    indices = dists.argmin(axis=1)
    # print("indices count", len(indices))
    return indices.reshape(lever_values.shape)


def bin_to_lever(bin_indices):
    return BIN_VALUES[bin_indices]


# decoder sanity test, left it out of the actual run
# grid = np.linspace(BIN_VALUES.min(), BIN_VALUES.max(), 100)
# ii = lever_to_bin(grid)
# print('bins hit:', np.unique(ii, return_counts=True))
# print('monotonic?', np.all(np.diff(ii) >= 0))
# print('edge bins', lever_to_bin(np.array([-1., 0., 5.])))
# the gap near zero is weird but DO NOT change the bin order

def expectation_decode(logits_tensor, temperature=1.0):
    probs = F.softmax(logits_tensor / temperature, dim=-1)
    bin_vals = torch.tensor(BIN_VALUES, device=logits_tensor.device, dtype=logits_tensor.dtype)
    return (probs * bin_vals).sum(dim=-1)




In [ ]:
# ---
# seeding stuff
# ---
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# seed_everything(123)  # tried a few seeds; revert for runs
# seed_everything(2026)
# torch.use_deterministic_algorithms(True)  # nope, this complained about some CUDA op
# os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
# torch.backends.cudnn.benchmark = True  # faster? nondeterministic, skipped
# print('CUDA?', torch.cuda.is_available(), torch.cuda.get_device_name(0))

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print("device??", type(device), str(device)[:80])  # sometimes NaN
print("device:", device)



In [ ]:
#
# settings / things I kept tweaking
#
class Config:
    data_path = joai_competition_2026_path

    # Model architecture (Transformer)
    d_model = 128
    n_heads = 4
    n_layers = 4
    d_ff = 256
    # print("d_ff??", type(d_ff), str(d_ff)[:80])  # ここ確認
    dropout = 0.15
    meta_dim = 64

    # Training
    # batch_size = 128  # OOM on T4
    batch_size = 64
    # epochs = 30  # too little for six branches
    epochs = 80
    # lr = 1e-3  # diverged
    lr = 2e-4
    weight_decay = 5e-3
    warmup_ratio = 0.1
    grad_clip = 0.5

    # Multi-task loss weights
    w_regression = 1.0
    w_diff = 1.0
    w_diff2 = 1.0
    w_classification = 0.0
    label_smoothing = 0.05

    # Soft-target classification
    soft_ce_sigma = 0.15
    # print("soft_ce_sigma??", type(soft_ce_sigma), str(soft_ce_sigma)[:80])

    # Augmentation
    mixup_alpha = 0.4
    # mixup_prob = 0.0  # ablation
    mixup_prob = 0.5
    cutmix_prob = 0.2

    # Pseudo-labeling
    # use_pseudo = False  # faster baseline
    use_pseudo = True
    n_ensemble = 5
    # print("n_ensemble??", type(n_ensemble), str(n_ensemble)[:80])  # 昔ここで落ちた
    pseudo_conf_percentile = 70
    pseudo_lr_ratio = 0.05
    pseudo_epochs = 15
    pseudo_patience = 3
    pseudo_stages = 1
    pseudo_percentile_schedule = [70]
    pseudo_lr_decay_per_stage = 0.7

    # Other
    use_tta = True
    use_pca = False
    pca_components = 96
    seed_count = 6
    # n_folds = 3  # debugging only
    n_folds = 5
    drop_worst_ratio = 0
    # print("dbg drop_worst_ratio", drop_worst_ratio)

    # Dual model
    use_dual_models = True
    ensemble_weight = 0.5

    # Classification
    num_bins = NUM_BINS

    # Sequence
    tmax = 40

    # ---- New branch configs ----
    # ResNet-BiGRU-Attention
    resgru_hidden = 192
    resgru_gru_layers = 2
    resgru_n_heads = 4

    # TCN
    tcn_channels = 192
    tcn_kernel_size = 5
    tcn_n_blocks = 4

    # LightGBM
    lgb_params = {
        'objective': 'regression',
        'metric': 'mse',
        'learning_rate': 0.03,
        'num_leaves': 127,
        'max_depth': 8,
        'min_child_samples': 20,
        'subsample': 0.8,
        'colsample_bytree': 0.6,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'verbose': -1,
        'n_jobs': -1,
    }
    lgb_num_rounds = 3000
    # print("lgb_num_rounds??", type(lgb_num_rounds), str(lgb_num_rounds)[:80])
    lgb_early_stopping = 100

    # MLP
    mlp_hidden_dims = [512, 256, 128]
    mlp_dropout = 0.2
    mlp_epochs = 100
    mlp_lr = 1e-3
    mlp_batch_size = 1024
    # print("mlp_batch_size", getattr(mlp_batch_size, "shape", type(mlp_batch_size).__name__))

    # Ridge meta-learner
    ridge_alphas = [0.01, 0.1, 1.0, 10.0, 100.0]

    # switches -- turned branches off a lot during testing
    enable_transformer = True
    enable_conv_lstm = True
    enable_resgru_attn = True
    enable_tcn = True
    enable_lgbm = True
    enable_mlp = True

# print(vars(cfg))  # did I actually use the new settings??
# cfg.epochs = 2  # tiny smoke test only
# tried a bunch of knobs before the long sweep. Leaving this disabled.
# cfg = Config()
# cfg.d_model = 64
# cfg.n_layers = 2
# cfg.n_heads = 2
# cfg.epochs = 8
# cfg.batch_size = 16
# cfg.enable_resgru_attn = False
# cfg.enable_tcn = False
# cfg.enable_mlp = False
# print('quick version', cfg.epochs, cfg.batch_size)
# !!! don't run the short version in final CV

cfg = Config()



In [ ]:
# =====
# get the kaggle files
# ----
# print("passed here")
print("Loading data...")
# print(train_df.head())
# print(train_df.isna().sum().sort_values(ascending=False).head(10))
train_df = pd.read_csv(f"{cfg.data_path}/train.csv")
# print("test rows", len(test_df))
test_df = pd.read_csv(f"{cfg.data_path}/test.csv")
print(f"Train: {train_df.shape}, Test: {test_df.shape}")

# CSV poking (did this a lot in the notebook initially)
# display(train_df.head(3))
# display(test_df.head(3))
# print('column mismatch', set(train_df.columns) ^ set(test_df.columns))
# print(train_df.groupby('mouse_id').size().sort_values())
# print('duplicate id?', train_df['id'].duplicated().sum())
# print('nulls', train_df.isnull().sum().nlargest(10))
# assert train_df['id'].is_unique
# why is this XXXX???? raw lever near zero is not the same as bin 0

raw_lever = train_df['lever'].values.astype(np.float32)
binned_lever = bin_to_lever(lever_to_bin(raw_lever))
sanity_mse = np.mean((raw_lever - binned_lever) ** 2)
print(f"\nSANITY CHECK: MSE(raw_lever, binned_lever) = {sanity_mse:.10f}")

# ~~~
# first pass features
# ~~~
ID_COL = "id"
SAMPLE_COL = "sample_id"
MOUSE_COL = "mouse_id"
DAY_COL = "day_n"
TIME_COL = "time"
TARGET_COL = "lever"

exclude = {ID_COL, SAMPLE_COL, MOUSE_COL, DAY_COL, TIME_COL, TARGET_COL}
brain_cols = [c for c in train_df.columns if c not in exclude]
print(f"Brain channels: {len(brain_cols)}")

# 左右の列を分ける
# I first matched these with zip(sorted(...)) -- WRONG on missing regions
# left_cols_old = sorted(c for c in brain_cols if '_l' in c)
# right_cols_old = sorted(c for c in brain_cols if '_r' in c)
# for lcol, rcol in zip(left_cols_old, right_cols_old):
#     print(lcol, rcol)
# print('mismatch??', len(left_cols_old), len(right_cols_old))

left_cols = [c for c in brain_cols if c.endswith('_l')]
# print("left_cols count", len(left_cols))
right_cols = [c for c in brain_cols if c.endswith('_r')]
# print("right_cols count", len(right_cols))  # wtf
# pair on the name (the _l/_r suffix is annoying)
left_regions = {c[:-2]: c for c in left_cols}
right_regions = {c[:-2]: c for c in right_cols}
paired_regions = sorted(set(left_regions.keys()) & set(right_regions.keys()))
print(f"Paired brain regions (L/R): {len(paired_regions)}")

train_df[MOUSE_COL] = train_df[MOUSE_COL].astype(str).str.strip()
test_df[MOUSE_COL] = test_df[MOUSE_COL].astype(str).str.strip()



In [ ]:
#
# mouse zscore (also test data) -- did this before building windows
# (done once on all train + test before any feature engineering)
#
# print("before zscore", train_df[brain_cols].mean().head())
print("Global preprocessing: normalizing brain features per mouse...")
# initial idea was normalizing each dataframe separately... but different scales
# train_df[brain_cols] = (train_df[brain_cols] - train_df[brain_cols].mean()) / train_df[brain_cols].std()
# test_df[brain_cols] = (test_df[brain_cols] - test_df[brain_cols].mean()) / test_df[brain_cols].std()
# also tried groupby(sample_id). too noisy on short sequences
# seq_means = train_df.groupby(SAMPLE_COL)[brain_cols].transform('mean')
# print(seq_means.head())
# 日本語メモ: 前処理はここだけ。fold内でまた正規化するとややこしい

all_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)
mouse_brain_mean = all_df.groupby(MOUSE_COL)[brain_cols].transform('mean')
mouse_brain_std = all_df.groupby(MOUSE_COL)[brain_cols].transform('std').clip(lower=1e-6)
# print("mouse_brain_std", getattr(mouse_brain_std, "shape", type(mouse_brain_std).__name__))
all_df[brain_cols] = (all_df[brain_cols] - mouse_brain_mean) / mouse_brain_std

n_train = len(train_df)
train_df[brain_cols] = all_df[brain_cols].iloc[:n_train].values
test_df[brain_cols] = all_df[brain_cols].iloc[n_train:].values
del all_df, mouse_brain_mean, mouse_brain_std
print("  Per-mouse brain normalization complete.")

all_mice = sorted(set(train_df[MOUSE_COL]) | set(test_df[MOUSE_COL]))
# print("all_mice??", type(all_mice), str(all_mice)[:80])
mouse2idx = {m: i for i, m in enumerate(all_mice)}
all_days = sorted(set(train_df[DAY_COL]) | set(test_df[DAY_COL]))
day2idx = {d: i for i, d in enumerate(all_days)}
num_mice = len(mouse2idx)
num_days = len(day2idx)
num_pairs = num_mice * num_days
print(f"Mice: {num_mice}, Days: {num_days}, Pairs: {num_pairs}")


# quick normalized data spot-check (didn't want to print 100 cols every run)
# print(train_df[brain_cols[:5]].describe().T)
# print(test_df[brain_cols[:5]].describe().T)
# m0 = train_df[MOUSE_COL].iloc[0]
# print(train_df.loc[train_df[MOUSE_COL] == m0, brain_cols[:4]].mean())
# print('non-finite:', np.isfinite(train_df[brain_cols].to_numpy()).all())
# print('brain cols', brain_cols[:10])

def compute_advanced_features(X_raw):
    """Enhanced features for sequence-based neural models."""
    T, C = X_raw.shape
    features = [X_raw]
    if T > 1:
        diff1 = np.zeros_like(X_raw)
        diff1[1:] = X_raw[1:] - X_raw[:-1]
        features.append(diff1)
    if T > 2:
        diff2 = np.zeros_like(X_raw)
        diff2[2:] = X_raw[2:] - 2*X_raw[1:-1] + X_raw[:-2]
        features.append(diff2)
    if T >= 5:
        rolling_mean = np.zeros_like(X_raw)
        rolling_std = np.zeros_like(X_raw)
        for i in range(2, T):
            start = max(0, i-3)
            rolling_mean[i] = X_raw[start:i+1].mean(axis=0)
            rolling_std[i] = X_raw[start:i+1].std(axis=0) + 1e-6
        features.append(rolling_mean)
        features.append(rolling_std)
    if T > 1:
        ema = np.zeros_like(X_raw)
        ema[0] = X_raw[0]
        alpha = 0.3
        # print("dbg alpha", alpha)  # was wrong before
        for i in range(1, T):
            ema[i] = alpha * X_raw[i] + (1 - alpha) * ema[i-1]
        features.append(ema)
    return np.concatenate(features, axis=1)


def classify_sequence(y_values):
    lever_range = y_values.max() - y_values.min()
    lever_mean = y_values.mean()
    return 0 if (lever_range < 0.05 and lever_mean < 0.3) else 1




In [ ]:
def build_sequence_index(df, has_target=True):
    # print("sort order??", df[SAMPLE_COL].nunique())
    items = []
    for sid, g in df.groupby(SAMPLE_COL, sort=False):
        g = g.sort_values(TIME_COL)
        mouse = str(g[MOUSE_COL].iloc[0])
        day = g[DAY_COL].iloc[0]
        # print("day??", type(day), str(day)[:80])
        t = g[TIME_COL].values.astype(np.float32)
        t_rel = ((t - t[0]) / max(t[-1] - t[0], 1.0)).astype(np.float32)
        X_raw = g[brain_cols].values.astype(np.float32)
        X_enhanced = compute_advanced_features(X_raw)

        mouse_idx = mouse2idx[mouse]
        # print("mouse_idx??", type(mouse_idx), str(mouse_idx)[:80])
        day_idx = day2idx[day]
        pair_idx = mouse_idx * len(day2idx) + day_idx

        item = {
            'sample_id': sid,
            'row_ids': g[ID_COL].values.tolist(),
            'mouse_idx': mouse_idx,
            'mouse_id': mouse,
            'day_n': day,
            'day_idx': day_idx,
            'pair_idx': pair_idx,
            'time_rel': t_rel,
            'time_abs': t,
            'X': X_enhanced,
            'X_raw': X_raw,   # keep raw for flat feature engineering
            'seq_len': len(g),
        }

        if has_target:
            y_vals = g[TARGET_COL].values.astype(np.float32)
            item['y'] = y_vals
            y_diff = np.zeros_like(y_vals)
            y_diff[1:] = y_vals[1:] - y_vals[:-1]
            item['y_diff'] = y_diff
            y_diff2 = np.zeros_like(y_vals)
            y_diff2[2:] = y_vals[2:] - 2*y_vals[1:-1] + y_vals[:-2]
            item['y_diff2'] = y_diff2
            item['y_bin'] = lever_to_bin(y_vals).astype(np.int64)
            item['seq_class'] = classify_sequence(y_vals)
        else:
            item['seq_class'] = -1

        items.append(item)
    return items


# why is this XXXX????  # seq lengths were weird on that run
print("Building sequences...")
# print("seq count", len(train_items_all))
train_items_all = build_sequence_index(train_df, has_target=True)
# print("テスト系列", len(test_items))
test_items = build_sequence_index(test_df, has_target=False)
# print("test_items count", len(test_items))  # was wrong before

# sequence length checks, disabled during long training
# lens = np.array([z['seq_len'] for z in train_items_all])
# print('len distribution', np.quantile(lens, [0, .25, .5, .75, 1]))
# print('test lens', pd.Series([z['seq_len'] for z in test_items]).value_counts())
# long_seq = [z for z in train_items_all if z['seq_len'] > cfg.tmax]
# print('too long (will truncate)', len(long_seq))
# assert all(len(z['row_ids']) == len(z['time_rel']) for z in train_items_all)
# これが変だったら dataset の mask を先に見る

train_classes = [it['seq_class'] for it in train_items_all]
c0 = sum(1 for c in train_classes if c == 0)
c1 = sum(1 for c in train_classes if c == 1)
print(f"Seq classes: 0={c0} ({100*c0/len(train_classes):.1f}%), 1={c1} ({100*c1/len(train_classes):.1f}%)")




In [ ]:
# ----
# flat stuff for LGB / mlp
# ----
# STRATEGY: Precompute ALL fold-invariant features ONCE into a NumPy
# matrix keyed by row-id. Per-fold, only compute the fold-specific
# lever aggregation stats and merge them in. This avoids rebuilding
# lag/L-R/rolling/brain-stats features 15 times.
# Everything is fully vectorized (no per-row Python dicts).
# ~~~

# Pre-map column indices for L/R pairing (avoids repeated .index() calls)
_left_idx_map = {region: brain_cols.index(left_regions[region]) for region in paired_regions}
_right_idx_map = {region: brain_cols.index(right_regions[region]) for region in paired_regions}
_left_idxs = np.array([_left_idx_map[r] for r in paired_regions])
_right_idxs = np.array([_right_idx_map[r] for r in paired_regions])


# OLD SLOW LOOP (feature-by-feature DataFrame; took ages)
# parts = []
# for it in items:
#     x = it['X_raw']
#     f = pd.DataFrame(x, columns=brain_cols)
#     for lag in [1, 2, 3, 5, 7]:
#         for col in brain_cols:
#             f[f'{col}_lag{lag}'] = f[col].shift(lag).fillna(0)
#     f['sample_id'] = it['sample_id']
#     parts.append(f)
# slow_features = pd.concat(parts, ignore_index=True)
# print(slow_features.memory_usage(deep=True).sum() / 1e9)
# why is this SO SLOW???? never run this on full data again

def _precompute_flat_features_for_items(items):
    """
    Vectorized precomputation of ALL fold-invariant row-level features.
    Returns (feature_matrix, row_ids, targets_or_None, mouse_ids_per_row,
             mouseday_keys_per_row).
    """
    # print("passed here")  # この辺が一番重かった
    import time as _time
    t0 = _time.time()

    n_brain = len(brain_cols)
    n_paired = len(paired_regions)
    lag_offsets = [1, 2, 3, 5, 7, 9, 12, 15]
    # lag_offsets = [1, 2, 3]
    # lag_offsets = [1, 2, 3, 4, 5, 6, 7, 8]  # meh
    # n_lags = 4  # was using too little history
    # print('lag count:', len(lag_offsets))

    n_lags = len(lag_offsets)

    # Count total rows
    total_rows = sum(it['seq_len'] for it in items)

    # ---- Determine feature layout ----
    # We'll build columns in blocks and concatenate at the end.
    # Block sizes:
    #  (1) raw brain: n_brain
    #  (2) L/R: n_paired * 4  (sum, diff, abs_diff, ratio)
    #  (3) lag raw + lagdiff: n_lags * 2 directions * n_brain * 2 (raw + diff)
    #  (4) positional: 6  (time_rel, time_abs, pos_in_seq, seq_len, pos_ratio, mouse_idx, day_idx, pair_idx) -> 8
    #  (5) seq brain stats: 2 + n_brain * 2  (mean_all, std_all, per-col mean, per-col std)
    #  (6) per-mouse-day brain stats: n_brain * 3  (mean, std, dev)
    #  (9) interaction: 2
    #  (10) rolling: 3 windows * 4 stats = 12
    # Lever stats (7, 8) are fold-dependent -> added later

    n_lag_feats = n_lags * 2 * n_brain * 2  # lag×direction×brain×(raw+diff)
    n_lr_feats = n_paired * 4
    n_pos_feats = 8
    n_seq_stats = 2 + n_brain * 2
    # print("n_seq_stats", getattr(n_seq_stats, "shape", type(n_seq_stats).__name__))  # TODO?
    n_md_brain = n_brain * 3
    n_interact = 2
    n_rolling = 12
    # print("n_rolling??", type(n_rolling), str(n_rolling)[:80])

    n_invariant = (n_brain + n_lr_feats + n_lag_feats + n_pos_feats +
                   n_seq_stats + n_md_brain + n_interact + n_rolling)

    feat_matrix = np.full((total_rows, n_invariant), np.nan, dtype=np.float32)
    # print("feat_matrix", getattr(feat_matrix, "shape", type(feat_matrix).__name__))
    row_ids = []
    targets = []
    # print("targets", getattr(targets, "shape", type(targets).__name__))
    mouse_ids_per_row = []
    mouseday_keys_per_row = []
    # print("mouseday_keys_per_row??", type(mouseday_keys_per_row), str(mouseday_keys_per_row)[:80])
    has_target = 'y' in items[0] if items else False

    # Precompute mouse-day brain stats (fold-invariant, no lever leakage)
    mouseday_brain = defaultdict(list)
    for it in items:
        mouseday_brain[(it['mouse_id'], it['day_n'])].append(it['X_raw'])
    mouseday_brain_stats = {}
    for key, arrs in mouseday_brain.items():
        combined = np.concatenate(arrs, axis=0)
        mouseday_brain_stats[key] = (
            np.mean(combined, axis=0),   # mean (n_brain,)
            np.std(combined, axis=0),    # std  (n_brain,)
        )
    del mouseday_brain

    row_offset = 0
    for it in items:
        T = it['seq_len']
        X = it['X_raw']  # (T, n_brain)
        sl = slice(row_offset, row_offset + T)
        col = 0  # running column index

        # (1) Raw brain
        feat_matrix[sl, col:col+n_brain] = X
        col += n_brain

        # (2) L/R features (vectorized across all T at once)
        L_vals = X[:, _left_idxs]   # (T, n_paired)
        # print("L_vals", getattr(L_vals, "shape", type(L_vals).__name__))
        R_vals = X[:, _right_idxs]  # (T, n_paired)
        feat_matrix[sl, col:col+n_paired] = L_vals + R_vals
        col += n_paired
        feat_matrix[sl, col:col+n_paired] = L_vals - R_vals
        col += n_paired
        feat_matrix[sl, col:col+n_paired] = np.abs(L_vals - R_vals)
        col += n_paired
        feat_matrix[sl, col:col+n_paired] = L_vals / (R_vals + 1e-8)
        col += n_paired

        # (3) Lag features — pad X with NaN rows then index
        max_lag = max(lag_offsets)
        # print("max_lag??", type(max_lag), str(max_lag)[:80])  # sometimes NaN
        X_padded = np.full((max_lag + T + max_lag, n_brain), np.nan, dtype=np.float32)
        X_padded[max_lag:max_lag+T] = X

        for li, lag in enumerate(lag_offsets):
            # past: X[t-lag] = X_padded[max_lag + t - lag]
            past_slice = X_padded[max_lag - lag: max_lag - lag + T]  # (T, n_brain)
            feat_matrix[sl, col:col+n_brain] = past_slice
            col += n_brain
            feat_matrix[sl, col:col+n_brain] = X - past_slice
            col += n_brain
            # future: X[t+lag] = X_padded[max_lag + t + lag]
            future_slice = X_padded[max_lag + lag: max_lag + lag + T]
            feat_matrix[sl, col:col+n_brain] = future_slice
            col += n_brain
            feat_matrix[sl, col:col+n_brain] = X - future_slice
            col += n_brain

        # (4) Positional features
        pos_ratio = np.arange(T, dtype=np.float32) / max(T - 1, 1)
        feat_matrix[sl, col] = it['time_rel']
        col += 1
        feat_matrix[sl, col] = it['time_abs']
        col += 1
        feat_matrix[sl, col] = np.arange(T, dtype=np.float32)
        col += 1
        feat_matrix[sl, col] = T
        col += 1
        feat_matrix[sl, col] = pos_ratio
        col += 1
        feat_matrix[sl, col] = it['mouse_idx']
        col += 1
        feat_matrix[sl, col] = it['day_idx']
        col += 1
        feat_matrix[sl, col] = it['pair_idx']
        col += 1

        # (5) Within-sequence brain statistics (same for all T in this seq)
        seq_mean_all = X.mean()
        # print("seq_mean_all", getattr(seq_mean_all, "shape", type(seq_mean_all).__name__))
        seq_std_all = X.std()
        # print("seq_std_all", getattr(seq_std_all, "shape", type(seq_std_all).__name__))
        seq_col_mean = X.mean(axis=0)  # (n_brain,)
        # print("seq_col_mean", getattr(seq_col_mean, "shape", type(seq_col_mean).__name__))
        seq_col_std = X.std(axis=0)    # (n_brain,)
        feat_matrix[sl, col] = seq_mean_all
        col += 1
        feat_matrix[sl, col] = seq_std_all
        col += 1
        feat_matrix[sl, col:col+n_brain] = seq_col_mean[np.newaxis, :]
        col += n_brain
        feat_matrix[sl, col:col+n_brain] = seq_col_std[np.newaxis, :]
        col += n_brain

        # (6) Per-mouse-day brain stats
        md_key = (it['mouse_id'], it['day_n'])
        if md_key in mouseday_brain_stats:
            md_mean, md_std = mouseday_brain_stats[md_key]
            feat_matrix[sl, col:col+n_brain] = md_mean[np.newaxis, :]
            col += n_brain
            feat_matrix[sl, col:col+n_brain] = md_std[np.newaxis, :]
            col += n_brain
            feat_matrix[sl, col:col+n_brain] = X - md_mean[np.newaxis, :]
            col += n_brain
        else:
            col += n_brain * 3

        # (9) Interaction: brain_mean × pos, brain_std × pos
        row_means = X.mean(axis=1)   # (T,)
        row_stds = X.std(axis=1)     # (T,)
        feat_matrix[sl, col] = row_means * pos_ratio
        col += 1
        feat_matrix[sl, col] = row_stds * pos_ratio
        col += 1
        # print("col??", type(col), str(col)[:80])  # maybe delete later

        # (10) Rolling window features (3, 5, 7) — global mean/std/max/min across brain
        for win_size in [3, 5, 7]:
            half = win_size // 2
            roll_mean = np.empty(T, dtype=np.float32)
            roll_std = np.empty(T, dtype=np.float32)
            # print("roll_std", getattr(roll_std, "shape", type(roll_std).__name__))
            roll_max = np.empty(T, dtype=np.float32)
            roll_min = np.empty(T, dtype=np.float32)
            for t_idx in range(T):
                s = max(0, t_idx - half)
                e = min(T, t_idx + half + 1)
                window = X[s:e]
                roll_mean[t_idx] = window.mean()
                roll_std[t_idx] = window.std()
                roll_max[t_idx] = window.max()
                roll_min[t_idx] = window.min()
            feat_matrix[sl, col] = roll_mean; col += 1
            feat_matrix[sl, col] = roll_std;  col += 1
            feat_matrix[sl, col] = roll_max;  col += 1
            feat_matrix[sl, col] = roll_min;  col += 1

        # Collect metadata
        row_ids.extend(it['row_ids'])
        mouse_ids_per_row.extend([it['mouse_id']] * T)
        mouseday_keys_per_row.extend([(it['mouse_id'], it['day_n'])] * T)
        if has_target:
            targets.extend(it['y'].tolist())

        row_offset += T

    # Build column names (for debugging / feature importance)
    col_names = list(brain_cols)  # (1)
    for r in paired_regions:
        col_names += [f'{r}_sum', f'{r}_diff', f'{r}_abs_diff', f'{r}_ratio']
    for lag in lag_offsets:
        for direction in ['past', 'future']:
            for ci, col_name in enumerate(brain_cols):
                col_names.append(f'{col_name}_lag_{direction}_{lag}')
            for ci, col_name in enumerate(brain_cols):
                col_names.append(f'{col_name}_lagdiff_{direction}_{lag}')
    col_names += ['time_rel', 'time_abs', 'pos_in_seq', 'seq_len',
                  'pos_ratio', 'mouse_idx', 'day_idx', 'pair_idx']
    col_names += ['seq_brain_mean_all', 'seq_brain_std_all']
    for c in brain_cols:
        col_names.append(f'{c}_seq_mean')
    for c in brain_cols:
        col_names.append(f'{c}_seq_std')
    for c in brain_cols:
        col_names.append(f'{c}_md_brain_mean')
    for c in brain_cols:
        col_names.append(f'{c}_md_brain_std')
    for c in brain_cols:
        col_names.append(f'{c}_md_brain_dev')
    col_names += ['brain_mean_x_pos', 'brain_std_x_pos']
    # print("col_names count", len(col_names))
    for ws in [3, 5, 7]:
        col_names += [f'brain_rollmean_{ws}', f'brain_rollstd_{ws}',
                      f'brain_rollmax_{ws}', f'brain_rollmin_{ws}']

    t1 = _time.time()
    print(f"  [precompute_flat] {total_rows} rows, {n_invariant} features in {t1-t0:.1f}s")

    # feature diagnostics from one of the broken runs
    # print('flat feat?', feat_matrix.shape)
    # print('finite ratio', np.isfinite(feat_matrix).mean())
    # print('range', np.nanmin(feat_matrix), np.nanmax(feat_matrix))
    # print('column names (last)', col_names[-15:])
    # assert len(row_ids) == feat_matrix.shape[0]
    # assert len(col_names) == feat_matrix.shape[1]
    # np.save('/content/flat_features_debug.npy', feat_matrix[:500])

    return (feat_matrix, col_names, row_ids,
            np.array(targets, dtype=np.float32) if targets else None,
            mouse_ids_per_row, mouseday_keys_per_row)




In [ ]:
# ---- Lever aggregation stat names ----
MOUSE_LEVER_STAT_NAMES = ['mean','std','min','max','median','q25','q75',
                          'mode','active_ratio','skew','kurtosis']
MOUSEDAY_LEVER_STAT_NAMES = ['mean','std','min','max','median','q25','q75',
                             'active_ratio','range']


# leakage check: this list MUST be the train side of the current fold
# print('stats from sequences', len(train_items_for_stats))
# print('mice in this fold', len(set(x['mouse_id'] for x in train_items_for_stats)))
# old approach mistakenly pooled val lever for the aggregate stats :/
# all_stats_items = train_items_all
# mouse_lever_stats = _compute_lever_agg_stats(all_stats_items)
# leave that off. val target leakage was a very bad idea

def _compute_lever_agg_stats(train_items_for_stats):
    """Compute per-mouse and per-mouse-day lever stats from a train split."""
    mouse_lever_stats = {}
    mouseday_lever_stats = {}

    mouse_levers = defaultdict(list)
    mouseday_levers = defaultdict(list)
    for it in train_items_for_stats:
        mouse_levers[it['mouse_id']].extend(it['y'].tolist())
        mouseday_levers[(it['mouse_id'], it['day_n'])].extend(it['y'].tolist())

    for mid, vals in mouse_levers.items():
        v = np.array(vals, dtype=np.float32)
        mouse_lever_stats[mid] = np.array([
            np.mean(v), np.std(v), np.min(v), np.max(v), np.median(v),
            np.percentile(v, 25), np.percentile(v, 75),
            float(scipy_stats.mode(lever_to_bin(v), keepdims=False).mode),
            float(np.mean(v > 0.5)),
            float(scipy_stats.skew(v)),
            float(scipy_stats.kurtosis(v)),
        ], dtype=np.float32)

    for key, vals in mouseday_levers.items():
        v = np.array(vals, dtype=np.float32)
        mouseday_lever_stats[key] = np.array([
            np.mean(v), np.std(v), np.min(v), np.max(v), np.median(v),
            np.percentile(v, 25), np.percentile(v, 75),
            float(np.mean(v > 0.5)),
            float(np.max(v) - np.min(v)),
        ], dtype=np.float32)

    return mouse_lever_stats, mouseday_lever_stats


def _append_lever_stats(feat_matrix, col_names, mouse_ids_per_row,
                        mouseday_keys_per_row, train_items_for_stats):
    """
    Append fold-specific lever aggregation columns to a precomputed
    feature matrix. Returns new (feat_matrix, col_names).
    """
    n_rows = feat_matrix.shape[0]
    n_mouse_stats = len(MOUSE_LEVER_STAT_NAMES)
    n_md_stats = len(MOUSEDAY_LEVER_STAT_NAMES)
    # print("n_md_stats", getattr(n_md_stats, "shape", type(n_md_stats).__name__))

    mouse_stats, md_stats = _compute_lever_agg_stats(train_items_for_stats)

    extra = np.full((n_rows, n_mouse_stats + n_md_stats), np.nan, dtype=np.float32)
    # print("extra??", type(extra), str(extra)[:80])

    for i in range(n_rows):
        mid = mouse_ids_per_row[i]
        md_key = mouseday_keys_per_row[i]
        if mid in mouse_stats:
            extra[i, :n_mouse_stats] = mouse_stats[mid]
        if md_key in md_stats:
            extra[i, n_mouse_stats:] = md_stats[md_key]

    new_matrix = np.concatenate([feat_matrix, extra], axis=1)
    # print("new_matrix", getattr(new_matrix, "shape", type(new_matrix).__name__))

    new_col_names = list(col_names)
    # print("new_col_names count", len(new_col_names))
    for name in MOUSE_LEVER_STAT_NAMES:
        new_col_names.append(f'mouse_lever_{name}')
    for name in MOUSEDAY_LEVER_STAT_NAMES:
        new_col_names.append(f'md_lever_{name}')

    return new_matrix, new_col_names




In [ ]:
# ---- Precompute fold-invariant features ONCE for train and test ----
print("Precomputing flat features (fold-invariant, done once)...")
(_train_flat_matrix, _train_flat_cols, _train_flat_ids,
 _train_flat_targets, _train_mouse_ids, _train_md_keys) = \
    _precompute_flat_features_for_items(train_items_all)

(_test_flat_matrix, _test_flat_cols, _test_flat_ids,
 _, _test_mouse_ids, _test_md_keys) = \
    _precompute_flat_features_for_items(test_items)

# Build row-id -> row-index maps for fast slicing
_train_id_to_rowidx = {rid: i for i, rid in enumerate(_train_flat_ids)}
_test_id_to_rowidx = {rid: i for i, rid in enumerate(_test_flat_ids)}

print(f"  Train flat: {_train_flat_matrix.shape}, Test flat: {_test_flat_matrix.shape}")
print(f"  Invariant features: {len(_train_flat_cols)}")


# quick debug for flat matrices after precompute
# print('train flat', _train_flat_matrix.shape)
# print('test flat', _test_flat_matrix.shape)
# print('same feature order?', _train_flat_cols == _test_flat_cols)
# assert len(_train_flat_ids) == len(set(_train_flat_ids))
# _train_flat_matrix = np.nan_to_num(_train_flat_matrix)  # NO -- hides problems
# print('first ids', _train_flat_ids[:4])

def build_flat_features_fast(items, train_items_for_stats, source='train'):
    """
    Fast fold-specific flat feature builder.
    Slices from precomputed matrices and appends lever stats.
    """
    # which cached matrix are we slicing?
    # print("feature rows", len(items))  # TODO ゼロのときどうする？
    if source == 'train':
        base_matrix = _train_flat_matrix
        base_cols = _train_flat_cols
        base_ids = _train_flat_ids
        id_to_idx = _train_id_to_rowidx
        base_targets = _train_flat_targets
        # print("base_targets", getattr(base_targets, "shape", type(base_targets).__name__))
        base_mouse_ids = _train_mouse_ids
        base_md_keys = _train_md_keys
    else:
        base_matrix = _test_flat_matrix
        base_cols = _test_flat_cols
        base_ids = _test_flat_ids
        id_to_idx = _test_id_to_rowidx
        base_targets = None
        base_mouse_ids = _test_mouse_ids
        base_md_keys = _test_md_keys

    # Gather row indices for the requested items
    row_indices = []
    for it in items:
        for rid in it['row_ids']:
            row_indices.append(id_to_idx[rid])
    row_indices = np.array(row_indices, dtype=np.int64)

    sliced_matrix = base_matrix[row_indices]
    sliced_ids = [base_ids[i] for i in row_indices]
    sliced_mouse = [base_mouse_ids[i] for i in row_indices]
    sliced_md = [base_md_keys[i] for i in row_indices]
    # print("sliced_md??", type(sliced_md), str(sliced_md)[:80])
    sliced_targets = base_targets[row_indices] if base_targets is not None else None
    # print("sliced_targets", getattr(sliced_targets, "shape", type(sliced_targets).__name__))

    # Append lever stats (fold-specific)
    final_matrix, final_cols = _append_lever_stats(
        sliced_matrix, base_cols, sliced_mouse, sliced_md, train_items_for_stats
    )

    # Return as lightweight dict-like object instead of DataFrame for speed
    return FlatFeatureSet(final_matrix, final_cols, sliced_ids, sliced_targets)


class FlatFeatureSet:
    """Lightweight container for flat features, avoids DataFrame overhead."""
    __slots__ = ['matrix', 'col_names', 'ids', 'targets']
    # print("__slots__??", type(__slots__), str(__slots__)[:80])  # wtf

    def __init__(self, matrix, col_names, ids, targets):
        self.matrix = matrix            # np.float32 (N, F)
        self.col_names = col_names      # list of str
        self.ids = np.array(ids)        # np.array of str
        self.targets = targets          # np.float32 (N,) or None

    @property
    def shape(self):
        return self.matrix.shape




In [ ]:
# ~~~
# scaling, PCA helpers
# ~~~
def compute_stats(items):
    all_data = np.concatenate([it['X'] for it in items], axis=0)
    mean = all_data.mean(axis=0).astype(np.float32)
    std = np.maximum(all_data.std(axis=0).astype(np.float32), 1e-6)
    # print("std", getattr(std, "shape", type(std).__name__))
    return mean, std


# PCA debug from the first draft
# full_X = np.concatenate([it['X'] for it in items], axis=0)
# print('raw feature dim', full_X.shape[-1])
# print('std min?', full_X.std(axis=0).min())
# pca_components = 64  # at one point tried 64. score got worse?
# hard-coded 96 is a bit arbitrary tbh

def fit_preprocessing(items, use_pca=True, pca_components=96, seed=42):
    mean_feat, std_feat = compute_stats(items)
    C_feat = mean_feat.shape[0]
    pca = None
    # print("pca", type(pca).__name__)
    if use_pca and C_feat > pca_components:
        all_data = np.concatenate([(it['X'] - mean_feat) / std_feat for it in items], axis=0)
        pca = PCA(n_components=pca_components, random_state=seed)
        pca.fit(all_data)
        C_feat = pca_components
        print(f"[Preproc seed={seed}] PCA explained variance: {pca.explained_variance_ratio_.sum():.4f}")
    return mean_feat, std_feat, pca, C_feat




In [ ]:
#
# folds (split by sequence, not individual rows)
#
def get_kfold_splits(n_items, n_folds=5, seed=42):
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    # print("kf??", type(kf), str(kf)[:80])  # ここ確認
    indices = np.arange(n_items)
    return list(kf.split(indices))


#
# dataset - y, dy, bins all together
# ------------------
# check the padding / masks without building an entire DataLoader
# ds0 = LeverDataset(train_items_all[:3], *compute_stats(train_items_all[:3]), tmax=cfg.tmax)
# ex = ds0[0]
# print({k: tuple(v.shape) if hasattr(v, 'shape') else v for k, v in ex.items()})
# print('mask:', ex['mask'])
# assert ex['mask'].sum() == min(cfg.tmax, train_items_all[0]['seq_len'])
# first version forgot to clip seq_len to tmax, remember that!

class LeverDataset(Dataset):
    # print("dataset ready?")
    def __init__(self, items, mean, std, pca=None, tmax=40,
                 with_target=True, augment=False):
        self.items = items
        self.mean = mean
        self.std = std
        self.pca = pca
        self.tmax = tmax
        self.with_target = with_target
        self.augment = augment
        self.C = pca.n_components_ if pca else mean.shape[0]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        it = self.items[idx]
        # print("it??", type(it), str(it)[:80])
        X = it['X'].copy()
        # print("X??", type(X), str(X)[:80])
        T = X.shape[0]

        X = (X - self.mean) / self.std
        if self.pca is not None:
            X = self.pca.transform(X).astype(np.float32)

        if self.augment and np.random.rand() < 0.5:
            X += np.random.randn(*X.shape).astype(np.float32) * 0.02

        X_pad = np.zeros((self.tmax, self.C), dtype=np.float32)
        t_pad = np.zeros((self.tmax, 1), dtype=np.float32)
        mask = np.zeros((self.tmax,), dtype=np.float32)

        L = min(T, self.tmax)
        X_pad[:L] = X[:L]
        t_pad[:L] = it['time_rel'][:L].reshape(-1, 1)
        mask[:L] = 1.0

        out = {
            'X': torch.from_numpy(X_pad),
            'time': torch.from_numpy(t_pad),
            'mask': torch.from_numpy(mask),
            'mouse': torch.tensor(it['mouse_idx'], dtype=torch.long),
            'pair': torch.tensor(it['pair_idx'], dtype=torch.long),
            'seq_len': L,
            'seq_class': torch.tensor(it['seq_class'], dtype=torch.long),
        }

        if self.with_target:
            y_pad = np.zeros((self.tmax,), dtype=np.float32)
            y_pad[:L] = it['y'][:L]
            out['y'] = torch.from_numpy(y_pad)

            y_diff_pad = np.zeros((self.tmax,), dtype=np.float32)
            y_diff_pad[:L] = it['y_diff'][:L]
            out['y_diff'] = torch.from_numpy(y_diff_pad)

            y_diff2_pad = np.zeros((self.tmax,), dtype=np.float32)
            # print("y_diff2_pad??", type(y_diff2_pad), str(y_diff2_pad)[:80])  # optional
            y_diff2_pad[:L] = it['y_diff2'][:L]
            out['y_diff2'] = torch.from_numpy(y_diff2_pad)

            y_bin_pad = np.zeros((self.tmax,), dtype=np.int64)
            y_bin_pad[:L] = it['y_bin'][:L]
            out['y_bin'] = torch.from_numpy(y_bin_pad)

        return out




In [ ]:
#
# NN architectures, too many versions...
#

# older conv attempt was just one kernel
# class TemporalConvOld(nn.Module):
#     def __init__(self, c):
#         super().__init__()
#         self.conv = nn.Conv1d(c, c, 3, padding=1)
#     def forward(self, x):
#         return F.relu(self.conv(x.transpose(1, 2))).transpose(1, 2)
# dropped it after messing with multiscale

class MultiScaleTemporalConv(nn.Module):
    def __init__(self, channels, scales=[3, 5, 7]):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Conv1d(channels, channels, k, padding=k//2) for k in scales
        ])
        self.norms = nn.ModuleList([nn.BatchNorm1d(channels) for _ in scales])
        self.fusion = nn.Linear(channels * len(scales), channels)

    def forward(self, x):
        x_t = x.transpose(1, 2)
        feats = [F.gelu(norm(conv(x_t))) for conv, norm in zip(self.convs, self.norms)]
        # print("feats", getattr(feats, "shape", type(feats).__name__))
        h = torch.cat(feats, dim=1).transpose(1, 2)
        return self.fusion(h) + x




In [ ]:
# ---- BRANCH 1: Transformer (original) ----
# attention shapes were a recurring source of pain
# print('d_model % n_heads', cfg.d_model % cfg.n_heads)
# assert cfg.d_model % cfg.n_heads == 0
# tried batch_first=False with transpose in forward. skip.
# encoder_layer = nn.TransformerEncoderLayer(d_model=cfg.d_model, nhead=cfg.n_heads)
# hidden = encoder(hidden.transpose(0, 1)).transpose(0, 1)
# なぜここだけ shape がズレる???

class MultiTaskTransformer(nn.Module):
    # print("Transformer dim check")
    def __init__(self, num_mice, num_days, brain_dim, num_bins,
                 d_model=512, n_heads=16, n_layers=12, d_ff=1536,
                 dropout=0.15, meta_dim=128, tmax=40):
        super().__init__()
        self.d_model = d_model
        self.tmax = tmax
        self.num_bins = num_bins

        self.mouse_emb = nn.Embedding(num_mice, meta_dim)
        self.pair_emb = nn.Embedding(num_mice * num_days, meta_dim)
        self.multi_temp = MultiScaleTemporalConv(brain_dim)

        in_dim = brain_dim + meta_dim * 2 + 1
        # print("dbg in_dim", in_dim)
        self.in_proj = nn.Sequential(
            nn.Linear(in_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout)
        )
        self.pos_emb = nn.Embedding(tmax, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        self.shared_trunk = nn.Sequential(
            nn.Linear(d_model, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
        )

        self.head_reg = nn.Linear(128, 1)
        self.head_diff = nn.Linear(128, 1)
        self.head_diff2 = nn.Linear(128, 1)
        self.head_cls = nn.Linear(128, num_bins)

    def forward(self, X, time, mouse, pair, mask, use_causal=True):
        B, T, C = X.shape
        X = self.multi_temp(X)

        m = self.mouse_emb(mouse)
        p = self.pair_emb(pair)
        # print("p??", type(p), str(p)[:80])
        meta = torch.cat([m, p], dim=-1).unsqueeze(1).expand(B, T, -1)

        h = torch.cat([X, meta, time], dim=-1)
        # print("h??", type(h), str(h)[:80])  # was wrong before
        h = self.in_proj(h)

        pos_ids = torch.arange(T, device=h.device).unsqueeze(0).expand(B, T)
        h = h + self.pos_emb(pos_ids)

        pad_mask = mask < 0.5
        h = self.encoder(h, src_key_padding_mask=pad_mask)

        feat = self.shared_trunk(h)
        reg_out = self.head_reg(feat).squeeze(-1)
        diff_out = self.head_diff(feat).squeeze(-1)
        cls_out = self.head_cls(feat)
        diff2_out = self.head_diff2(feat).squeeze(-1)
        return reg_out, diff_out, diff2_out, cls_out




In [ ]:
# ---- BRANCH 2: Conv1D-LSTM (original) ----
# earlier model was a plain GRU, no multiscale conv
# self.gru = nn.GRU(in_dim, hidden_dim, batch_first=True, bidirectional=True)
# feat, _ = self.gru(h)
# feat = self.dropout(feat)
# print('lstm feat size', feat.size())
# NOTE: the LSTM hidden size is per direction

class MultiTaskConv1dLSTM(nn.Module):
    # print("conv/lstm dims")
    def __init__(self, num_mice, num_days, brain_dim, num_bins,
                 hidden_dim=256, lstm_layers=3, dropout=0.15,
                 meta_dim=128, tmax=40):
        super().__init__()
        self.num_bins = num_bins

        self.mouse_emb = nn.Embedding(num_mice, meta_dim)
        self.pair_emb = nn.Embedding(num_mice * num_days, meta_dim)

        in_dim = brain_dim + meta_dim * 2 + 1
        # print("dbg in_dim", in_dim)  # optional

        self.conv1 = nn.Conv1d(in_dim, hidden_dim, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.conv2 = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.conv3 = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(hidden_dim)
        self.conv4 = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=1, padding=0)
        self.bn4 = nn.BatchNorm1d(hidden_dim)

        self.lstm = nn.LSTM(
            input_size=hidden_dim, hidden_size=256, num_layers=lstm_layers,
            batch_first=True, dropout=dropout if lstm_layers > 1 else 0.0,
            bidirectional=True,
        )
        self.lstm_drop = nn.Dropout(dropout)

        self.shared_trunk = nn.Sequential(
            nn.Linear(512, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
        )

        self.head_reg = nn.Linear(128, 1)
        self.head_diff = nn.Linear(128, 1)
        self.head_cls = nn.Linear(128, num_bins)
        self.head_diff2 = nn.Linear(128, 1)

        self._init_weights()

    def _init_weights(self):
        for name, p in self.named_parameters():
            if 'lstm' in name:
                if 'weight_ih' in name: nn.init.xavier_normal_(p.data)
                elif 'weight_hh' in name: nn.init.orthogonal_(p.data)
                elif 'bias_ih' in name:
                    p.data.fill_(0)
                    n = p.size(0)
                    p.data[(n // 4):(n // 2)].fill_(1)
                elif 'bias_hh' in name: p.data.fill_(0)
            elif ('conv' in name or 'head' in name or 'shared' in name):
                if 'weight' in name and p.dim() >= 2: nn.init.xavier_normal_(p.data)
                elif 'bias' in name: p.data.fill_(0)

    def forward(self, X, time, mouse, pair, mask, use_causal=True):
        B, T, C = X.shape
        m = self.mouse_emb(mouse)
        p = self.pair_emb(pair)
        meta = torch.cat([m, p], dim=-1).unsqueeze(1).expand(B, T, -1)
        h = torch.cat([X, meta, time], dim=-1).transpose(1, 2)

        h = F.gelu(self.bn1(self.conv1(h)))
        # print("h??", type(h), str(h)[:80])
        h = F.gelu(self.bn2(self.conv2(h))) + h
        h = F.gelu(self.bn3(self.conv3(h))) + h
        h = F.gelu(self.bn4(self.conv4(h))) + h
        h = h.transpose(1, 2)

        lengths = mask.sum(dim=1).long().clamp(min=1)
        # print("dbg lengths", lengths)  # maybe delete later
        lengths_sorted, sort_idx = torch.sort(lengths, descending=True)
        h_sorted = h.index_select(0, sort_idx)

        packed = nn.utils.rnn.pack_padded_sequence(
            h_sorted, lengths_sorted.cpu(), batch_first=True, enforce_sorted=True)
        packed, _ = self.lstm(packed)
        h_out, _ = nn.utils.rnn.pad_packed_sequence(packed, batch_first=True, total_length=T)

        _, unsort_idx = torch.sort(sort_idx)
        h_out = h_out.index_select(0, unsort_idx)
        h_out = self.lstm_drop(h_out) * mask.unsqueeze(-1)

        feat = self.shared_trunk(h_out)
        reg_out = self.head_reg(feat).squeeze(-1)
        diff_out = self.head_diff(feat).squeeze(-1)
        cls_out = self.head_cls(feat)
        diff2_out = self.head_diff2(feat).squeeze(-1)
        # print("diff2_out??", type(diff2_out), str(diff2_out)[:80])  # was wrong before
        return reg_out, diff_out, diff2_out, cls_out




In [ ]:
# ---- BRANCH 3: ResNet -> BiGRU -> Self-Attention ----
class ResBlock1d(nn.Module):
    def __init__(self, channels, kernel_size=3, dropout=0.1):
        super().__init__()
        pad = kernel_size // 2
        self.net = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size, padding=pad),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(channels, channels, kernel_size, padding=pad),
            nn.BatchNorm1d(channels),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(self.net(x) + x)


# abandoned single residual block sanity net
# self.pre_conv = nn.Conv1d(brain_dim, hidden_dim, 1)
# self.res_blocks = nn.Sequential(ResBlock1d(hidden_dim))
# self.pool = nn.AdaptiveAvgPool1d(1)
# output = self.pool(self.res_blocks(self.pre_conv(X.transpose(1, 2)))).squeeze(-1)
# no temporal output that way -- needs per-frame preds!

class MultiTaskResGRUAttn(nn.Module):
    """ResNet-1D -> BiGRU -> Multi-Head Self-Attention -> Heads"""
    def __init__(self, num_mice, num_days, brain_dim, num_bins,
                 hidden_dim=192, gru_layers=2, n_heads=4,
                 dropout=0.15, meta_dim=64, tmax=40):
        super().__init__()
        self.num_bins = num_bins

        self.mouse_emb = nn.Embedding(num_mice, meta_dim)
        self.pair_emb = nn.Embedding(num_mice * num_days, meta_dim)

        in_dim = brain_dim + meta_dim * 2 + 1
        # print("dbg in_dim", in_dim)

        # ResNet-1D blocks
        self.in_conv = nn.Sequential(
            nn.Conv1d(in_dim, hidden_dim, 1),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
        )
        self.res_blocks = nn.Sequential(
            ResBlock1d(hidden_dim, kernel_size=3, dropout=dropout),
            ResBlock1d(hidden_dim, kernel_size=5, dropout=dropout),
            ResBlock1d(hidden_dim, kernel_size=7, dropout=dropout),
            ResBlock1d(hidden_dim, kernel_size=3, dropout=dropout),
        )

        # BiGRU
        self.gru = nn.GRU(
            input_size=hidden_dim, hidden_size=hidden_dim,
            num_layers=gru_layers, batch_first=True,
            dropout=dropout if gru_layers > 1 else 0.0,
            bidirectional=True,
        )
        self.gru_proj = nn.Linear(hidden_dim * 2, hidden_dim)
        self.gru_norm = nn.LayerNorm(hidden_dim)

        # Self-Attention
        self.self_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=n_heads,
            dropout=dropout, batch_first=True,
        )
        self.attn_norm = nn.LayerNorm(hidden_dim)
        self.attn_ff = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
        )
        self.ff_norm = nn.LayerNorm(hidden_dim)

        # Shared trunk
        self.shared_trunk = nn.Sequential(
            nn.Linear(hidden_dim, 128), nn.LayerNorm(128),
            nn.GELU(), nn.Dropout(dropout),
        )

        # Heads
        self.head_reg = nn.Linear(128, 1)
        self.head_diff = nn.Linear(128, 1)
        self.head_diff2 = nn.Linear(128, 1)
        self.head_cls = nn.Linear(128, num_bins)

    def forward(self, X, time, mouse, pair, mask, use_causal=True):
        B, T, C = X.shape

        m = self.mouse_emb(mouse)
        p = self.pair_emb(pair)
        # print("p??", type(p), str(p)[:80])
        meta = torch.cat([m, p], dim=-1).unsqueeze(1).expand(B, T, -1)
        h = torch.cat([X, meta, time], dim=-1)  # (B, T, in_dim)

        # ResNet-1D
        h = h.transpose(1, 2)  # (B, in_dim, T)
        h = self.in_conv(h)
        h = self.res_blocks(h)
        h = h.transpose(1, 2)  # (B, T, hidden_dim)

        # BiGRU
        lengths = mask.sum(dim=1).long().clamp(min=1)
        # print("dbg lengths", lengths)
        lengths_sorted, sort_idx = torch.sort(lengths, descending=True)
        h_sorted = h.index_select(0, sort_idx)
        packed = nn.utils.rnn.pack_padded_sequence(
            h_sorted, lengths_sorted.cpu(), batch_first=True, enforce_sorted=True)
        packed_out, _ = self.gru(packed)
        h_gru, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True, total_length=T)
        _, unsort_idx = torch.sort(sort_idx)
        h_gru = h_gru.index_select(0, unsort_idx)
        h_gru = self.gru_norm(self.gru_proj(h_gru))
        h = h + h_gru  # residual
        # print("h??", type(h), str(h)[:80])

        # Self-Attention
        pad_mask = mask < 0.5
        attn_out, _ = self.self_attn(h, h, h, key_padding_mask=pad_mask)
        h = self.attn_norm(h + attn_out)
        h = self.ff_norm(h + self.attn_ff(h))

        h = h * mask.unsqueeze(-1)
        # print("h??", type(h), str(h)[:80])

        feat = self.shared_trunk(h)
        # print("feat", getattr(feat, "shape", type(feat).__name__))
        reg_out = self.head_reg(feat).squeeze(-1)
        diff_out = self.head_diff(feat).squeeze(-1)
        diff2_out = self.head_diff2(feat).squeeze(-1)
        cls_out = self.head_cls(feat)
        return reg_out, diff_out, diff2_out, cls_out




In [ ]:
# ---- BRANCH 4: TCN (non-causal, kernel_size=5) ----
class TCNBlock(nn.Module):
    """Non-causal temporal convolutional block with residual connection."""
    def __init__(self, channels, kernel_size=5, dilation=1, dropout=0.15):
        super().__init__()
        pad = dilation * (kernel_size - 1) // 2  # symmetric padding for non-causal
        self.net = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size, padding=pad, dilation=dilation),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(channels, channels, kernel_size, padding=pad, dilation=dilation),
            nn.BatchNorm1d(channels),
        )
        self.act = nn.GELU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = self.net(x)
        # Ensure same length
        if out.shape[-1] != x.shape[-1]:
            out = out[:, :, :x.shape[-1]]
            # print("out??", type(out), str(out)[:80])  # wtf
        return self.act(out + x)


# TCN debugging from an older test
# dilations = [1, 1, 1, 1]     # receptive field way too short
# dilations = [1, 2, 4, 8, 16]
# k = 3  # tried this as well
# output_len = (T + 2 * padding - dilation * (k - 1) - 1) + 1
# print('expected vs actual length', T, output_len)
# WHY IS THIS XXXX???? padding and dilation again

class MultiTaskTCN(nn.Module):
    """Non-causal TCN backbone with multi-task heads."""
    def __init__(self, num_mice, num_days, brain_dim, num_bins,
                 n_channels=192, kernel_size=5, n_blocks=4,
                 dropout=0.15, meta_dim=64, tmax=40):
        super().__init__()
        self.num_bins = num_bins

        self.mouse_emb = nn.Embedding(num_mice, meta_dim)
        self.pair_emb = nn.Embedding(num_mice * num_days, meta_dim)

        in_dim = brain_dim + meta_dim * 2 + 1

        self.in_conv = nn.Sequential(
            nn.Conv1d(in_dim, n_channels, 1),
            nn.BatchNorm1d(n_channels),
            nn.GELU(),
        )

        # Stacked TCN blocks with exponentially increasing dilations
        self.tcn_blocks = nn.ModuleList([
            TCNBlock(n_channels, kernel_size=kernel_size,
                     dilation=2**i, dropout=dropout)
            for i in range(n_blocks)
        ])

        self.shared_trunk = nn.Sequential(
            nn.Linear(n_channels, 128), nn.LayerNorm(128),
            nn.GELU(), nn.Dropout(dropout),
        )

        self.head_reg = nn.Linear(128, 1)
        self.head_diff = nn.Linear(128, 1)
        self.head_diff2 = nn.Linear(128, 1)
        self.head_cls = nn.Linear(128, num_bins)

    def forward(self, X, time, mouse, pair, mask, use_causal=True):
        B, T, C = X.shape

        m = self.mouse_emb(mouse)
        p = self.pair_emb(pair)
        # print("p??", type(p), str(p)[:80])
        meta = torch.cat([m, p], dim=-1).unsqueeze(1).expand(B, T, -1)
        h = torch.cat([X, meta, time], dim=-1)

        h = h.transpose(1, 2)  # (B, in_dim, T)
        h = self.in_conv(h)
        for block in self.tcn_blocks:
            h = block(h)
        h = h.transpose(1, 2)  # (B, T, n_channels)
        h = h * mask.unsqueeze(-1)
        # print("h??", type(h), str(h)[:80])

        feat = self.shared_trunk(h)
        # print("feat", getattr(feat, "shape", type(feat).__name__))
        reg_out = self.head_reg(feat).squeeze(-1)
        diff_out = self.head_diff(feat).squeeze(-1)
        diff2_out = self.head_diff2(feat).squeeze(-1)
        cls_out = self.head_cls(feat)
        return reg_out, diff_out, diff2_out, cls_out




In [ ]:
# ---- BRANCH 6: Pure MLP (row-level, pytorch) ----
# old nonlinearity was ReLU only:
# layers.extend([nn.Linear(d0, d1), nn.ReLU()])
# also had BatchNorm1d here, not ideal with tiny last batch
# nn.BatchNorm1d(d1)
# print('MLP dims', input_dim, hidden_dims)

class PureMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=[512, 256, 128], dropout=0.2):
        super().__init__()
        layers = []
        prev = input_dim
        # print("prev??", type(prev), str(prev)[:80])
        for hd in hidden_dims:
            layers.extend([
                nn.Linear(prev, hd),
                nn.BatchNorm1d(hd),
                nn.GELU(),
                nn.Dropout(dropout),
            ])
            prev = hd
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)




In [ ]:
# ------------------
# loss bits
# ------------------
def masked_mse(yhat, y, mask):
    return ((yhat - y) ** 2 * mask).sum() / (mask.sum() + 1e-6)


_BIN_VALUES_TENSOR = None
def _get_bin_values_tensor(device):
    global _BIN_VALUES_TENSOR
    if _BIN_VALUES_TENSOR is None or _BIN_VALUES_TENSOR.device != device:
        _BIN_VALUES_TENSOR = torch.tensor(BIN_VALUES, dtype=torch.float32, device=device)
    return _BIN_VALUES_TENSOR


def make_soft_targets(y, sigma):
    bv = _get_bin_values_tensor(y.device)
    dists = (y.unsqueeze(-1) - bv.view(1, 1, -1)) ** 2
    logits = -dists / (2 * sigma ** 2)
    # print("logits", getattr(logits, "shape", type(logits).__name__))  # sometimes NaN
    return F.softmax(logits, dim=-1)


def masked_soft_ce_loss(logits, soft_targets, mask):
    log_probs = F.log_softmax(logits, dim=-1)
    # print("log_probs??", type(log_probs), str(log_probs)[:80])
    ce = -(soft_targets * log_probs).sum(dim=-1)
    return (ce * mask).sum() / (mask.sum() + 1e-6)


# quick loss variants (keeping these OFF)
# total = l_reg
# total = cfg.w_regression * l_reg + 0.1 * l_diff
# total += 0.05 * l_cls   # bins didn't help in that run
# print('loss parts', l_reg.item(), l_diff.item(), l_diff2.item(), l_cls.item())
# print('mask sum', mask.sum().item())
# why did classification blow up when sigma got tiny???

def multitask_loss(reg_out, diff_out, diff2_out, cls_out, y, y_diff, y_diff2, mask, cfg,
                   soft_targets=None):
    l_reg = masked_mse(reg_out, y, mask)
    l_diff = masked_mse(diff_out, y_diff, mask)
    l_diff2 = masked_mse(diff2_out, y_diff2, mask)
    if soft_targets is None:
        soft_targets = make_soft_targets(y, cfg.soft_ce_sigma)
    l_cls = masked_soft_ce_loss(cls_out, soft_targets, mask)
    total = (cfg.w_regression * l_reg
            + cfg.w_diff * l_diff
            + cfg.w_diff2 * l_diff2
            + cfg.w_classification * l_cls)
    return total, l_reg, l_diff, l_diff2, l_cls




In [ ]:
# ~~~
# mixup/cutmix; embeddings were a headache
# ~~~
def apply_mixup(batch, alpha=0.4):
    """Mixup that also mixes mouse/pair embeddings via soft interpolation."""
    B = batch['X'].shape[0]
    lam = np.random.beta(alpha, alpha)
    # print("lam??", type(lam), str(lam)[:80])
    lam = max(lam, 1 - lam)
    perm = torch.randperm(B)

    batch['X'] = lam * batch['X'] + (1 - lam) * batch['X'][perm]
    batch['time'] = lam * batch['time'] + (1 - lam) * batch['time'][perm]
    batch['mask'] = batch['mask'] * batch['mask'][perm]

    # Store perm and lambda so embeddings can be mixed in the forward pass
    batch['_mixup_perm'] = perm
    batch['_mixup_lam'] = lam

    if 'y' in batch:
        batch['y'] = lam * batch['y'] + (1 - lam) * batch['y'][perm]
        batch['y_diff'] = lam * batch['y_diff'] + (1 - lam) * batch['y_diff'][perm]
        batch['y_diff2'] = lam * batch['y_diff2'] + (1 - lam) * batch['y_diff2'][perm]

    return batch, lam, perm


# mask bookkeeping test that I kept forgetting to run
# print('lambda', lam, 'perm', perm[:5])
# assert batch['mask'].shape == batch['y'].shape
# print('X before aug', batch['X'].shape)
# print('finite Y?', torch.isfinite(batch['y']).all())
# batch['X'] += torch.randn_like(batch['X']) * 0.005  # tiny jitter trial

def apply_cutmix(batch, cfg):
    B, T, C = batch['X'].shape
    perm = torch.randperm(B)

    cut_ratio = np.random.uniform(0.2, 0.5)
    # print("dbg cut_ratio", cut_ratio)
    cut_len = max(1, int(T * cut_ratio))
    cut_start = np.random.randint(0, T - cut_len + 1)
    cut_end = cut_start + cut_len
    # print("cut_end??", type(cut_end), str(cut_end)[:80])

    cut_mask = torch.ones(T, device=batch['X'].device)
    cut_mask[cut_start:cut_end] = 0
    cut_mask = cut_mask.view(1, T)
    lam = 1.0 - cut_ratio

    batch['X'] = batch['X'] * cut_mask.unsqueeze(-1) + batch['X'][perm] * (1 - cut_mask).unsqueeze(-1)
    batch['time'] = batch['time'] * cut_mask.unsqueeze(-1) + batch['time'][perm] * (1 - cut_mask).unsqueeze(-1)
    batch['mask'] = batch['mask'] * batch['mask'][perm]

    # For CutMix, embeddings at cut region come from permuted samples
    batch['_cutmix_perm'] = perm
    batch['_cutmix_mask'] = cut_mask  # (1, T)

    if 'y' in batch:
        batch['y'] = batch['y'] * cut_mask.squeeze(0) + batch['y'][perm] * (1 - cut_mask).squeeze(0)
        batch['y_diff'] = batch['y_diff'] * cut_mask.squeeze(0) + batch['y_diff'][perm] * (1 - cut_mask).squeeze(0)
        batch['y_diff2'] = batch['y_diff2'] * cut_mask.squeeze(0) + batch['y_diff2'][perm] * (1 - cut_mask).squeeze(0)

    return batch, lam, perm




In [ ]:
# this used to just call the model on mixed X without mixed embeddings
# reg, diff, diff2, cls = model(X, time, mouse, pair, mask, use_causal=False)
# return reg, diff, diff2, cls
# print('embedding mix?', aug_batch.get('_mixup_perm', None))
# print('X dim', X.size(-1))
# バッチの mouse embedding はどう混ぜる??? の残骸

def forward_with_embed_mixing(model, X, time, mouse, pair, mask, aug_batch):
    """
    Wrapper that handles embedding mixing for Mixup/CutMix.
    For Mixup: interpolate embeddings with lambda.
    For CutMix: swap embeddings in cut region.
    Falls back to standard forward if no mixing info present.
    """
    # Check if we need to mix embeddings
    has_mixup = '_mixup_perm' in aug_batch
    has_cutmix = '_cutmix_perm' in aug_batch

    if not has_mixup and not has_cutmix:
        return model(X, time, mouse, pair, mask, use_causal=False)

    # We need to access the embedding layers directly
    B, T, C = X.shape

    if hasattr(model, 'multi_temp'):
        X_proc = model.multi_temp(X)
    else:
        X_proc = X

    m_emb = model.mouse_emb(mouse)   # (B, meta_dim)
    # print("m_emb??", type(m_emb), str(m_emb)[:80])
    p_emb = model.pair_emb(pair)      # (B, meta_dim)

    if has_mixup:
        perm = aug_batch['_mixup_perm'].to(mouse.device)
        lam = aug_batch['_mixup_lam']
        m_emb = lam * m_emb + (1 - lam) * m_emb[perm]
        p_emb = lam * p_emb + (1 - lam) * p_emb[perm]

    if has_cutmix:
        perm = aug_batch['_cutmix_perm'].to(mouse.device)
        cut_mask_t = aug_batch['_cutmix_mask'].to(mouse.device)  # (1, T)
        m_emb_perm = model.mouse_emb(mouse[perm])
        p_emb_perm = model.pair_emb(pair[perm])
        # Expand embeddings to (B, T, meta_dim)
        m_exp = m_emb.unsqueeze(1).expand(B, T, -1)
        m_perm_exp = m_emb_perm.unsqueeze(1).expand(B, T, -1)
        p_exp = p_emb.unsqueeze(1).expand(B, T, -1)
        p_perm_exp = p_emb_perm.unsqueeze(1).expand(B, T, -1)
        # print("p_perm_exp??", type(p_perm_exp), str(p_perm_exp)[:80])
        cm = cut_mask_t.unsqueeze(-1)  # (1, T, 1)
        meta = torch.cat([
            m_exp * cm + m_perm_exp * (1 - cm),
            p_exp * cm + p_perm_exp * (1 - cm),
        ], dim=-1)

        h = torch.cat([X_proc, meta, time], dim=-1)
        # print("h??", type(h), str(h)[:80])

        # Now we need to run the rest of the model manually
        # This is model-specific, so we handle the common pattern
        return _forward_from_h(model, h, mask, T, B)

    # Mixup path: simple expand
    meta = torch.cat([m_emb, p_emb], dim=-1).unsqueeze(1).expand(B, T, -1)
    h = torch.cat([X_proc, meta, time], dim=-1)
    return _forward_from_h(model, h, mask, T, B)




In [ ]:
# avoid trying to handle every architecture with one forward hack again
# print('model type:', type(model).__name__)
# print('embedding shape:', h.shape, 'mask:', mask.shape)
# assert h.shape[:2] == mask.shape
# BUG: manual head path has to mirror model.forward exactly

def _forward_from_h(model, h, mask, T, B):
    """Continue forward pass from concatenated input h for various model types."""
    if isinstance(model, MultiTaskTransformer):
        h = model.in_proj(h)
        pos_ids = torch.arange(T, device=h.device).unsqueeze(0).expand(B, T)
        h = h + model.pos_emb(pos_ids)
        pad_mask = mask < 0.5
        # print("pad_mask valid", pad_mask.sum().item() if hasattr(pad_mask, "sum") else pad_mask)
        h = model.encoder(h, src_key_padding_mask=pad_mask)
        feat = model.shared_trunk(h)
        return (model.head_reg(feat).squeeze(-1),
                model.head_diff(feat).squeeze(-1),
                model.head_diff2(feat).squeeze(-1),
                model.head_cls(feat))

    elif isinstance(model, MultiTaskConv1dLSTM):
        h = h.transpose(1, 2)
        h = F.gelu(model.bn1(model.conv1(h)))
        h = F.gelu(model.bn2(model.conv2(h))) + h
        h = F.gelu(model.bn3(model.conv3(h))) + h
        h = F.gelu(model.bn4(model.conv4(h))) + h
        h = h.transpose(1, 2)
        # print("h??", type(h), str(h)[:80])  # TODO?
        lengths = mask.sum(dim=1).long().clamp(min=1)
        lengths_sorted, sort_idx = torch.sort(lengths, descending=True)
        h_sorted = h.index_select(0, sort_idx)
        packed = nn.utils.rnn.pack_padded_sequence(
            h_sorted, lengths_sorted.cpu(), batch_first=True, enforce_sorted=True)
        packed, _ = model.lstm(packed)
        h_out, _ = nn.utils.rnn.pad_packed_sequence(packed, batch_first=True, total_length=T)
        _, unsort_idx = torch.sort(sort_idx)
        h_out = h_out.index_select(0, unsort_idx)
        h_out = model.lstm_drop(h_out) * mask.unsqueeze(-1)
        feat = model.shared_trunk(h_out)
        return (model.head_reg(feat).squeeze(-1),
                model.head_diff(feat).squeeze(-1),
                model.head_diff2(feat).squeeze(-1),
                model.head_cls(feat))

    elif isinstance(model, MultiTaskResGRUAttn):
        h = h.transpose(1, 2)
        h = model.in_conv(h)
        h = model.res_blocks(h)
        h = h.transpose(1, 2)
        lengths = mask.sum(dim=1).long().clamp(min=1)
        lengths_sorted, sort_idx = torch.sort(lengths, descending=True)
        h_sorted = h.index_select(0, sort_idx)
        packed = nn.utils.rnn.pack_padded_sequence(
            h_sorted, lengths_sorted.cpu(), batch_first=True, enforce_sorted=True)
        packed_out, _ = model.gru(packed)
        h_gru, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True, total_length=T)
        _, unsort_idx = torch.sort(sort_idx)
        h_gru = h_gru.index_select(0, unsort_idx)
        h_gru = model.gru_norm(model.gru_proj(h_gru))
        h = h + h_gru
        pad_mask = mask < 0.5
        attn_out, _ = model.self_attn(h, h, h, key_padding_mask=pad_mask)
        h = model.attn_norm(h + attn_out)
        h = model.ff_norm(h + model.attn_ff(h))
        h = h * mask.unsqueeze(-1)
        # print("h??", type(h), str(h)[:80])
        feat = model.shared_trunk(h)
        return (model.head_reg(feat).squeeze(-1),
                model.head_diff(feat).squeeze(-1),
                model.head_diff2(feat).squeeze(-1),
                model.head_cls(feat))

    elif isinstance(model, MultiTaskTCN):
        h = h.transpose(1, 2)
        h = model.in_conv(h)
        for block in model.tcn_blocks:
            h = block(h)
        h = h.transpose(1, 2)
        h = h * mask.unsqueeze(-1)
        feat = model.shared_trunk(h)
        # print("feat", getattr(feat, "shape", type(feat).__name__))  # maybe delete later
        return (model.head_reg(feat).squeeze(-1),
                model.head_diff(feat).squeeze(-1),
                model.head_diff2(feat).squeeze(-1),
                model.head_cls(feat))

    else:
        raise ValueError(f"Unknown model type: {type(model)}")




In [ ]:
#  . . .
# val / metrics
#  . . .
@torch.no_grad()
# debug-only metric implementation, not using numpy each batch anymore
# sq = []
# for batch in loader:
#     yhat, *_ = model(batch['X'].to(device), batch['time'].to(device),
#                      batch['mouse'].to(device), batch['pair'].to(device),
#                      batch['mask'].to(device), use_causal=False)
#     sq.extend(((yhat.cpu() - batch['y']) ** 2).flatten().numpy())
# print(np.mean(sq))  # WRONG: this also measures padded positions!

def evaluate_model_mse(model, loader):
    """Evaluate a single sequence model, return val regression MSE."""
    model.eval()
    total_sq, total_n = 0.0, 0.0
    for batch in loader:
        X = batch['X'].to(device)
        time = batch['time'].to(device)
        mask = batch['mask'].to(device)
        mouse = batch['mouse'].to(device)
        pair = batch['pair'].to(device)
        y = batch['y'].to(device)
        reg_out, _, _, _ = model(X, time, mouse, pair, mask, use_causal=False)
        total_sq += ((reg_out - y) ** 2 * mask).sum().item()
        # print("dbg total_sq", total_sq)
        total_n += mask.sum().item()
        # print("dbg total_n", total_n)
    return total_sq / (total_n + 1e-8)




In [ ]:
# ----
# constructors (keep dimensions in sync)
# ----
# constructor test from a scratch runtime (commented for normal run)
# model0 = new_transformer(None, np.zeros(len(brain_cols), dtype=np.float32))
# print('params', sum(p.numel() for p in model0.parameters()))
# del model0
# torch.cuda.empty_cache()
# d_model needs to be a multiple of n_heads

def new_transformer(pca, mean_feat):
    return MultiTaskTransformer(
        num_mice=num_mice, num_days=num_days,
        brain_dim=(pca.n_components_ if pca else mean_feat.shape[0]),
        num_bins=NUM_BINS,
        d_model=cfg.d_model, n_heads=cfg.n_heads, n_layers=cfg.n_layers,
        d_ff=cfg.d_ff, dropout=cfg.dropout, meta_dim=cfg.meta_dim, tmax=cfg.tmax,
    ).to(device)


def new_conv_lstm(pca, mean_feat):
    return MultiTaskConv1dLSTM(
        num_mice=num_mice, num_days=num_days,
        brain_dim=(pca.n_components_ if pca else mean_feat.shape[0]),
        num_bins=NUM_BINS,
        hidden_dim=256, lstm_layers=3, dropout=cfg.dropout,
        meta_dim=cfg.meta_dim, tmax=cfg.tmax,
    ).to(device)


def new_resgru_attn(pca, mean_feat):
    return MultiTaskResGRUAttn(
        num_mice=num_mice, num_days=num_days,
        brain_dim=(pca.n_components_ if pca else mean_feat.shape[0]),
        num_bins=NUM_BINS,
        hidden_dim=cfg.resgru_hidden, gru_layers=cfg.resgru_gru_layers,
        n_heads=cfg.resgru_n_heads,
        dropout=cfg.dropout, meta_dim=cfg.meta_dim, tmax=cfg.tmax,
    ).to(device)


def new_tcn(pca, mean_feat):
    return MultiTaskTCN(
        num_mice=num_mice, num_days=num_days,
        brain_dim=(pca.n_components_ if pca else mean_feat.shape[0]),
        num_bins=NUM_BINS,
        n_channels=cfg.tcn_channels, kernel_size=cfg.tcn_kernel_size,
        n_blocks=cfg.tcn_n_blocks,
        dropout=cfg.dropout, meta_dim=cfg.meta_dim, tmax=cfg.tmax,
    ).to(device)




In [ ]:
# ------------------
# prediction helper, seq models
# ------------------
@torch.no_grad()
# predicting the diff head looked tempting; not used
# pred = reg_out + cfg.diff_weight * torch.cumsum(diff_out, dim=1)
# pred = expectation_decode(cls_out)
# print('reg min/max', reg_out.min().item(), reg_out.max().item())
# print('tta passes', n_tta)
# why did the cls decoder collapse most rows near the low bins??

def predict_seq_model(model, loader, items, use_tta=False, n_tta=5):
    """Predict with a single sequence model. Returns id_to_pred dict."""
    model.eval()
    all_preds = []
    n_passes = n_tta if use_tta else 1
    for tta_i in range(n_passes):
        preds_tta = []
        for batch in loader:
            X = batch['X'].to(device)
            if tta_i > 0:
                X = X + torch.randn_like(X) * 0.01
                # print("X??", type(X), str(X)[:80])
            time = batch['time'].to(device)
            mask = batch['mask'].to(device)
            mouse = batch['mouse'].to(device)
            pair = batch['pair'].to(device)
            reg_out, _, _, _ = model(X, time, mouse, pair, mask, use_causal=False)
            preds_tta.append(reg_out.cpu().numpy())
        all_preds.append(np.concatenate(preds_tta, axis=0))

    preds = np.mean(all_preds, axis=0)
    id_to_pred = {}
    # print("id_to_pred", getattr(id_to_pred, "shape", type(id_to_pred).__name__))  # ここ確認
    for i, it in enumerate(items):
        for rid, val in zip(it['row_ids'], preds[i, :len(it['row_ids'])]):
            id_to_pred[rid] = float(val)
    return id_to_pred


@torch.no_grad()
def predict_seq_model_oof(model, loader, items, use_tta=False, n_tta=5):
    """Predict OOF (validation) with a sequence model. Returns id_to_pred dict."""
    return predict_seq_model(model, loader, items, use_tta=use_tta, n_tta=n_tta)




In [ ]:
# ----
# fold loaders
# ----
# old attempt to speed things up with workers
# train_loader = DataLoader(train_ds, batch_size=cfg.batch_size,
#                           shuffle=True, num_workers=2, pin_memory=True)
# val_loader = DataLoader(val_ds, batch_size=cfg.batch_size,
#                         shuffle=False, num_workers=2)
# print('train batches', len(train_loader), 'val batches', len(val_loader))
# Colab worker hangs sometimes, left default

def make_loaders_for_fold(train_indices, val_indices, seed):
    train_items = [train_items_all[i] for i in train_indices]
    # print("train_items count", len(train_items))
    val_items = [train_items_all[i] for i in val_indices]

    all_items_combined = train_items_all + test_items
    # print("all_items_combined count", len(all_items_combined))
    mean_feat, std_feat, pca, C_feat = fit_preprocessing(
        all_items_combined, use_pca=cfg.use_pca, pca_components=cfg.pca_components, seed=seed
    )

    train_ds = LeverDataset(train_items, mean_feat, std_feat, pca,
                            tmax=cfg.tmax, with_target=True, augment=True)
    val_ds = LeverDataset(val_items, mean_feat, std_feat, pca,
                          tmax=cfg.tmax, with_target=True, augment=False)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size,
                              shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size,
                            shuffle=False, num_workers=2, pin_memory=True)
    test_ds = LeverDataset(test_items, mean_feat, std_feat, pca,
                           tmax=cfg.tmax, with_target=False, augment=False)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size,
                             shuffle=False, num_workers=2, pin_memory=True)
    return train_items, val_items, train_loader, val_loader, test_loader, mean_feat, std_feat, pca




In [ ]:
#
# train loop for the sequence nets
#
def train_neural_model(model, train_loader, val_loader, run_label,
                       epochs=None, lr=None, patience=12):
    """Generic training loop for any multi-task sequence model."""
    # print("passed here")  # optimizer/scheduler check
    if epochs is None:
        epochs = cfg.epochs
    if lr is None:
        lr = cfg.lr

    opt = torch.optim.AdamW(model.parameters(), lr=lr,
                             weight_decay=cfg.weight_decay, betas=(0.9, 0.999))
    total_steps = epochs * len(train_loader)
    warmup_steps = int(cfg.warmup_ratio * total_steps)

    def lr_lambda(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

    best_val_mse = float('inf')
    best_state = None
    bad_epochs = 0
    # print("dbg bad_epochs", bad_epochs)

    # optimizer experiments. disabled, these silently changed CV a lot
    # opt = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0.9)
    # opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
    # sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3,
    #                                            steps_per_epoch=len(train_loader), epochs=epochs)
    # print('warmup steps', warmup_steps, 'total steps', total_steps)
    # print('optimizer', opt.param_groups[0]['lr'])
    # gradient exploding??? should check unclipped norm

    for epoch in range(1, epochs + 1):
        model.train()
        train_regs = []

        for batch in train_loader:
            X = batch['X'].to(device)
            time = batch['time'].to(device)
            mask = batch['mask'].to(device)
            mouse = batch['mouse'].to(device)
            pair = batch['pair'].to(device)
            y = batch['y'].to(device)
            y_diff = batch['y_diff'].to(device)
            y_diff2 = batch['y_diff2'].to(device)

            # Augmentation
            soft_targets = None
            aug_batch = {
                'X': X, 'time': time, 'mask': mask,
                'mouse': mouse, 'pair': pair,
                'y': y, 'y_diff': y_diff, 'y_diff2': y_diff2,
            }

            r = np.random.rand()
            if r < cfg.mixup_prob and cfg.mixup_alpha > 0:
                aug_batch, lam, perm = apply_mixup(aug_batch, alpha=cfg.mixup_alpha)
                soft_targets = make_soft_targets(aug_batch['y'], cfg.soft_ce_sigma)
            elif r < cfg.mixup_prob + cfg.cutmix_prob:
                aug_batch, lam, perm = apply_cutmix(aug_batch, cfg)
                soft_targets = make_soft_targets(aug_batch['y'], cfg.soft_ce_sigma)

            X = aug_batch['X']
            time_t = aug_batch['time']
            mask_t = aug_batch['mask']
            y = aug_batch['y']
            y_diff = aug_batch['y_diff']
            # print("y_diff??", type(y_diff), str(y_diff)[:80])
            y_diff2 = aug_batch['y_diff2']

            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                reg_out, diff_out, diff2_out, cls_out = forward_with_embed_mixing(
                    model, X, time_t, mouse, pair, mask_t, aug_batch)
                total, l_reg, _, _, _ = multitask_loss(
                    reg_out, diff_out, diff2_out, cls_out,
                    y, y_diff, y_diff2, mask_t, cfg, soft_targets=soft_targets)

            scaler.scale(total).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(opt)
            scaler.update()
            sched.step()

            train_regs.append(l_reg.item())

        val_mse = evaluate_model_mse(model, val_loader)
        # overfitting check, only for the occasional local run
        # print(f'epoch={epoch}, mean reg loss={np.mean(train_regs):.6f}')
        # print('best state?', best_state is not None)
        # print('lr', opt.param_groups[0]['lr'])
        # if np.isnan(val_mse): raise RuntimeError('nan val')

        # print("val_mse?", val_mse.item() if hasattr(val_mse, "item") else val_mse)

        if epoch % 10 == 0 or epoch <= 3:
            print(f"  [{run_label}] Ep {epoch:03d} | train_reg={np.mean(train_regs):.6f} | "
                  f"val_MSE={val_mse:.6f} | best={best_val_mse:.6f}")

        if val_mse < best_val_mse - 1e-6:
            best_val_mse = val_mse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            # print("best_state??", type(best_state), str(best_state)[:80])
            bad_epochs = 0
        else:
            bad_epochs += 1
            # print("dbg bad_epochs", bad_epochs)
            if bad_epochs >= patience:
                print(f"  [{run_label}] Early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    print(f"  [{run_label}] Best val MSE: {best_val_mse:.6f}")
    return model, best_val_mse




In [ ]:
#  . . .
# LGBM branch
#  . . .
def train_lgb_branch(train_feat, val_feat, test_feat, run_label):
    """Train LightGBM on flat row-level features. Accepts FlatFeatureSet objects."""
    # print("lgb matrices", train_feat.matrix.shape, val_feat.matrix.shape)
    import lightgbm as lgb

    X_train = train_feat.matrix.astype(np.float32)
    y_train = train_feat.targets.astype(np.float32)
    X_val = val_feat.matrix.astype(np.float32)
    y_val = val_feat.targets.astype(np.float32)
    X_test = test_feat.matrix.astype(np.float32)

    dtrain = lgb.Dataset(X_train, label=y_train, feature_name=train_feat.col_names, free_raw_data=False)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain, free_raw_data=False)

    callbacks = [
        lgb.early_stopping(cfg.lgb_early_stopping, verbose=True),
        lgb.log_evaluation(200),
    ]

    # earlier scikit API version (disabled to avoid different early stopping behavior)
    # mdl = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.03,
    #                         num_leaves=31, subsample=0.9)
    # mdl.fit(X_train, y_train, eval_set=[(X_val, y_val)])
    # print('importance', sorted(zip(train_feat.col_names, mdl.feature_importances_),
    #                            key=lambda p: -p[1])[:20])
    # print('train NA?', np.isnan(X_train).sum())

    model = lgb.train(
        cfg.lgb_params, dtrain,
        num_boost_round=cfg.lgb_num_rounds,
        valid_sets=[dval],
        valid_names=['val'],
        callbacks=callbacks,
    )

    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)

    val_mse = np.mean((val_pred - y_val) ** 2)
    # print("val_mse?", val_mse.item() if hasattr(val_mse, "item") else val_mse)
    print(f"  [{run_label} LGB] val MSE: {val_mse:.6f} (best iter: {model.best_iteration})")

    val_id_to_pred = dict(zip(val_feat.ids, val_pred.astype(float)))
    test_id_to_pred = dict(zip(test_feat.ids, test_pred.astype(float)))
    # print("test_id_to_pred", getattr(test_id_to_pred, "shape", type(test_id_to_pred).__name__))

    return val_id_to_pred, test_id_to_pred, val_mse




In [ ]:
# ---
# MLP branch
# ---
def train_mlp_branch(train_feat, val_feat, test_feat, run_label):
    """Train pure MLP on flat row-level features. Accepts FlatFeatureSet objects."""

    # print("MLP branch, passed here")
    X_train = train_feat.matrix.astype(np.float32)
    y_train = train_feat.targets.astype(np.float32)
    X_val = val_feat.matrix.astype(np.float32)
    y_val = val_feat.targets.astype(np.float32)
    X_test = test_feat.matrix.astype(np.float32)

    # normalize here too... yes, this is for the MLP only
    feat_mean = np.nanmean(X_train, axis=0)
    feat_std = np.nanstd(X_train, axis=0) + 1e-8
    # print("feat_std", getattr(feat_std, "shape", type(feat_std).__name__))  # optional
    X_train = np.nan_to_num((X_train - feat_mean) / feat_std, 0.0)
    X_val = np.nan_to_num((X_val - feat_mean) / feat_std, 0.0)
    X_test = np.nan_to_num((X_test - feat_mean) / feat_std, 0.0)

    input_dim = X_train.shape[1]
    model = PureMLP(input_dim, cfg.mlp_hidden_dims, cfg.mlp_dropout).to(device)
    print(f"  [{run_label} MLP] params: {sum(p.numel() for p in model.parameters()):,}, "
          f"features: {input_dim}")

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.mlp_lr, weight_decay=1e-4)
    # print("opt", type(opt).__name__)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.mlp_epochs)

    X_train_t = torch.from_numpy(X_train).to(device)
    y_train_t = torch.from_numpy(y_train).to(device)
    X_val_t = torch.from_numpy(X_val).to(device)
    y_val_t = torch.from_numpy(y_val).to(device)
    X_test_t = torch.from_numpy(X_test).to(device)

    best_val_mse = float('inf')
    # print("best_val_mse?", best_val_mse.item() if hasattr(best_val_mse, "item") else best_val_mse)
    best_state = None
    patience = 15
    bad = 0
    # print("bad??", type(bad), str(bad)[:80])
    bs = cfg.mlp_batch_size

    # stale MLP baseline with linear ridge, sometimes useful sanity check
    # from sklearn.linear_model import Ridge as _Ridge
    # m0 = _Ridge(alpha=10.0)
    # m0.fit(X_train, y_train)
    # print('linear baseline val', ((m0.predict(X_val) - y_val) ** 2).mean())
    # del m0
    # cfg.mlp_dropout = 0.4  # tried more regularization

    for epoch in range(1, cfg.mlp_epochs + 1):
        model.train()
        perm = torch.randperm(len(X_train_t))
        # print("perm??", type(perm), str(perm)[:80])
        epoch_loss = 0.0
        n_batches = 0
        for i in range(0, len(X_train_t), bs):
            idx = perm[i:i+bs]
            xb = X_train_t[idx]
            yb = y_train_t[idx]
            opt.zero_grad()
            pred = model(xb)
            loss = F.mse_loss(pred, yb)
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
            n_batches += 1
        sched.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_t)
            val_mse = F.mse_loss(val_pred, y_val_t).item()

        if epoch % 20 == 0 or epoch <= 3:
            print(f"  [{run_label} MLP] Ep {epoch:03d} | train={epoch_loss/n_batches:.6f} | "
                  f"val={val_mse:.6f} | best={best_val_mse:.6f}")

        if val_mse < best_val_mse - 1e-6:
            best_val_mse = val_mse
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print(f"  [{run_label} MLP] Early stop at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    model.to(device).eval()

    with torch.no_grad():
        val_pred = model(X_val_t).cpu().numpy()
        test_pred = model(X_test_t).cpu().numpy()

    print(f"  [{run_label} MLP] Best val MSE: {best_val_mse:.6f}")

    val_id_to_pred = dict(zip(val_feat.ids, val_pred.astype(float)))
    test_id_to_pred = dict(zip(test_feat.ids, test_pred.astype(float)))

    del model, X_train_t, y_train_t, X_val_t, y_val_t, X_test_t
    gc.collect()
    torch.cuda.empty_cache()

    return val_id_to_pred, test_id_to_pred, best_val_mse




In [ ]:
#
# all branches for one fold
# ------------------
def train_one_fold_all_branches(seed, fold_idx, train_indices, val_indices):
    """
    Train all 6 branches for one fold.
    Returns OOF predictions and test predictions per branch, plus val MSEs.
    """
    # print("fold sizes", len(train_indices), len(val_indices))
    seed_everything(seed + fold_idx)
    run_label = f"s{seed}_f{fold_idx}"

    train_items_fold = [train_items_all[i] for i in train_indices]
    # print("train_items_fold count", len(train_items_fold))
    val_items_fold = [train_items_all[i] for i in val_indices]

    # sequence preprocessing (fit on train fold)
    all_items_combined = train_items_all + test_items
    mean_feat, std_feat, pca, C_feat = fit_preprocessing(
        all_items_combined, use_pca=cfg.use_pca, pca_components=cfg.pca_components, seed=seed
    )

    train_ds = LeverDataset(train_items_fold, mean_feat, std_feat, pca,
                            tmax=cfg.tmax, with_target=True, augment=True)
    val_ds = LeverDataset(val_items_fold, mean_feat, std_feat, pca,
                          tmax=cfg.tmax, with_target=True, augment=False)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size,
                              shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size,
                            shuffle=False, num_workers=2, pin_memory=True)
    test_ds = LeverDataset(test_items, mean_feat, std_feat, pca,
                           tmax=cfg.tmax, with_target=False, augment=False)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size,
                             shuffle=False, num_workers=2, pin_memory=True)

    # Collect results: branch_name -> (oof_id_to_pred, test_id_to_pred, val_mse)
    results = {}

    # ==== BRANCH 1: Transformer ====
    # during smoke tests I used only one branch to catch shape problems
    # cfg.enable_transformer = True
    # cfg.enable_conv_lstm = False
    # cfg.enable_resgru_attn = False
    # cfg.enable_tcn = False
    # cfg.enable_lgbm = False
    # cfg.enable_mlp = False
    # print('flags', cfg.enable_transformer, cfg.enable_lgbm)
    # not changing flags from inside this fold(!)

    if cfg.enable_transformer:
        print(f"\n--- [{run_label}] BRANCH 1: Transformer ---")
        m1 = new_transformer(pca, mean_feat)
        # print("m1??", type(m1), str(m1)[:80])
        print(f"  Params: {sum(p.numel() for p in m1.parameters()):,}")
        m1, v1 = train_neural_model(m1, train_loader, val_loader, f"{run_label}_TF")
        oof1 = predict_seq_model(m1, val_loader, val_items_fold, use_tta=cfg.use_tta, n_tta=3)
        test1 = predict_seq_model(m1, test_loader, test_items, use_tta=cfg.use_tta, n_tta=5)
        # print("test1", getattr(test1, "shape", type(test1).__name__))
        results['transformer'] = (oof1, test1, v1)
        del m1; gc.collect(); torch.cuda.empty_cache()

    # ==== BRANCH 2: Conv1D-LSTM ====
    if cfg.enable_conv_lstm:
        print(f"\n--- [{run_label}] BRANCH 2: Conv1D-LSTM ---")
        m2 = new_conv_lstm(pca, mean_feat)
        # print("m2??", type(m2), str(m2)[:80])
        print(f"  Params: {sum(p.numel() for p in m2.parameters()):,}")
        m2, v2 = train_neural_model(m2, train_loader, val_loader, f"{run_label}_LSTM")
        oof2 = predict_seq_model(m2, val_loader, val_items_fold, use_tta=cfg.use_tta, n_tta=3)
        test2 = predict_seq_model(m2, test_loader, test_items, use_tta=cfg.use_tta, n_tta=5)
        # print("test2", getattr(test2, "shape", type(test2).__name__))
        results['conv_lstm'] = (oof2, test2, v2)
        del m2; gc.collect(); torch.cuda.empty_cache()

    # ==== BRANCH 3: ResNet-BiGRU-Attention ====
    if cfg.enable_resgru_attn:
        print(f"\n--- [{run_label}] BRANCH 3: ResNet-BiGRU-Attention ---")
        m3 = new_resgru_attn(pca, mean_feat)
        print(f"  Params: {sum(p.numel() for p in m3.parameters()):,}")
        m3, v3 = train_neural_model(m3, train_loader, val_loader, f"{run_label}_ResGRU")
        oof3 = predict_seq_model(m3, val_loader, val_items_fold, use_tta=cfg.use_tta, n_tta=3)
        test3 = predict_seq_model(m3, test_loader, test_items, use_tta=cfg.use_tta, n_tta=5)
        # print("test3", getattr(test3, "shape", type(test3).__name__))
        results['resgru_attn'] = (oof3, test3, v3)
        del m3; gc.collect(); torch.cuda.empty_cache()

    # ==== BRANCH 4: TCN ====
    if cfg.enable_tcn:
        print(f"\n--- [{run_label}] BRANCH 4: TCN ---")
        m4 = new_tcn(pca, mean_feat)
        print(f"  Params: {sum(p.numel() for p in m4.parameters()):,}")
        m4, v4 = train_neural_model(m4, train_loader, val_loader, f"{run_label}_TCN")
        oof4 = predict_seq_model(m4, val_loader, val_items_fold, use_tta=cfg.use_tta, n_tta=3)
        test4 = predict_seq_model(m4, test_loader, test_items, use_tta=cfg.use_tta, n_tta=5)
        results['tcn'] = (oof4, test4, v4)
        del m4; gc.collect(); torch.cuda.empty_cache()

    # flat features (skip if both branches disabled)
    if cfg.enable_lgbm or cfg.enable_mlp:
        print(f"\n--- [{run_label}] Building flat features (fast) ---")
        train_flat = build_flat_features_fast(train_items_fold,
                                              train_items_for_stats=train_items_fold,
                                              source='train')
        val_flat = build_flat_features_fast(val_items_fold,
                                            train_items_for_stats=train_items_fold,
                                            source='train')
        test_flat = build_flat_features_fast(test_items,
                                             train_items_for_stats=train_items_fold,
                                             source='test')
        print(f"  Flat features: train={train_flat.shape}, val={val_flat.shape}, test={test_flat.shape}")

        # ==== BRANCH 5: LightGBM ====
        if cfg.enable_lgbm:
            print(f"\n--- [{run_label}] BRANCH 5: LightGBM ---")
            oof5, test5, v5 = train_lgb_branch(train_flat, val_flat, test_flat, run_label)
            results['lgbm'] = (oof5, test5, v5)

        # ==== BRANCH 6: Pure MLP ====
        if cfg.enable_mlp:
            print(f"\n--- [{run_label}] BRANCH 6: Pure MLP ---")
            oof6, test6, v6 = train_mlp_branch(train_flat, val_flat, test_flat, run_label)
            results['mlp'] = (oof6, test6, v6)

        # Clean up
        del train_flat, val_flat, test_flat
        gc.collect()

    return results, train_indices, val_indices




In [ ]:
# ----
# run the folds + ridge at the end
# ----
# print(sample_sub.iloc[:3])  # idの順番確認
sample_sub = pd.read_csv(f"{cfg.data_path}/sample_submission.csv")
ids = sample_sub['id'].values

# ID order is NOT the sequence order. remember this.
all_train_ids = []
for it in train_items_all:
    all_train_ids.extend(it['row_ids'])
all_train_ids_set = set(all_train_ids)

# target lookup by row ID
train_id_to_target = {}
# print("train_id_to_target", getattr(train_id_to_target, "shape", type(train_id_to_target).__name__))
for it in train_items_all:
    for rid, yv in zip(it['row_ids'], it['y']):
        train_id_to_target[rid] = float(yv)

BRANCH_NAMES = []
# print("BRANCH_NAMES count", len(BRANCH_NAMES))
if cfg.enable_transformer: BRANCH_NAMES.append('transformer')
if cfg.enable_conv_lstm: BRANCH_NAMES.append('conv_lstm')
if cfg.enable_resgru_attn: BRANCH_NAMES.append('resgru_attn')
if cfg.enable_tcn: BRANCH_NAMES.append('tcn')
if cfg.enable_lgbm: BRANCH_NAMES.append('lgbm')
if cfg.enable_mlp: BRANCH_NAMES.append('mlp')
print(f"Enabled branches: {BRANCH_NAMES}")

# OOF arrays: branch -> id -> prediction
oof_all = {b: {} for b in BRANCH_NAMES}     # accumulated across folds
test_all = {b: [] for b in BRANCH_NAMES}     # list of dicts per fold
val_mses_all = {b: [] for b in BRANCH_NAMES}

base_seed = 42
run_count = 0
# print("run_count??", type(run_count), str(run_count)[:80])

# Colab落ちないで... このループ結構長い
# cfg.seed_count = 1  # debug only; set this BEFORE starting if needed
# smoke-test loop I was using before the actual sweep
# for seed_idx in range(1):
#     split_seed = base_seed
#     fold_splits = get_kfold_splits(len(train_items_all), n_folds=2, seed=split_seed)
#     train_indices, val_indices = fold_splits[0]
#     print(len(train_indices), len(val_indices))
#     r, _, _ = train_one_fold_all_branches(split_seed, 0, train_indices, val_indices)
#     print(r.keys())
# GPUが足りない... 6 branches × folds is ridiculous

for seed_idx in range(cfg.seed_count):
    seed = base_seed + seed_idx * 100
    fold_splits = get_kfold_splits(len(train_items_all), n_folds=cfg.n_folds, seed=seed)

    for fold_idx, (train_indices, val_indices) in enumerate(fold_splits):
        run_count += 1
        print("\n" + "=" * 70)
        print(f"RUN {run_count}/{cfg.seed_count * cfg.n_folds} | seed={seed}, fold={fold_idx+1}/{cfg.n_folds}")
        print(f"  Train: {len(train_indices)}, Val: {len(val_indices)}")
        print("=" * 70)

        results, _, _ = train_one_fold_all_branches(seed, fold_idx, train_indices, val_indices)

        for bname in BRANCH_NAMES:
            oof_pred, test_pred, val_mse = results[bname]

            # Accumulate OOF (each val sample appears exactly once per seed)
            for rid, val in oof_pred.items():
                if rid not in oof_all[bname]:
                    oof_all[bname][rid] = []
                oof_all[bname][rid].append(val)

            test_all[bname].append(test_pred)
            val_mses_all[bname].append(val_mse)

            print(f"  Branch {bname}: val_MSE={val_mse:.6f}")



In [ ]:
#  . . .
# how did each branch do?
#  . . .
print("\n" + "=" * 70)
print("PER-BRANCH SUMMARY (mean val MSE across folds):")
for bname in BRANCH_NAMES:
    mses = val_mses_all[bname]
    # print("mses?", mses.item() if hasattr(mses, "item") else mses)
    print(f"  {bname:20s}: mean={np.mean(mses):.6f} | "
          f"folds: {', '.join(f'{v:.6f}' for v in mses)}")

# =====
# OOF and test matrix for ridge
# =====
print("\n" + "=" * 70)
# print("passed here")
print("Building Ridge meta-learner...")

# 複数seedのOOFはここで平均
oof_ids_ordered = sorted(all_train_ids_set)
n_oof = len(oof_ids_ordered)
# print("n_oof??", type(n_oof), str(n_oof)[:80])
oof_matrix = np.zeros((n_oof, len(BRANCH_NAMES)), dtype=np.float32)
# print("oof_matrix", getattr(oof_matrix, "shape", type(oof_matrix).__name__))
oof_targets = np.zeros(n_oof, dtype=np.float32)

for i, rid in enumerate(oof_ids_ordered):
    oof_targets[i] = train_id_to_target[rid]
    for j, bname in enumerate(BRANCH_NAMES):
        if rid in oof_all[bname]:
            oof_matrix[i, j] = np.mean(oof_all[bname][rid])
        else:
            oof_matrix[i, j] = np.nan  # should not happen with full K-fold

# Check for NaN
nan_mask = np.isnan(oof_matrix).any(axis=1)
# print("nan_mask valid", nan_mask.sum().item() if hasattr(nan_mask, "sum") else nan_mask)
if nan_mask.any():
    print(f"WARNING: {nan_mask.sum()} OOF rows have NaN, dropping them for Ridge fit")
    oof_matrix = oof_matrix[~nan_mask]
    # print("oof_matrix", getattr(oof_matrix, "shape", type(oof_matrix).__name__))
    oof_targets = oof_targets[~nan_mask]
    oof_ids_clean = [oof_ids_ordered[i] for i in range(n_oof) if not nan_mask[i]]
else:
    oof_ids_clean = oof_ids_ordered

print(f"OOF matrix shape: {oof_matrix.shape}")

# Test: average across all folds×seeds
n_test = len(ids)
test_matrix = np.zeros((n_test, len(BRANCH_NAMES)), dtype=np.float32)
for j, bname in enumerate(BRANCH_NAMES):
    preds_stacked = np.zeros((len(test_all[bname]), n_test), dtype=np.float32)
    for k, d in enumerate(test_all[bname]):
        preds_stacked[k] = np.array([d[rid] for rid in ids], dtype=np.float32)
    test_matrix[:, j] = preds_stacked.mean(axis=0)

print(f"Test matrix shape: {test_matrix.shape}")

# ridge alpha -- had 10 in an earlier attempt
from sklearn.linear_model import RidgeCV

# other ways to blend that I scribbled down
# weights = np.ones(X_meta.shape[1]) / X_meta.shape[1]
# weighted_oof = X_meta @ weights
# print('fixed mean', np.mean((weighted_oof - y_meta) ** 2))
# from scipy.optimize import nnls
# coef, _ = nnls(X_meta, y_meta)
# coef = coef / max(coef.sum(), 1e-9)
# print('nonnegative weights', coef)
# ridge = Ridge(alpha=100.0, positive=True)  # old attempt
# print('meta NaN columns', np.isnan(X_meta).sum(axis=0))

ridge = RidgeCV(alphas=cfg.ridge_alphas, scoring='neg_mean_squared_error')
ridge.fit(oof_matrix, oof_targets)
print(f"\nRidge best alpha: {ridge.alpha_}")
print(f"Ridge coefficients: {dict(zip(BRANCH_NAMES, ridge.coef_))}")
print(f"Ridge intercept: {ridge.intercept_:.6f}")

# OOF prediction quality
oof_ridge_pred = ridge.predict(oof_matrix)
oof_ridge_mse = np.mean((oof_ridge_pred - oof_targets) ** 2)
print(f"Ridge OOF MSE: {oof_ridge_mse:.6f}")

# Also show simple average OOF MSE for comparison
oof_simple_avg = oof_matrix.mean(axis=1)
oof_simple_mse = np.mean((oof_simple_avg - oof_targets) ** 2)
print(f"Simple average OOF MSE: {oof_simple_mse:.6f}")

# Per-branch OOF MSE
for j, bname in enumerate(BRANCH_NAMES):
    branch_mse = np.mean((oof_matrix[:, j] - oof_targets) ** 2)
    print(f"  {bname:20s} OOF MSE: {branch_mse:.6f}")

# Pairwise prediction correlation matrix (OOF)
print(f"\n{'='*70}")
print("BRANCH PAIRWISE PREDICTION CORRELATIONS (OOF):")
print(f"{'='*70}")
n_branches = len(BRANCH_NAMES)
corr_matrix = np.corrcoef(oof_matrix.T)  # (n_branches, n_branches)
# print("corr_matrix", getattr(corr_matrix, "shape", type(corr_matrix).__name__))
# Header
header = f"{'':20s}" + "".join(f"{b:>14s}" for b in BRANCH_NAMES)
print(header)
for i, bname_i in enumerate(BRANCH_NAMES):
    row_str = f"{bname_i:20s}" + "".join(f"{corr_matrix[i, j]:14.4f}" for j in range(n_branches))
    print(row_str)

# finally the test predictions
test_ridge_pred = ridge.predict(test_matrix)
# print("test_ridge_pred", type(test_ridge_pred).__name__)

sub = sample_sub.copy()
sub['lever'] = test_ridge_pred.astype(np.float32)
sub.to_csv("submission_diverse_ensemble.csv", index=False)
print(f"\nSaved: submission_diverse_ensemble.csv")

# Also save simple average as backup
sub_avg = sample_sub.copy()
sub_avg['lever'] = test_matrix.mean(axis=1).astype(np.float32)
sub_avg.to_csv("submission_simple_average.csv", index=False)
print(f"Saved: submission_simple_average.csv")

# Save per-branch test predictions
detailed_df = pd.DataFrame({'id': ids})
for j, bname in enumerate(BRANCH_NAMES):
    detailed_df[f'lever_{bname}'] = test_matrix[:, j]
detailed_df['lever_ridge'] = test_ridge_pred
detailed_df['lever_simple_avg'] = test_matrix.mean(axis=1)
detailed_df.to_csv("test_detailed_all_branches.csv", index=False)
print(f"Saved: test_detailed_all_branches.csv")

# Save OOF predictions
oof_df = pd.DataFrame({'id': oof_ids_clean})
for j, bname in enumerate(BRANCH_NAMES):
    oof_df[f'pred_{bname}'] = oof_matrix[:, j]
oof_df['pred_ridge'] = oof_ridge_pred
oof_df['target'] = oof_targets
# old output debugging, leave it off for submission run
# display(oof_df.head(10))
# display(submission.head(10))
# assert submission['id'].tolist() == sample_sub['id'].tolist()
# print('pred quantiles', submission['lever'].quantile([0, .01, .5, .99, 1]))
# submission.to_csv('submission_temp.csv', index=False)

oof_df.to_csv("oof_all_branches.csv", index=False)
print(f"Saved: oof_all_branches.csv")

print(f"\n{'='*70}")
print(f"FINAL SUMMARY")
print(f"{'='*70}")
print(f"Branches: {BRANCH_NAMES}")
print(f"Ridge meta-learner alpha: {ridge.alpha_}")
print(f"Ridge OOF MSE: {oof_ridge_mse:.6f}")
print(f"Simple avg OOF MSE: {oof_simple_mse:.6f}")
print(f"Improvement from Ridge: {oof_simple_mse - oof_ridge_mse:.6f}")
print(f"CV: {cfg.n_folds}-fold x {cfg.seed_count} seeds")

# Multi-task v3 / another run
Same input data, different model setup. This runs AFTER the multi-branch version.


In [ ]:
# -*- coding: utf-8 -*-
"""
JOAI2026 - Multi-Task Learning v3 (5-Fold CV + LightGBM + Ridge Meta)
Three NN heads sharing one backbone:
  1. Regression head  → predicts raw lever          (MSE,      weight=1.0)
  2. Diff head        → predicts lever[t]-lever[t-1] (MSE,      weight=1.0)
  3. Classification   → predicts lever bin           (Soft CE,  weight=0.05)

+ LightGBM branch with extensive feature engineering
+ Ridge regression meta-learner to combine NN ensemble + LightGBM

Changed from MCCV to 5-fold CV with seed bagging:
  - Each seed produces a different 5-fold split (shuffled by seed).
  - All 5 folds are trained per seed.
  - Preprocessing (normalization, optional PCA) is fit transductively
    on all train + test data.
  - Final predictions are averaged across all seeds × folds.

Inference uses regression head only.
Early stopping on val regression MSE (the actual competition metric).
"""

import os, math, random, gc
import numpy as np
import pandas as pd
from scipy import signal
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import lightgbm as lgb

# kagglehub auth may pop up if not logged in already
# second attempt / separate run, different feature code below
# %pip install -q kagglehub
# from google.colab import drive
# drive.mount('/content/drive')
# print('torch', torch.__version__)
# print('gpu', torch.cuda.get_device_properties(0))
# TODO: do NOT overwrite results of previous run accidentally

import kagglehub
kagglehub.login()

joai_competition_2026_path = kagglehub.competition_download('joai-competition-2026')
# print("joai_competition_2026_path??", type(joai_competition_2026_path), str(joai_competition_2026_path)[:80])  # was wrong before
print('Data source import complete.')



In [ ]:
# =====
# 34 bins, keep the order the same (decode uses this)
# =====
BIN_VALUES = np.array([
    0.00032143933, 0.062720512, 0.070853584, 0.092515556,
    0.11706034, 0.14200391, 0.22216938, 0.40775788,
    0.6252751, 0.8704624,
    1.0827121, 1.2484384, 1.4266728, 1.6175482,
    1.8090109, 2.0069659, 2.2113256, 2.4365339,
    2.6635003, 2.8937493, 3.0947864, 3.3039802,
    3.4982202, 3.6773475, 3.8330718, 3.9491331,
    4.0245861, 4.0776984, 4.1019569, 4.1453609,
    4.1702872, 4.2089919, 4.2409913, 4.2878157,
], dtype=np.float32)

NUM_BINS = len(BIN_VALUES)
print(f"Number of bins: {NUM_BINS}")


def lever_to_bin(lever_values):
    """Snap continuous lever values to nearest bin index."""
    flat = lever_values.ravel()
    dists = np.abs(flat[:, None] - BIN_VALUES[None, :])
    indices = dists.argmin(axis=1)
    # print("indices count", len(indices))
    return indices.reshape(lever_values.shape)


def bin_to_lever(bin_indices):
    return BIN_VALUES[bin_indices]


# bin lookup alternatives (left disabled)
# bin_idx = np.digitize(y, (BIN_VALUES[:-1] + BIN_VALUES[1:]) / 2)
# decoded = BIN_VALUES[bin_idx]
# print('difference vs current', np.mean((decoded - y) ** 2))
# print(BIN_VALUES[:8], BIN_VALUES[-8:])

def expectation_decode(logits_tensor, temperature=1.0):
    probs = F.softmax(logits_tensor / temperature, dim=-1)
    bin_vals = torch.tensor(BIN_VALUES, device=logits_tensor.device, dtype=logits_tensor.dtype)
    # print("bin_vals", getattr(bin_vals, "shape", type(bin_vals).__name__))
    return (probs * bin_vals).sum(dim=-1)




In [ ]:
# ---
# seeding stuff
# ---
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# seed_everything(123)  # tried a few seeds; revert for runs
# torch.backends.cudnn.benchmark = True   # slightly faster but results shift
# seed_everything(3407)
# print('rng check', np.random.rand(3), torch.rand(3))
# last time I had forgotten to seed CUDA...

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print("device??", type(device), str(device)[:80])
print("device:", device)



In [ ]:
# ------------------
# settings / things I kept tweaking
# ------------------
class Config:
    data_path = joai_competition_2026_path

    # Model architecture
    d_model = 128
    n_heads = 4
    n_layers = 4
    d_ff = 256
    dropout = 0.15
    meta_dim = 64
    # print("dbg meta_dim", meta_dim)

    # Training
    # batch_size = 32  # fits slower
    batch_size = 64
    # epochs = 25  # short comparison
    epochs = 80
    # lr = 3e-4  # tried this
    lr = 2e-4
    weight_decay = 5e-3
    warmup_ratio = 0.1
    grad_clip = 0.5

    # Multi-task loss weights
    w_regression = 1.0
    w_diff = 1.0
    w_diff2 = 1.0
    w_classification = 0.0
    label_smoothing = 0.05      # only used as fallback; soft CE is primary

    # Soft-target classification
    soft_ce_sigma = 0.15        # Gaussian σ in lever-value space for soft targets

    # Augmentation
    mixup_alpha = 0.4           # Beta distribution param; 0 = disabled
    mixup_prob = 0.5            # probability of applying mixup per batch
    cutmix_prob = 0.2           # probability of applying cutmix per batch

    # Pseudo-labeling (multi-stage)
    # use_pseudo = False  # first baseline
    use_pseudo = True
    # print("use_pseudo??", type(use_pseudo), str(use_pseudo)[:80])  # ここ確認
    n_ensemble = 5
    pseudo_conf_percentile = 70
    pseudo_lr_ratio = 0.05
    pseudo_epochs = 15
    pseudo_patience = 3
    # print("pseudo_patience??", type(pseudo_patience), str(pseudo_patience)[:80])
    pseudo_stages = 1           # number of pseudo-labeling rounds
    pseudo_percentile_schedule = [70]  # progressively include more data
    pseudo_lr_decay_per_stage = 0.7             # decay LR each stage

    # Other
    # use_tta = False  # faster check
    use_tta = True
    use_pca = False
    pca_components = 96
    seed_count = 1              # number of fold-split seeds (different CV splits)
    n_folds = 5                 # K in K-fold CV
    nn_seed_count = 2           # number of NN weight-init seeds per fold (averaged before Ridge)
    drop_worst_ratio = 0      # fraction of (seed×fold) runs to drop
    # print("dbg drop_worst_ratio", drop_worst_ratio)

    # Dual model
    use_dual_models = True
    ensemble_weight = 0.5

    # Classification
    num_bins = NUM_BINS

    # Sequence
    tmax = 40

    # LightGBM
    use_lgbm = True
    lgbm_n_seeds = 2            # number of LightGBM seeds for bagging

    # Ridge meta-learner
    # ridge_alpha = 0.1  # overfit
    ridge_alpha = 1.0

# print(vars(cfg))  # did I actually use the new settings??
# cfg.epochs = 2  # tiny smoke test only
# second run used 2 networks + LGBM, not all six branches
# cfg = Config()
# cfg.use_dual_models = False   # one-net baseline
# cfg.use_pseudo = False
# cfg.use_tta = False
# cfg.n_folds = 2
# cfg.epochs = 3
# cfg.seed_count = 1
# cfg.nn_seed_count = 1
# cfg.lgbm_n_seeds = 1
# cfg.ridge_alpha = 10.0
# do not change cfg after the CV loop has started

cfg = Config()



In [ ]:
# ------------------
# get the kaggle files
# ------------------
# print("passed here")
print("Loading data...")
# print(train_df.head())
# print(train_df.isna().sum().sort_values(ascending=False).head(10))
train_df = pd.read_csv(f"{cfg.data_path}/train.csv")
# print("test rows", len(test_df))
test_df = pd.read_csv(f"{cfg.data_path}/test.csv")
print(f"Train: {train_df.shape}, Test: {test_df.shape}")

# Sanity check: binning floor error
# EDA leftovers; remove the display calls for unattended Colab run
# display(train_df.sample(5, random_state=42))
# print('unique mice', train_df['mouse_id'].nunique())
# print('days', train_df['day_n'].value_counts().head())
# print('id overlap', set(train_df.id) & set(test_df.id))
# print('lever stats', train_df['lever'].describe())
# print('time dtype', train_df['time'].dtype)
# train_df.plot(x='time', y='lever', kind='scatter')

raw_lever = train_df['lever'].values.astype(np.float32)
binned_lever = bin_to_lever(lever_to_bin(raw_lever))
sanity_mse = np.mean((raw_lever - binned_lever) ** 2)
print(f"\nSANITY CHECK: MSE(raw_lever, binned_lever) = {sanity_mse:.10f}")

#  . . .
# features etc
#  . . .
ID_COL = "id"
SAMPLE_COL = "sample_id"
MOUSE_COL = "mouse_id"
DAY_COL = "day_n"
TIME_COL = "time"
TARGET_COL = "lever"
# print("TARGET_COL", getattr(TARGET_COL, "shape", type(TARGET_COL).__name__))

exclude = {ID_COL, SAMPLE_COL, MOUSE_COL, DAY_COL, TIME_COL, TARGET_COL}
brain_cols = [c for c in train_df.columns if c not in exclude]
# print("brain_cols count", len(brain_cols))
print(f"Brain channels: {len(brain_cols)}")

train_df[MOUSE_COL] = train_df[MOUSE_COL].astype(str).str.strip()
test_df[MOUSE_COL] = test_df[MOUSE_COL].astype(str).str.strip()

all_mice = sorted(set(train_df[MOUSE_COL]) | set(test_df[MOUSE_COL]))
mouse2idx = {m: i for i, m in enumerate(all_mice)}
all_days = sorted(set(train_df[DAY_COL]) | set(test_df[DAY_COL]))
day2idx = {d: i for i, d in enumerate(all_days)}
num_mice = len(mouse2idx)
num_days = len(day2idx)
num_pairs = num_mice * num_days
print(f"Mice: {num_mice}, Days: {num_days}, Pairs: {num_pairs}")



In [ ]:
# ----
# per mouse norm. uses train+test!
# (transductive: uses all train + test rows)
# Applied in-place before feature engineering.
# ~~~
# print("norm check", train_df[brain_cols].describe())
print("Applying per-mouse brain feature normalization (train + test)...")
# normalization experiments (DON'T mix these with real one)
# all_df = pd.concat([train_df, test_df], ignore_index=True)
# global_mu = all_df[brain_cols].mean(axis=0)
# global_sd = all_df[brain_cols].std(axis=0).clip(lower=1e-6)
# all_df[brain_cols] = (all_df[brain_cols] - global_mu) / global_sd
# per-day normalization made the small day groups super noisy
# print('mouse normalization?')
# これ、testを使っているのでtransductive。ちゃんと覚えておく

combined_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)
mouse_brain_stats = {}
for mouse, grp in combined_df.groupby(MOUSE_COL):
    m = grp[brain_cols].values.astype(np.float32).mean(axis=0)
    s = np.maximum(grp[brain_cols].values.astype(np.float32).std(axis=0), 1e-6)
    mouse_brain_stats[mouse] = (m, s)

for mouse, (m, s) in mouse_brain_stats.items():
    train_mask = train_df[MOUSE_COL] == mouse
    test_mask  = test_df[MOUSE_COL]  == mouse
    train_df.loc[train_mask, brain_cols] = (
        train_df.loc[train_mask, brain_cols].values - m) / s
    test_df.loc[test_mask, brain_cols] = (
        test_df.loc[test_mask, brain_cols].values - m) / s

print("Per-mouse normalization complete.")
del combined_df


def compute_advanced_features(X_raw):
    T, C = X_raw.shape
    features = [X_raw]
    if T > 1:
        diff1 = np.zeros_like(X_raw)
        # print("diff1??", type(diff1), str(diff1)[:80])  # TODO?
        diff1[1:] = X_raw[1:] - X_raw[:-1]
        features.append(diff1)
    if T > 2:
        diff2 = np.zeros_like(X_raw)
        # print("diff2??", type(diff2), str(diff2)[:80])
        diff2[2:] = X_raw[2:] - 2*X_raw[1:-1] + X_raw[:-2]
        features.append(diff2)
    if T >= 5:
        rolling_mean = np.zeros_like(X_raw)
        # print("rolling_mean", getattr(rolling_mean, "shape", type(rolling_mean).__name__))
        rolling_std = np.zeros_like(X_raw)
        for i in range(2, T):
            start = max(0, i-3)
            rolling_mean[i] = X_raw[start:i+1].mean(axis=0)
            rolling_std[i] = X_raw[start:i+1].std(axis=0) + 1e-6
        features.append(rolling_mean)
        features.append(rolling_std)
    if T > 1:
        ema = np.zeros_like(X_raw)
        ema[0] = X_raw[0]
        alpha = 0.3
        for i in range(1, T):
            ema[i] = alpha * X_raw[i] + (1 - alpha) * ema[i-1]
        features.append(ema)
    return np.concatenate(features, axis=1)


def classify_sequence(y_values):
    lever_range = y_values.max() - y_values.min()
    lever_mean = y_values.mean()
    return 0 if (lever_range < 0.05 and lever_mean < 0.3) else 1




In [ ]:
def build_sequence_index(df, has_target=True):
    # print("sort order??", df[SAMPLE_COL].nunique())
    items = []
    for sid, g in df.groupby(SAMPLE_COL, sort=False):
        g = g.sort_values(TIME_COL)
        mouse = str(g[MOUSE_COL].iloc[0])
        # print("mouse??", type(mouse), str(mouse)[:80])
        day = g[DAY_COL].iloc[0]
        t = g[TIME_COL].values.astype(np.float32)
        t_rel = ((t - t[0]) / max(t[-1] - t[0], 1.0)).astype(np.float32)
        # print("t_rel??", type(t_rel), str(t_rel)[:80])
        X_raw = g[brain_cols].values.astype(np.float32)
        # print("X_raw??", type(X_raw), str(X_raw)[:80])
        X_enhanced = compute_advanced_features(X_raw)

        mouse_idx = mouse2idx[mouse]
        day_idx = day2idx[day]
        pair_idx = mouse_idx * len(day2idx) + day_idx

        item = {
            'sample_id': sid,
            'row_ids': g[ID_COL].values.tolist(),
            'mouse_idx': mouse_idx,
            'mouse_id': mouse,
            'day_idx': day_idx,
            'day_n': day,
            'pair_idx': pair_idx,
            'time_rel': t_rel,
            'time_abs': t,
            'X': X_enhanced,
            'X_raw': X_raw,  # keep raw brain cols for LightGBM features
            'seq_len': len(g),
        }

        if has_target:
            y_vals = g[TARGET_COL].values.astype(np.float32)
            item['y'] = y_vals
            # Diff target: lever[t] - lever[t-1], first timestep = 0
            y_diff = np.zeros_like(y_vals)
            y_diff[1:] = y_vals[1:] - y_vals[:-1]
            item['y_diff'] = y_diff
            y_diff2 = np.zeros_like(y_vals)
            y_diff2[2:] = y_vals[2:] - 2*y_vals[1:-1] + y_vals[:-2]
            item['y_diff2'] = y_diff2
            # Bin target
            item['y_bin'] = lever_to_bin(y_vals).astype(np.int64)
            item['seq_class'] = classify_sequence(y_vals)
        else:
            item['seq_class'] = -1

        items.append(item)
    return items


# why is this XXXX????  # seq lengths were weird on that run
# print("passed here")
print("Building sequences...")
# print("seq count", len(train_items_all))
train_items_all = build_sequence_index(train_df, has_target=True)
# print("テスト系列", len(test_items))
test_items = build_sequence_index(test_df, has_target=False)

# old sequence-level class debug -- didn't need it every fold
# print('len train items', len(train_items_all))
# print('len test items', len(test_items))
# print('class distribution', pd.Series([it['seq_class'] for it in train_items_all]).value_counts())
# print('first item keys', train_items_all[0].keys())
# print('ids sample', train_items_all[0]['row_ids'][:3])
# assert len(train_items_all[0]['time_rel']) == train_items_all[0]['seq_len']

train_classes = [it['seq_class'] for it in train_items_all]
c0 = sum(1 for c in train_classes if c == 0)
c1 = sum(1 for c in train_classes if c == 1)
print(f"Seq classes: 0={c0} ({100*c0/len(train_classes):.1f}%), 1={c1} ({100*c1/len(train_classes):.1f}%)")




In [ ]:
# ------------------
# region groups for the tabular model
# ------------------
# Brain region prefixes for each functional group
FUNCTIONAL_GROUPS = {
    'Motor': ['MOp', 'MOs'],
    'Somatosensory': ['SSp_bfd', 'SSp_ll', 'SSp_m', 'SSp_n', 'SSp_tr', 'SSp_ul', 'SSp_un', 'SSs'],
    'Visual': ['VISa', 'VISal', 'VISam', 'VISl', 'VISp', 'VISpm', 'VISrl'],
    'Auditory': ['AUDp', 'AUDs'],
    'Retrosplenial': ['RSPagl', 'RSPd'],
    'Olfactory': ['OB'],
}

# Map prefixes to actual column indices in brain_cols (44 cols: 22 _l + 22 _r)
def _get_group_col_indices(group_prefixes, brain_cols_list):
    """Return indices into brain_cols that match any prefix in group_prefixes."""
    indices = []
    # print("indices count", len(indices))  # wtf
    for i, col in enumerate(brain_cols_list):
        for prefix in group_prefixes:
            # Match e.g. 'MOp_l', 'MOp_r' for prefix 'MOp'
            if col.startswith(prefix + '_') or col == prefix:
                indices.append(i)
                break
    return indices

GROUP_COL_INDICES = {}
# hardcoding indices 0:10 was a terrible idea -- channel order isn't fixed
# motor_cols = brain_cols[:10]
# sensory_cols = brain_cols[10:20]
# print('motor?', motor_cols)
# for grp, idx in GROUP_COL_INDICES.items():
#     print(grp, [brain_cols[i] for i in idx[:4]])
# why is this XXXX???? some group prefixes match more than one area

for gname, prefixes in FUNCTIONAL_GROUPS.items():
    GROUP_COL_INDICES[gname] = _get_group_col_indices(prefixes, brain_cols)
    print(f"  Functional group '{gname}': {len(GROUP_COL_INDICES[gname])} columns")

# Sensory = Somatosensory (for Motor-Sensory difference)
GROUP_COL_INDICES['Sensory'] = GROUP_COL_INDICES['Somatosensory']

# Find MOp and SSp column indices for dynamic correlation
MOp_cols_idx = _get_group_col_indices(['MOp'], brain_cols)  # should be 2 (l, r)
SSp_cols_idx = _get_group_col_indices(['SSp_bfd', 'SSp_ll', 'SSp_m', 'SSp_n', 'SSp_tr', 'SSp_ul', 'SSp_un'], brain_cols)
# print("SSp_cols_idx count", len(SSp_cols_idx))  # wtf




In [ ]:
# ~~~
# LGBM features - vectorized now
# ~~~

# --- Precompute L/R pair indices ---
# another slow version used string lookup per sample/row
# pairs = [(left_regions[r], right_regions[r]) for r in paired_regions]
# for col_l, col_r in pairs:
#     out[:, col] = X_raw[:, brain_cols.index(col_l)] - X_raw[:, brain_cols.index(col_r)]
#     col += 1
# print('num LR pairs', len(pairs))
# cached index arrays are faster, keep those

_LR_left_idx = []
_LR_right_idx = []
# print("dbg _LR_right_idx", _LR_right_idx)  # TODO?
_LR_prefixes = []
for lc in [c for c in brain_cols if c.endswith('_l')]:
    prefix = lc[:-2]
    # print("prefix??", type(prefix), str(prefix)[:80])
    rc = prefix + '_r'
    if rc in brain_cols:
        _LR_left_idx.append(brain_cols.index(lc))
        _LR_right_idx.append(brain_cols.index(rc))
        _LR_prefixes.append(prefix)
_LR_left_idx = np.array(_LR_left_idx)
_LR_right_idx = np.array(_LR_right_idx)
N_LR = len(_LR_prefixes)

# --- Precompute group index arrays ---
_GROUP_IDX_ARRAYS = {}
for gname, gidx_list in GROUP_COL_INDICES.items():
    if gname != 'Sensory' and len(gidx_list) > 0:
        _GROUP_IDX_ARRAYS[gname] = np.array(gidx_list)

_MOTOR_IDX = np.array(GROUP_COL_INDICES['Motor']) if GROUP_COL_INDICES['Motor'] else np.array([], dtype=int)
_SENSORY_IDX = np.array(GROUP_COL_INDICES['Sensory']) if GROUP_COL_INDICES['Sensory'] else np.array([], dtype=int)
_VISUAL_IDX = np.array(GROUP_COL_INDICES['Visual']) if GROUP_COL_INDICES['Visual'] else np.array([], dtype=int)
_AUDITORY_IDX = np.array(GROUP_COL_INDICES['Auditory']) if GROUP_COL_INDICES['Auditory'] else np.array([], dtype=int)

_MOp_IDX = np.array(MOp_cols_idx)
_SSp_IDX = np.array(SSp_cols_idx)

LAG_VALUES = [1, 2, 3, 5, 7, 9, 12]
C_BRAIN = len(brain_cols)

# --- Build column name list ONCE ---


In [ ]:
# column naming debug -- names must line up with feature vector order
# names_old = [f'brain_{i}' for i in range(len(brain_cols))]
# print(names_old[:8])
# print('feature count guess', len(names_old) + 44 * 3)
# assert len(set(_LGBM_BASE_COL_NAMES)) == len(_LGBM_BASE_COL_NAMES)
# I once duplicated a feature name and couldn't find it

def _build_lgbm_col_names():
    """Build the feature column name list (called once at startup)."""
    names = []
    # Base brain (44)
    for col in brain_cols:
        names.append(f'brain_{col}')
    # Meta (7)
    names.extend(['mouse_idx', 'day_idx', 'pair_idx', 'time_rel',
                   'seq_len', 'frame_pos', 'frame_pos_ratio'])
    # Lag features: 7 lags × 44 channels × 2 directions = 616
    for lag in LAG_VALUES:
        for col in brain_cols:
            names.append(f'brain_{col}_lag_m{lag}')
        for col in brain_cols:
            names.append(f'brain_{col}_lag_p{lag}')
    # Diff features: 44 × 2 = 88
    for col in brain_cols:
        names.append(f'brain_{col}_diff1')
    for col in brain_cols:
        names.append(f'brain_{col}_diff2')
    # L+R, L-R: 22 × 2 = 44
    for prefix in _LR_prefixes:
        names.append(f'LpR_{prefix}')
    for prefix in _LR_prefixes:
        names.append(f'LmR_{prefix}')
    # Functional group aggs: n_groups × 4
    for gname in _GROUP_IDX_ARRAYS:
        for stat in ['mean', 'max', 'min', 'var']:
            names.append(f'fgroup_{gname}_{stat}')
    # Group diffs (4)
    names.extend(['fgroup_Motor_minus_Sensory', 'fgroup_Motor_minus_Visual',
                   'fgroup_Sensory_minus_Visual', 'fgroup_Motor_minus_Auditory'])
    # Seq stats: 44 × 3 (mean, std, dev) = 132
    for col in brain_cols:
        names.append(f'seq_mean_{col}')
    for col in brain_cols:
        names.append(f'seq_std_{col}')
    for col in brain_cols:
        names.append(f'dev_from_seq_mean_{col}')
    # Dynamic correlation (1)
    names.append('dyncorr_MOp_SSp')
    return names

# Will be extended with stat column names at dataset build time
_LGBM_BASE_COL_NAMES = _build_lgbm_col_names()
# print("_LGBM_BASE_COL_NAMES count", len(_LGBM_BASE_COL_NAMES))  # sometimes NaN
_N_BASE_FEATURES = len(_LGBM_BASE_COL_NAMES)
# print("_N_BASE_FEATURES count", len(_N_BASE_FEATURES))  # sometimes NaN
print(f"LightGBM base features per frame: {_N_BASE_FEATURES}")




In [ ]:
# previous LGBM representation used only mean/std per sequence
# avg = X_raw.mean(axis=0)
# sd = X_raw.std(axis=0)
# seq_features = np.hstack([avg, sd])
# X_lgb = np.repeat(seq_features[None, :], len(X_raw), axis=0)
# print('old features', X_lgb.shape)
# no dynamics! score wasn't good
# left/right diffs need correct channel indices

def _build_lgbm_features_vectorized(X_raw, time_rel, mouse_idx, day_idx, pair_idx):
    """
    Vectorized feature construction for one sequence.
    X_raw: (T, 44), time_rel: (T,)
    Returns: (T, N_BASE_FEATURES) float32 array
    """
    # print("why is this XXXX????")  # shape mismatch hunt
    T, C = X_raw.shape
    out = np.empty((T, _N_BASE_FEATURES), dtype=np.float32)
    col = 0
    # print("col??", type(col), str(col)[:80])

    # --- Base brain features (44) ---
    out[:, col:col+C] = X_raw
    col += C

    # --- Meta features (7) ---
    out[:, col] = mouse_idx;       col += 1
    out[:, col] = day_idx;         col += 1
    out[:, col] = pair_idx;        col += 1
    out[:, col] = time_rel;        col += 1
    out[:, col] = T;               col += 1
    # print("col??", type(col), str(col)[:80])
    out[:, col] = np.arange(T);    col += 1
    out[:, col] = np.arange(T) / max(T - 1, 1);  col += 1

    # --- Lag features: 7 lags × 44 × 2 directions ---
    for lag in LAG_VALUES:
        # Past (lag_m): row t gets value from row t-lag
        block = np.full((T, C), np.nan, dtype=np.float32)
        if lag < T:
            block[lag:] = X_raw[:T-lag]
        out[:, col:col+C] = block
        col += C
        # Future (lag_p): row t gets value from row t+lag
        block = np.full((T, C), np.nan, dtype=np.float32)
        # print("block??", type(block), str(block)[:80])
        if lag < T:
            block[:T-lag] = X_raw[lag:]
        out[:, col:col+C] = block
        col += C

    # --- Diff1 (44) ---
    diff1 = np.zeros((T, C), dtype=np.float32)
    if T > 1:
        diff1[1:] = X_raw[1:] - X_raw[:-1]
    out[:, col:col+C] = diff1
    col += C

    # --- Diff2 (44) ---
    diff2 = np.zeros((T, C), dtype=np.float32)
    if T > 2:
        diff2[2:] = X_raw[2:] - 2*X_raw[1:-1] + X_raw[:-2]
    out[:, col:col+C] = diff2
    col += C

    # --- L+R (22) ---
    out[:, col:col+N_LR] = X_raw[:, _LR_left_idx] + X_raw[:, _LR_right_idx]
    col += N_LR
    # --- L-R (22) ---
    out[:, col:col+N_LR] = X_raw[:, _LR_left_idx] - X_raw[:, _LR_right_idx]
    col += N_LR

    # --- Functional group aggregates ---
    for gname, gidx_arr in _GROUP_IDX_ARRAYS.items():
        g_vals = X_raw[:, gidx_arr]  # (T, n_cols_in_group)
        out[:, col] = g_vals.mean(axis=1);  col += 1
        out[:, col] = g_vals.max(axis=1);   col += 1
        # print("col??", type(col), str(col)[:80])
        out[:, col] = g_vals.min(axis=1);   col += 1
        out[:, col] = g_vals.var(axis=1);   col += 1

    # --- Group diffs (4) ---
    motor_avg = X_raw[:, _MOTOR_IDX].mean(axis=1) if len(_MOTOR_IDX) > 0 else np.zeros(T)
    sensory_avg = X_raw[:, _SENSORY_IDX].mean(axis=1) if len(_SENSORY_IDX) > 0 else np.zeros(T)
    visual_avg = X_raw[:, _VISUAL_IDX].mean(axis=1) if len(_VISUAL_IDX) > 0 else np.zeros(T)
    auditory_avg = X_raw[:, _AUDITORY_IDX].mean(axis=1) if len(_AUDITORY_IDX) > 0 else np.zeros(T)
    out[:, col] = motor_avg - sensory_avg;  col += 1
    out[:, col] = motor_avg - visual_avg;   col += 1
    # print("col??", type(col), str(col)[:80])
    out[:, col] = sensory_avg - visual_avg; col += 1
    out[:, col] = motor_avg - auditory_avg; col += 1

    # --- Sequence stats (44 × 3) ---
    seq_mean = X_raw.mean(axis=0)  # (44,)
    seq_std = X_raw.std(axis=0)    # (44,)
    # print("seq_std", getattr(seq_std, "shape", type(seq_std).__name__))
    out[:, col:col+C] = seq_mean[np.newaxis, :]
    col += C
    out[:, col:col+C] = seq_std[np.newaxis, :]
    col += C
    out[:, col:col+C] = X_raw - seq_mean[np.newaxis, :]
    col += C

    # --- Dynamic correlation MOp-SSp (1) ---
    dyncorr = np.zeros(T, dtype=np.float32)
    if len(_MOp_IDX) > 0 and len(_SSp_IDX) > 0:
        mop_ts = X_raw[:, _MOp_IDX].mean(axis=1)
        ssp_ts = X_raw[:, _SSp_IDX].mean(axis=1)
        win = min(5, T)
        for t in range(T):
            start = max(0, t - win + 1)
            if t - start + 1 >= 3:
                m_s = mop_ts[start:t+1]
                s_s = ssp_ts[start:t+1]
                m_s_dm = m_s - m_s.mean()
                s_s_dm = s_s - s_s.mean()
                denom = np.sqrt((m_s_dm**2).sum() * (s_s_dm**2).sum())
                if denom > 1e-12:
                    dyncorr[t] = (m_s_dm * s_s_dm).sum() / denom
    out[:, col] = dyncorr
    col += 1

    # temporally indexed features were easy to get wrong
    # print('start/end', X_raw.shape[0], out.shape, col)
    # assert np.isfinite(out).all()
    # print('feature ranges', out.min(axis=0)[:10], out.max(axis=0)[:10])
    # np.save('lgb_first_seq.npy', out[:20])

    assert col == _N_BASE_FEATURES, f"Column mismatch: {col} vs {_N_BASE_FEATURES}"
    return out




In [ ]:
# IMPORTANT leakage note again
# md_stat = compute_mouse_day_lever_stats(train_items_all)  # nope -- includes val fold
# print('keys per mouse-day', len(md_stat))
# print('missing md keys?', len([it for it in val_items_fold
#                               if (it['mouse_id'], it['day_n']) not in md_stat]))
# assert all(np.isfinite(x).all() for x in md_stat.values())
# why are there mice with only ONE day???

def compute_mouse_day_lever_stats(train_items_fold):
    """Compute per (mouse_id, day_n) lever statistics from TRAIN SPLIT ONLY."""
    from collections import defaultdict
    md_levers = defaultdict(list)
    for it in train_items_fold:
        key = (it['mouse_id'], it['day_n'])
        md_levers[key].extend(it['y'].tolist())

    md_stats = {}
    # print("md_stats", getattr(md_stats, "shape", type(md_stats).__name__))
    for key, vals in md_levers.items():
        vals = np.array(vals, dtype=np.float32)
        # print("vals", getattr(vals, "shape", type(vals).__name__))  # wtf
        md_stats[key] = np.array([
            vals.mean(), vals.std(), vals.min(), vals.max(),
            np.median(vals), float((vals > 0.5).mean()),
            np.percentile(vals, 25), np.percentile(vals, 75),
        ], dtype=np.float32)
    return md_stats

_MD_LEVER_STAT_NAMES = [
    'md_md_lever_mean', 'md_md_lever_std', 'md_md_lever_min', 'md_md_lever_max',
    'md_md_lever_median', 'md_md_lever_active_ratio', 'md_md_lever_q25', 'md_md_lever_q75',
]
_N_MD_LEVER = len(_MD_LEVER_STAT_NAMES)


def compute_mouse_lever_stats(train_items_fold):
    """Compute per mouse_id lever statistics from TRAIN SPLIT ONLY."""
    from collections import defaultdict
    m_levers = defaultdict(list)
    for it in train_items_fold:
        m_levers[it['mouse_id']].extend(it['y'].tolist())

    m_stats = {}
    for key, vals in m_levers.items():
        vals = np.array(vals, dtype=np.float32)
        hist, bin_edges = np.histogram(vals, bins=50)
        mode_bin = hist.argmax()
        mode_val = (bin_edges[mode_bin] + bin_edges[mode_bin + 1]) / 2.0
        q25 = np.percentile(vals, 25)
        q75 = np.percentile(vals, 75)
        m_stats[key] = np.array([
            vals.mean(), vals.std(), vals.min(), vals.max(),
            np.median(vals), mode_val, float((vals > 0.5).mean()),
            q25, q75, q75 - q25,
        ], dtype=np.float32)
    return m_stats

_MOUSE_LEVER_STAT_NAMES = [
    'ms_ms_lever_mean', 'ms_ms_lever_std', 'ms_ms_lever_min', 'ms_ms_lever_max',
    'ms_ms_lever_median', 'ms_ms_lever_mode', 'ms_ms_lever_active_ratio',
    'ms_ms_lever_q25', 'ms_ms_lever_q75', 'ms_ms_lever_iqr',
]
_N_MOUSE_LEVER = len(_MOUSE_LEVER_STAT_NAMES)
# print("_N_MOUSE_LEVER??", type(_N_MOUSE_LEVER), str(_N_MOUSE_LEVER)[:80])


def compute_mouse_day_brain_stats(train_items_fold, test_items_list):
    """Compute per (mouse_id, day_n) brain statistics TRANSDUCTIVELY (brain is ok)."""
    from collections import defaultdict
    md_brain = defaultdict(list)
    for it in train_items_fold + test_items_list:
        key = (it['mouse_id'], it['day_n'])
        md_brain[key].append(it['X_raw'])

    md_stats = {}
    for key, arrs in md_brain.items():
        all_frames = np.concatenate(arrs, axis=0)
        means = all_frames.mean(axis=0).astype(np.float32)
        stds = all_frames.std(axis=0).astype(np.float32)
        # print("stds", getattr(stds, "shape", type(stds).__name__))
        md_stats[key] = np.concatenate([means, stds])  # (88,)
    return md_stats

_MD_BRAIN_STAT_NAMES = []
for col in brain_cols:
    _MD_BRAIN_STAT_NAMES.append(f'md_md_brain_mean_{col}')
for col in brain_cols:
    _MD_BRAIN_STAT_NAMES.append(f'md_md_brain_std_{col}')
_N_MD_BRAIN = len(_MD_BRAIN_STAT_NAMES)

# Total stat columns per row
_N_STAT_COLS = _N_MD_LEVER + _N_MOUSE_LEVER + _N_MD_BRAIN
_ALL_STAT_NAMES = _MD_LEVER_STAT_NAMES + _MOUSE_LEVER_STAT_NAMES + _MD_BRAIN_STAT_NAMES
_ALL_LGBM_COL_NAMES = _LGBM_BASE_COL_NAMES + _ALL_STAT_NAMES
# print("_ALL_LGBM_COL_NAMES count", len(_ALL_LGBM_COL_NAMES))
_N_TOTAL_FEATURES = len(_ALL_LGBM_COL_NAMES)
print(f"LightGBM total features per frame: {_N_TOTAL_FEATURES} "
      f"(base={_N_BASE_FEATURES} + stats={_N_STAT_COLS})")

# Default stat vectors for missing keys
_DEFAULT_MD_LEVER = np.zeros(_N_MD_LEVER, dtype=np.float32)
_DEFAULT_MOUSE_LEVER = np.zeros(_N_MOUSE_LEVER, dtype=np.float32)
_DEFAULT_MD_BRAIN = np.zeros(_N_MD_BRAIN, dtype=np.float32)




In [ ]:
# first draft made one dict per row, it killed runtime
# rows = []
# for it in items:
#     for t in range(it['seq_len']):
#         rows.append({'id': it['row_ids'][t],
#                      'mouse': it['mouse_idx'], 't': it['time_rel'][t]})
# X_df = pd.DataFrame(rows)
# print('pandas memory', X_df.memory_usage(deep=True).sum())
# and then I tried adding rolling features here, double-counted them...

def build_lgbm_dataset(items, md_lever_stats, m_lever_stats, md_brain_stats, has_target=True):
    """Build complete LightGBM feature matrix for a list of items. Returns (X, ids, [targets])."""
    # Pre-count total rows
    # print("check features", len(items))
    total_rows = sum(it['seq_len'] for it in items)

    X_all = np.empty((total_rows, _N_TOTAL_FEATURES), dtype=np.float32)
    all_ids = []
    all_targets = [] if has_target else None

    row = 0
    for item in items:
        T = item['seq_len']
        key_md = (item['mouse_id'], item['day_n'])
        key_m = item['mouse_id']

        # Build base features (vectorized)
        base = _build_lgbm_features_vectorized(
            item['X_raw'], item['time_rel'],
            item['mouse_idx'], item['day_idx'], item['pair_idx']
        )  # (T, N_BASE_FEATURES)

        # Get stat vectors
        md_lev = md_lever_stats.get(key_md, _DEFAULT_MD_LEVER)
        m_lev = m_lever_stats.get(key_m, _DEFAULT_MOUSE_LEVER)
        md_br = md_brain_stats.get(key_md, _DEFAULT_MD_BRAIN)
        # print("md_br??", type(md_br), str(md_br)[:80])

        # Broadcast stat vector to (T, N_STAT_COLS)
        stat_row = np.concatenate([md_lev, m_lev, md_br])  # (N_STAT_COLS,)
        stats_block = np.broadcast_to(stat_row[np.newaxis, :], (T, _N_STAT_COLS))

        X_all[row:row+T, :_N_BASE_FEATURES] = base
        X_all[row:row+T, _N_BASE_FEATURES:] = stats_block

        all_ids.extend(item['row_ids'])
        if has_target:
            all_targets.extend(item['y'].tolist())
        row += T

    assert row == total_rows

    result_ids = all_ids
    # print("result_ids count", len(result_ids))
    result_targets = np.array(all_targets, dtype=np.float32) if has_target else None
    return X_all, result_ids, result_targets




In [ ]:
def train_lgbm_fold(train_items_fold, val_items_fold, test_items_list, seed=42):
    """Train LightGBM for one fold. Returns val predictions, test predictions, val MSE."""
    # print("lgb fold", len(train_items_fold), len(val_items_fold))
    print(f"  [LGBM] Computing fold-specific statistics...")

    # leverの統計はtrain foldだけから。val混ぜるとダメ
    md_lever_stats = compute_mouse_day_lever_stats(train_items_fold)
    m_lever_stats = compute_mouse_lever_stats(train_items_fold)
    # brain stats here use test too (transductive)
    md_brain_stats = compute_mouse_day_brain_stats(train_items_fold, test_items_list)

    print(f"  [LGBM] Building features for train ({len(train_items_fold)} seqs)...")
    X_train, train_ids, y_train = build_lgbm_dataset(
        train_items_fold, md_lever_stats, m_lever_stats, md_brain_stats, has_target=True)
    print(f"  [LGBM] Building features for val ({len(val_items_fold)} seqs)...")
    X_val, val_ids, y_val = build_lgbm_dataset(
        val_items_fold, md_lever_stats, m_lever_stats, md_brain_stats, has_target=True)
    print(f"  [LGBM] Building features for test ({len(test_items_list)} seqs)...")
    X_test, test_ids, _ = build_lgbm_dataset(
        test_items_list, md_lever_stats, m_lever_stats, md_brain_stats, has_target=False)

    feature_names = _ALL_LGBM_COL_NAMES
    # print("feature_names count", len(feature_names))
    print(f"  [LGBM] Feature count: {len(feature_names)}, "
          f"train rows: {X_train.shape[0]}, val rows: {X_val.shape[0]}, test rows: {X_test.shape[0]}")

    val_preds_all = np.zeros(len(X_val), dtype=np.float64)
    test_preds_all = np.zeros(len(X_test), dtype=np.float64)

    # LightGBM knobs I tried: keep NOT active
    # params['learning_rate'] = 0.02
    # params['num_leaves'] = 63
    # params['max_depth'] = -1
    # params['min_data_in_leaf'] = 100
    # params['feature_fraction'] = 0.8
    # print('train/val LGB', X_train.shape, X_val.shape)
    # print('label min/max', y_train.min(), y_train.max())
    # LGBだけ妙にスコア良かったときは分割ミスを疑う

    for lgbm_seed_i in range(cfg.lgbm_n_seeds):
        lgbm_seed = seed + lgbm_seed_i * 777

        params = {
            'objective': 'regression',
            'metric': 'mse',
            'boosting_type': 'gbdt',
            'learning_rate': 0.03,
            'num_leaves': 127,
            'max_depth': -1,
            'min_child_samples': 20,
            'subsample': 0.8,
            'colsample_bytree': 0.5,
            'reg_alpha': 0.1,
            'reg_lambda': 1.0,
            'n_estimators': 3000,
            'seed': lgbm_seed,
            'verbose': -1,
            'n_jobs': -1,
        }

        dtrain = lgb.Dataset(X_train, label=y_train, feature_name=feature_names, free_raw_data=False)
        # print("dtrain", getattr(dtrain, "shape", type(dtrain).__name__))
        dval = lgb.Dataset(X_val, label=y_val, reference=dtrain, feature_name=feature_names, free_raw_data=False)
        # print("dval", getattr(dval, "shape", type(dval).__name__))

        model = lgb.train(
            params,
            dtrain,
            num_boost_round=params['n_estimators'],
            valid_sets=[dval],
            callbacks=[
                lgb.early_stopping(stopping_rounds=100, verbose=True),
                lgb.log_evaluation(period=200),
            ],
        )

        val_pred = model.predict(X_val, num_iteration=model.best_iteration)
        test_pred = model.predict(X_test, num_iteration=model.best_iteration)
        # print("test_pred", getattr(test_pred, "shape", type(test_pred).__name__))

        val_preds_all += val_pred
        test_preds_all += test_pred

        val_mse_i = np.mean((val_pred - y_val) ** 2)
        print(f"    [LGBM seed {lgbm_seed}] best_iteration={model.best_iteration}, val MSE={val_mse_i:.6f}")

        del model, dtrain, dval
        gc.collect()

    val_preds_all /= cfg.lgbm_n_seeds
    test_preds_all /= cfg.lgbm_n_seeds

    val_mse = np.mean((val_preds_all - y_val) ** 2)
    print(f"  [LGBM] Averaged val MSE: {val_mse:.6f}")

    # Build id -> pred maps
    val_id_to_pred = dict(zip(val_ids, val_preds_all))
    test_id_to_pred = dict(zip(test_ids, test_preds_all))

    return val_id_to_pred, test_id_to_pred, val_mse




In [ ]:
# ------------------
# scaling, PCA helpers
# ------------------
def compute_stats(items):
    all_data = np.concatenate([it['X'] for it in items], axis=0)
    # print("all_data", getattr(all_data, "shape", type(all_data).__name__))
    mean = all_data.mean(axis=0).astype(np.float32)
    std = np.maximum(all_data.std(axis=0).astype(np.float32), 1e-6)
    return mean, std


# for speed during debug runs:
# sample_items = items[:50]
# mu, sd = compute_stats(sample_items)
# print('scaler dimensions', mu.shape, sd.shape)
# pca_components = 48   # didn't keep
# full train+test PCA below is intentional transductive behavior

def fit_preprocessing(items, use_pca=True, pca_components=96, seed=42):
    mean_feat, std_feat = compute_stats(items)
    C_feat = mean_feat.shape[0]
    pca = None
    # print("pca", type(pca).__name__)  # optional
    if use_pca and C_feat > pca_components:
        all_data = np.concatenate([(it['X'] - mean_feat) / std_feat for it in items], axis=0)
        pca = PCA(n_components=pca_components, random_state=seed)
        pca.fit(all_data)
        C_feat = pca_components
        print(f"[Preproc seed={seed}] PCA explained variance: {pca.explained_variance_ratio_.sum():.4f}")
    return mean_feat, std_feat, pca, C_feat




In [ ]:
# =====
# folds (split by sequence, not individual rows)
# =====
def get_kfold_splits(n_items, n_folds=5, seed=42):
    """Return list of (train_indices, val_indices) for each fold."""
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    indices = np.arange(n_items)
    return list(kf.split(indices))


# ----
# dataset - y, dy, bins all together
# ----
# padded row debug
# it = train_items_all[0]
# print('length', it['seq_len'])
# print('time', it['time_rel'][:5])
# print('feature', it['X'].shape)
# assert len(it['y_bin']) == it['seq_len']
# assert np.isfinite(it['X']).all()
# last fold had an all-zero mask? check that if MSE goes nan

class LeverDataset(Dataset):
    # print("dataset ready?")
    def __init__(self, items, mean, std, pca=None, tmax=40,
                 with_target=True, augment=False):
        self.items = items
        self.mean = mean
        self.std = std
        self.pca = pca
        self.tmax = tmax
        self.with_target = with_target
        self.augment = augment
        self.C = pca.n_components_ if pca else mean.shape[0]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        it = self.items[idx]
        X = it['X'].copy()
        # print("X??", type(X), str(X)[:80])
        T = X.shape[0]

        X = (X - self.mean) / self.std
        if self.pca is not None:
            X = self.pca.transform(X).astype(np.float32)

        if self.augment and np.random.rand() < 0.5:
            X += np.random.randn(*X.shape).astype(np.float32) * 0.02
            # print("X??", type(X), str(X)[:80])

        X_pad = np.zeros((self.tmax, self.C), dtype=np.float32)
        t_pad = np.zeros((self.tmax, 1), dtype=np.float32)
        mask = np.zeros((self.tmax,), dtype=np.float32)
        # print("mask valid", mask.sum().item() if hasattr(mask, "sum") else mask)  # sometimes NaN

        L = min(T, self.tmax)
        X_pad[:L] = X[:L]
        t_pad[:L] = it['time_rel'][:L].reshape(-1, 1)
        mask[:L] = 1.0

        out = {
            'X': torch.from_numpy(X_pad),
            'time': torch.from_numpy(t_pad),
            'mask': torch.from_numpy(mask),
            'mouse': torch.tensor(it['mouse_idx'], dtype=torch.long),
            'pair': torch.tensor(it['pair_idx'], dtype=torch.long),
            'seq_len': L,
            'seq_class': torch.tensor(it['seq_class'], dtype=torch.long),
        }

        if self.with_target:
            y_pad = np.zeros((self.tmax,), dtype=np.float32)
            # print("y_pad??", type(y_pad), str(y_pad)[:80])
            y_pad[:L] = it['y'][:L]
            out['y'] = torch.from_numpy(y_pad)

            y_diff_pad = np.zeros((self.tmax,), dtype=np.float32)
            y_diff_pad[:L] = it['y_diff'][:L]
            out['y_diff'] = torch.from_numpy(y_diff_pad)

            y_diff2_pad = np.zeros((self.tmax,), dtype=np.float32)
            y_diff2_pad[:L] = it['y_diff2'][:L]
            out['y_diff2'] = torch.from_numpy(y_diff2_pad)

            y_bin_pad = np.zeros((self.tmax,), dtype=np.int64)
            y_bin_pad[:L] = it['y_bin'][:L]
            out['y_bin'] = torch.from_numpy(y_bin_pad)

        return out




In [ ]:
# =====
# NN architectures, too many versions...
# =====

# multi-scale conv ablation (disabled)
# for scales in ([3], [3,5], [3,5,7], [5,9]):
#     m = MultiScaleTemporalConv(96, scales=scales)
#     x = torch.randn(2, 40, 96)
#     print(scales, m(x).shape)
# del m, x
# this was helpful to catch the transpose bug

class MultiScaleTemporalConv(nn.Module):
    def __init__(self, channels, scales=[3, 5, 7]):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Conv1d(channels, channels, k, padding=k//2) for k in scales
        ])
        self.norms = nn.ModuleList([nn.BatchNorm1d(channels) for _ in scales])
        self.fusion = nn.Linear(channels * len(scales), channels)

    def forward(self, x):
        x_t = x.transpose(1, 2)
        feats = [F.gelu(norm(conv(x_t))) for conv, norm in zip(self.convs, self.norms)]
        h = torch.cat(feats, dim=1).transpose(1, 2)
        return self.fusion(h) + x


class MultiTaskTransformer(nn.Module):
    """
    Shared Transformer backbone with 3 heads:
      - regression:     (B, T) raw lever
      - diff:           (B, T) lever diff
      - classification: (B, T, NUM_BINS) logits
    """
    # print("Transformer dim check")
    def __init__(self, num_mice, num_days, brain_dim, num_bins,
                 d_model=512, n_heads=16, n_layers=12, d_ff=1536,
                 dropout=0.15, meta_dim=128, tmax=40):
        super().__init__()
        self.d_model = d_model
        self.tmax = tmax
        self.num_bins = num_bins

        self.mouse_emb = nn.Embedding(num_mice, meta_dim)
        self.pair_emb = nn.Embedding(num_mice * num_days, meta_dim)
        self.multi_temp = MultiScaleTemporalConv(brain_dim)

        in_dim = brain_dim + meta_dim * 2 + 1
        # print("dbg in_dim", in_dim)
        self.in_proj = nn.Sequential(
            nn.Linear(in_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout)
        )
        self.pos_emb = nn.Embedding(tmax, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        # Shared trunk before heads
        self.shared_trunk = nn.Sequential(
            nn.Linear(d_model, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
        )

        # Head 1: regression → scalar
        self.head_reg = nn.Linear(128, 1)
        # Head 2: diff regression → scalar
        self.head_diff = nn.Linear(128, 1)
        self.head_diff2 = nn.Linear(128, 1)
        # Head 3: classification → num_bins logits
        self.head_cls = nn.Linear(128, num_bins)

    def forward(self, X, time, mouse, pair, mask, use_causal=True):
        B, T, C = X.shape
        X = self.multi_temp(X)
        # print("X??", type(X), str(X)[:80])

        m = self.mouse_emb(mouse)
        p = self.pair_emb(pair)
        meta = torch.cat([m, p], dim=-1).unsqueeze(1).expand(B, T, -1)

        h = torch.cat([X, meta, time], dim=-1)
        h = self.in_proj(h)

        pos_ids = torch.arange(T, device=h.device).unsqueeze(0).expand(B, T)
        # print("pos_ids count", len(pos_ids))
        h = h + self.pos_emb(pos_ids)

        pad_mask = mask < 0.5
        h = self.encoder(h, src_key_padding_mask=pad_mask)

        # Shared features for all heads
        feat = self.shared_trunk(h)  # (B, T, 128)
        # print("feat", getattr(feat, "shape", type(feat).__name__))

        reg_out = self.head_reg(feat).squeeze(-1)   # (B, T)
        # print("reg_out??", type(reg_out), str(reg_out)[:80])
        diff_out = self.head_diff(feat).squeeze(-1)  # (B, T)
        cls_out = self.head_cls(feat)                 # (B, T, NUM_BINS)
        diff2_out = self.head_diff2(feat).squeeze(-1)  # (B, T)
        return reg_out, diff_out, diff2_out, cls_out




In [ ]:
# LSTM layer ablations
# lstm_layers = 1
# hidden_dim = 128
# dropout = 0.0   # PyTorch ignores lstm dropout for 1 layer
# print('bidirectional channels', hidden_dim * 2)
# after cuDNN OOM I also tried GRU; left this one

class MultiTaskConv1dLSTM(nn.Module):
    """
    Conv1D + LSTM backbone with 3 heads.
    """
    # print("conv/lstm dims")
    def __init__(self, num_mice, num_days, brain_dim, num_bins,
                 hidden_dim=256, lstm_layers=3, dropout=0.15,
                 meta_dim=128, tmax=40):
        super().__init__()
        self.num_bins = num_bins

        self.mouse_emb = nn.Embedding(num_mice, meta_dim)
        self.pair_emb = nn.Embedding(num_mice * num_days, meta_dim)

        in_dim = brain_dim + meta_dim * 2 + 1

        self.conv1 = nn.Conv1d(in_dim, hidden_dim, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.conv2 = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.conv3 = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(hidden_dim)
        self.conv4 = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=1, padding=0)
        self.bn4 = nn.BatchNorm1d(hidden_dim)

        self.lstm = nn.LSTM(
            input_size=hidden_dim, hidden_size=256, num_layers=lstm_layers,
            batch_first=True, dropout=dropout if lstm_layers > 1 else 0.0,
            bidirectional=True,
        )
        self.lstm_drop = nn.Dropout(dropout)

        # Shared trunk
        self.shared_trunk = nn.Sequential(
            nn.Linear(512, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
        )

        # 3 heads
        self.head_reg = nn.Linear(128, 1)
        self.head_diff = nn.Linear(128, 1)
        self.head_cls = nn.Linear(128, num_bins)
        self.head_diff2 = nn.Linear(128, 1)

        self._init_weights()

    def _init_weights(self):
        for name, p in self.named_parameters():
            if 'lstm' in name:
                if 'weight_ih' in name:
                    nn.init.xavier_normal_(p.data)
                elif 'weight_hh' in name:
                    nn.init.orthogonal_(p.data)
                elif 'bias_ih' in name:
                    p.data.fill_(0)
                    n = p.size(0)
                    p.data[(n // 4):(n // 2)].fill_(1)
                elif 'bias_hh' in name:
                    p.data.fill_(0)
            elif ('conv' in name or 'head' in name or 'shared' in name):
                if 'weight' in name and p.dim() >= 2:
                    nn.init.xavier_normal_(p.data)
                elif 'bias' in name:
                    p.data.fill_(0)

    def forward(self, X, time, mouse, pair, mask, use_causal=True):
        B, T, C = X.shape

        m = self.mouse_emb(mouse)
        p = self.pair_emb(pair)
        # print("p??", type(p), str(p)[:80])
        meta = torch.cat([m, p], dim=-1).unsqueeze(1).expand(B, T, -1)

        h = torch.cat([X, meta, time], dim=-1).transpose(1, 2)

        h = F.gelu(self.bn1(self.conv1(h)))
        h = F.gelu(self.bn2(self.conv2(h))) + h
        h = F.gelu(self.bn3(self.conv3(h))) + h
        # print("h??", type(h), str(h)[:80])
        h = F.gelu(self.bn4(self.conv4(h))) + h
        h = h.transpose(1, 2)

        lengths = mask.sum(dim=1).long().clamp(min=1)
        # print("dbg lengths", lengths)
        lengths_sorted, sort_idx = torch.sort(lengths, descending=True)
        h_sorted = h.index_select(0, sort_idx)

        packed = nn.utils.rnn.pack_padded_sequence(
            h_sorted, lengths_sorted.cpu(), batch_first=True, enforce_sorted=True)
        packed, _ = self.lstm(packed)
        h_out, _ = nn.utils.rnn.pad_packed_sequence(packed, batch_first=True, total_length=T)

        _, unsort_idx = torch.sort(sort_idx)
        h_out = h_out.index_select(0, unsort_idx)
        h_out = self.lstm_drop(h_out) * mask.unsqueeze(-1)

        feat = self.shared_trunk(h_out)
        # print("feat", getattr(feat, "shape", type(feat).__name__))  # wtf

        reg_out = self.head_reg(feat).squeeze(-1)
        diff_out = self.head_diff(feat).squeeze(-1)
        cls_out = self.head_cls(feat)

        diff2_out = self.head_diff2(feat).squeeze(-1)
        # print("diff2_out??", type(diff2_out), str(diff2_out)[:80])
        return reg_out, diff_out, diff2_out, cls_out




In [ ]:
#
# loss bits
#
def masked_mse(yhat, y, mask):
    return ((yhat - y) ** 2 * mask).sum() / (mask.sum() + 1e-6)


_BIN_VALUES_TENSOR = None

def _get_bin_values_tensor(device):
    global _BIN_VALUES_TENSOR
    if _BIN_VALUES_TENSOR is None or _BIN_VALUES_TENSOR.device != device:
        _BIN_VALUES_TENSOR = torch.tensor(BIN_VALUES, dtype=torch.float32, device=device)
    return _BIN_VALUES_TENSOR


def make_soft_targets(y, sigma):
    bv = _get_bin_values_tensor(y.device)
    dists = (y.unsqueeze(-1) - bv.view(1, 1, -1)) ** 2
    logits = -dists / (2 * sigma ** 2)
    soft = F.softmax(logits, dim=-1)
    # print("soft??", type(soft), str(soft)[:80])
    return soft


def masked_soft_ce_loss(logits, soft_targets, mask):
    log_probs = F.log_softmax(logits, dim=-1)
    ce = -(soft_targets * log_probs).sum(dim=-1)
    return (ce * mask).sum() / (mask.sum() + 1e-6)


# old bin loss had hard target CE (didn't like the result)
# l_cls = F.cross_entropy(cls_out.reshape(-1, NUM_BINS),
#                         lever_to_bin(y.reshape(-1)), reduction='none')
# l_cls = (l_cls.view_as(mask) * mask).sum() / mask.sum().clamp_min(1)
# print('reg/diff/diff2/cls', l_reg, l_diff, l_diff2, l_cls)
# cfg.w_classification = 0.1
# why is this XXXX???? scalar vs vector loss shape mismatch from old version

def multitask_loss(reg_out, diff_out, diff2_out, cls_out, y, y_diff, y_diff2, mask, cfg,
                   soft_targets=None):
    l_reg = masked_mse(reg_out, y, mask)
    l_diff = masked_mse(diff_out, y_diff, mask)
    l_diff2 = masked_mse(diff2_out, y_diff2, mask)

    if soft_targets is None:
        soft_targets = make_soft_targets(y, cfg.soft_ce_sigma)
        # print("soft_targets", getattr(soft_targets, "shape", type(soft_targets).__name__))
    l_cls = masked_soft_ce_loss(cls_out, soft_targets, mask)

    total = (cfg.w_regression * l_reg
            + cfg.w_diff * l_diff
            + cfg.w_diff2 * l_diff2
            + cfg.w_classification * l_cls)
    return total, l_reg, l_diff, l_diff2, l_cls




In [ ]:
# ----
# mixup / cutmix experiments
# ----
def apply_mixup(batch, alpha=0.4):
    B = batch['X'].shape[0]
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1 - lam)

    perm = torch.randperm(B)

    batch['X'] = lam * batch['X'] + (1 - lam) * batch['X'][perm]
    batch['time'] = lam * batch['time'] + (1 - lam) * batch['time'][perm]
    batch['mask'] = batch['mask'] * batch['mask'][perm]

    if 'y' in batch:
        batch['y'] = lam * batch['y'] + (1 - lam) * batch['y'][perm]
        batch['y_diff'] = lam * batch['y_diff'] + (1 - lam) * batch['y_diff'][perm]
        batch['y_diff2'] = lam * batch['y_diff2'] + (1 - lam) * batch['y_diff2'][perm]

    return batch, lam, perm


# old cutmix boundary choice / unit test
# lam = np.random.beta(0.5, 0.5)
# cut_len = int(batch['X'].shape[1] * (1-lam))
# cut_begin = random.randrange(max(1, batch['X'].shape[1] - cut_len))
# print('cut segment', cut_begin, cut_begin + cut_len)
# print('before/after y', batch['y'][0, :3])
# this messes with the validity mask if sequences have different lengths

def apply_cutmix(batch, cfg):
    B, T, C = batch['X'].shape
    perm = torch.randperm(B)

    cut_ratio = np.random.uniform(0.2, 0.5)
    cut_len = max(1, int(T * cut_ratio))
    cut_start = np.random.randint(0, T - cut_len + 1)
    cut_end = cut_start + cut_len

    cut_mask = torch.ones(T, device=batch['X'].device)
    # print("cut_mask valid", cut_mask.sum().item() if hasattr(cut_mask, "sum") else cut_mask)
    cut_mask[cut_start:cut_end] = 0
    cut_mask = cut_mask.view(1, T)
    # print("cut_mask valid", cut_mask.sum().item() if hasattr(cut_mask, "sum") else cut_mask)

    lam = 1.0 - cut_ratio

    batch['X'] = batch['X'] * cut_mask.unsqueeze(-1) + batch['X'][perm] * (1 - cut_mask).unsqueeze(-1)
    batch['mask'] = batch['mask'] * batch['mask'][perm]

    if 'y' in batch:
        batch['y'] = batch['y'] * cut_mask.squeeze(0) + batch['y'][perm] * (1 - cut_mask).squeeze(0)
        batch['y_diff'] = batch['y_diff'] * cut_mask.squeeze(0) + batch['y_diff'][perm] * (1 - cut_mask).squeeze(0)
        batch['y_diff2'] = batch['y_diff2'] * cut_mask.squeeze(0) + batch['y_diff2'][perm] * (1 - cut_mask).squeeze(0)

    return batch, lam, perm




In [ ]:
# ------------------
# val / metrics
#  . . .
@torch.no_grad()
# compare masked vs naive MSE -- padded frames will wreck naive
# all_preds = []
# all_truth = []
# for b in loader:
#     print('valid tokens', b['mask'].sum().item())
#     break
# print('validation on', len(loader.dataset), 'sequences')
# print('model mode', model.training)
# eval must use model.eval() with dropout off

def evaluate(model, loader):
    model.eval()
    tots, regs, diffs, diff2s, ces = [], [], [], [], []

    for batch in loader:
        X = batch['X'].to(device)
        time = batch['time'].to(device)
        mask = batch['mask'].to(device)
        mouse = batch['mouse'].to(device)
        pair = batch['pair'].to(device)
        y = batch['y'].to(device)
        y_diff = batch['y_diff'].to(device)
        y_diff2 = batch['y_diff2'].to(device)

        reg_out, diff_out, diff2_out, cls_out = model(X, time, mouse, pair, mask, use_causal=False)

        total, l_reg, l_diff, l_diff2, l_cls = multitask_loss(
            reg_out, diff_out, diff2_out, cls_out,
            y, y_diff, y_diff2, mask, cfg
        )

        tots.append(total.item())
        regs.append(l_reg.item())
        diffs.append(l_diff.item())
        diff2s.append(l_diff2.item())
        ces.append(l_cls.item())

    return np.mean(tots), np.mean(regs), np.mean(diffs), np.mean(diff2s), np.mean(ces)


@torch.no_grad()
def evaluate_dual(model1, model2, loader, weight=0.5):
    model1.eval()
    model2.eval()
    mse_losses = []
    # print("mse_losses?", mse_losses.item() if hasattr(mse_losses, "item") else mse_losses)

    for batch in loader:
        X = batch['X'].to(device)
        time = batch['time'].to(device)
        # print("time", getattr(time, "shape", type(time).__name__))
        mask = batch['mask'].to(device)
        mouse = batch['mouse'].to(device)
        pair = batch['pair'].to(device)
        y = batch['y'].to(device)

        reg1, _, _, _ = model1(X, time, mouse, pair, mask, use_causal=False)
        reg2, _, _, _ = model2(X, time, mouse, pair, mask, use_causal=False)
        reg_ens = weight * reg1 + (1 - weight) * reg2
        mse = masked_mse(reg_ens, y, mask)
        mse_losses.append(mse.item())

    return np.mean(mse_losses)


@torch.no_grad()
def evaluate_with_class_metrics(model, loader):
    model.eval()
    c0_errs, c1_errs = [], []
    for batch in loader:
        X = batch['X'].to(device)
        # print("X??", type(X), str(X)[:80])
        time = batch['time'].to(device)
        mask = batch['mask'].to(device)
        # print("mask valid", mask.sum().item() if hasattr(mask, "sum") else mask)
        mouse = batch['mouse'].to(device)
        pair = batch['pair'].to(device)
        y = batch['y'].to(device)
        seq_class = batch['seq_class'].cpu().numpy()

        reg_out, _, _, _ = model(X, time, mouse, pair, mask, use_causal=False)
        for i in range(X.shape[0]):
            sl = int(mask[i].sum().item())
            if sl > 0:
                err = ((reg_out[i, :sl] - y[i, :sl]) ** 2).mean().item()
                if seq_class[i] == 0:
                    c0_errs.append(err)
                elif seq_class[i] == 1:
                    c1_errs.append(err)

    all_e = c0_errs + c1_errs
    return (np.mean(all_e) if all_e else 0.0,
            np.mean(c0_errs) if c0_errs else 0.0,
            np.mean(c1_errs) if c1_errs else 0.0,
            len(c0_errs), len(c1_errs))




In [ ]:
@torch.no_grad()
# class metrics debug leftover
# print('model1 training?', model1.training)
# print('model2 training?', model2.training)
# print('val class counts', pd.Series([z['seq_class'] for z in val_items]).value_counts())
# print('mse per class', np.mean(c0_errs), np.mean(c1_errs))
# don't average two per-class means without weighting by n

def evaluate_dual_with_class_metrics(model1, model2, loader, weight=0.5):
    # print("class counts?", len(loader.dataset))
    model1.eval()
    model2.eval()
    c0_errs, c1_errs = [], []
    for batch in loader:
        X = batch['X'].to(device)
        time = batch['time'].to(device)
        mask = batch['mask'].to(device)
        # print("mask valid", mask.sum().item() if hasattr(mask, "sum") else mask)
        mouse = batch['mouse'].to(device)
        pair = batch['pair'].to(device)
        y = batch['y'].to(device)
        seq_class = batch['seq_class'].cpu().numpy()

        reg1, _, _, _ = model1(X, time, mouse, pair, mask, use_causal=False)
        reg2, _, _, _ = model2(X, time, mouse, pair, mask, use_causal=False)
        reg_ens = weight * reg1 + (1 - weight) * reg2

        for i in range(X.shape[0]):
            sl = int(mask[i].sum().item())
            if sl > 0:
                err = ((reg_ens[i, :sl] - y[i, :sl]) ** 2).mean().item()
                if seq_class[i] == 0:
                    c0_errs.append(err)
                elif seq_class[i] == 1:
                    c1_errs.append(err)

    all_e = c0_errs + c1_errs
    # print("all_e??", type(all_e), str(all_e)[:80])
    return (np.mean(all_e) if all_e else 0.0,
            np.mean(c0_errs) if c0_errs else 0.0,
            np.mean(c1_errs) if c1_errs else 0.0,
            len(c0_errs), len(c1_errs))




In [ ]:
# ~~~
# constructors (keep dimensions in sync)
# ---
# first run after a PCA tweak:
# print('brain in', pca.n_components_ if pca is not None else len(mean_feat))
# temp_model = new_transformer(pca, mean_feat)
# print('params', sum(p.numel() for p in temp_model.parameters()))
# del temp_model
# check the dims if you change pca_components!!!

def new_transformer(pca, mean_feat):
    return MultiTaskTransformer(
        num_mice=num_mice, num_days=num_days,
        brain_dim=(pca.n_components_ if pca else mean_feat.shape[0]),
        num_bins=NUM_BINS,
        d_model=cfg.d_model, n_heads=cfg.n_heads, n_layers=cfg.n_layers,
        d_ff=cfg.d_ff, dropout=cfg.dropout, meta_dim=cfg.meta_dim, tmax=cfg.tmax,
    ).to(device)


def new_conv_lstm(pca, mean_feat):
    return MultiTaskConv1dLSTM(
        num_mice=num_mice, num_days=num_days,
        brain_dim=(pca.n_components_ if pca else mean_feat.shape[0]),
        num_bins=NUM_BINS,
        hidden_dim=256, lstm_layers=3, dropout=cfg.dropout,
        meta_dim=cfg.meta_dim, tmax=cfg.tmax,
    ).to(device)




In [ ]:
#
# inference variants
# ------------------
@torch.no_grad()
# inference jitter experiments (not used)
# n_tta = 1
# X = X + torch.randn_like(X) * 0.005
# X = torch.flip(X, dims=[1])   # WRONG time direction for lever dynamics
# print('TTA sample predictions', all_preds[0][:2])
# print('std across TTA', np.std(all_preds, axis=0).mean())
# why did reversing time EVER seem like a good idea

def predict_with_tta(model, loader, items, n_tta=5):
    model.eval()
    all_preds = []
    for tta_i in range(n_tta):
        preds_tta = []
        # print("preds_tta", getattr(preds_tta, "shape", type(preds_tta).__name__))
        for batch in loader:
            X = batch['X'].to(device)
            if tta_i > 0:
                X = X + torch.randn_like(X) * 0.01
            time = batch['time'].to(device)
            mask = batch['mask'].to(device)
            mouse = batch['mouse'].to(device)
            pair = batch['pair'].to(device)
            reg_out, _, _, _ = model(X, time, mouse, pair, mask, use_causal=False)
            preds_tta.append(reg_out.cpu().numpy())
        all_preds.append(np.concatenate(preds_tta, axis=0))

    preds = np.mean(all_preds, axis=0)
    id_to_pred = {}
    # print("id_to_pred", getattr(id_to_pred, "shape", type(id_to_pred).__name__))
    for i, it in enumerate(items):
        for rid, val in zip(it['row_ids'], preds[i, :len(it['row_ids'])]):
            id_to_pred[rid] = float(val)
    return id_to_pred


@torch.no_grad()
def predict_dual_with_tta(model1, model2, loader, items, n_tta=5, weight=0.5):
    # print("tta???", n_tta)
    model1.eval()
    model2.eval()
    all_preds = []
    for tta_i in range(n_tta):
        preds_tta = []
        for batch in loader:
            X = batch['X'].to(device)
            if tta_i > 0:
                X = X + torch.randn_like(X) * 0.01
            time = batch['time'].to(device)
            mask = batch['mask'].to(device)
            mouse = batch['mouse'].to(device)
            pair = batch['pair'].to(device)
            reg1, _, _, _ = model1(X, time, mouse, pair, mask, use_causal=False)
            reg2, _, _, _ = model2(X, time, mouse, pair, mask, use_causal=False)
            preds_tta.append((weight * reg1 + (1 - weight) * reg2).cpu().numpy())
        all_preds.append(np.concatenate(preds_tta, axis=0))

    preds = np.mean(all_preds, axis=0)
    # print("preds", getattr(preds, "shape", type(preds).__name__))
    id_to_pred = {}
    for i, it in enumerate(items):
        for rid, val in zip(it['row_ids'], preds[i, :len(it['row_ids'])]):
            id_to_pred[rid] = float(val)
    return id_to_pred


@torch.no_grad()
def predict_standard(model, loader, items):
    model.eval()
    all_preds = []
    # print("all_preds", getattr(all_preds, "shape", type(all_preds).__name__))
    for batch in loader:
        X = batch['X'].to(device)
        time = batch['time'].to(device)
        mask = batch['mask'].to(device)
        mouse = batch['mouse'].to(device)
        pair = batch['pair'].to(device)
        reg_out, _, _, _ = model(X, time, mouse, pair, mask, use_causal=False)
        all_preds.append(reg_out.cpu().numpy())

    preds = np.concatenate(all_preds, axis=0)
    id_to_pred = {}
    for i, it in enumerate(items):
        for rid, val in zip(it['row_ids'], preds[i, :len(it['row_ids'])]):
            id_to_pred[rid] = float(val)
    return id_to_pred




In [ ]:
@torch.no_grad()
# dual average vs picking best single model
# weight = 1.0    # Transformer-only
# weight = 0.0    # LSTM-only
# weight = 0.6    # a not-quite-even blend
# print('ensemble weight', weight)
# if not (0 <= weight <= 1): raise ValueError('bad weight')
# val_mse doesn't always track test, avoid tuning blindly

def predict_dual_standard(model1, model2, loader, items, weight=0.5):
    model1.eval()
    model2.eval()
    all_preds = []
    for batch in loader:
        X = batch['X'].to(device)
        time = batch['time'].to(device)
        mask = batch['mask'].to(device)
        mouse = batch['mouse'].to(device)
        pair = batch['pair'].to(device)
        reg1, _, _, _ = model1(X, time, mouse, pair, mask, use_causal=False)
        reg2, _, _, _ = model2(X, time, mouse, pair, mask, use_causal=False)
        all_preds.append((weight * reg1 + (1 - weight) * reg2).cpu().numpy())

    preds = np.concatenate(all_preds, axis=0)
    id_to_pred = {}
    # print("id_to_pred", getattr(id_to_pred, "shape", type(id_to_pred).__name__))
    for i, it in enumerate(items):
        for rid, val in zip(it['row_ids'], preds[i, :len(it['row_ids'])]):
            id_to_pred[rid] = float(val)
    return id_to_pred


@torch.no_grad()
def predict_single_model_with_tta(model, loader, items, n_tta=5):
    model.eval()
    all_preds = []
    for tta_i in range(n_tta):
        preds_tta = []
        for batch in loader:
            X = batch['X'].to(device)
            if tta_i > 0:
                X = X + torch.randn_like(X) * 0.01
            time = batch['time'].to(device)
            mask = batch['mask'].to(device)
            mouse = batch['mouse'].to(device)
            pair = batch['pair'].to(device)
            reg_out, _, _, _ = model(X, time, mouse, pair, mask, use_causal=False)
            preds_tta.append(reg_out.cpu().numpy())
        all_preds.append(np.concatenate(preds_tta, axis=0))

    preds = np.mean(all_preds, axis=0)
    id_to_pred = {}
    # print("id_to_pred", getattr(id_to_pred, "shape", type(id_to_pred).__name__))  # sometimes NaN
    for i, it in enumerate(items):
        for rid, val in zip(it['row_ids'], preds[i, :len(it['row_ids'])]):
            id_to_pred[rid] = float(val)
    return id_to_pred


@torch.no_grad()
def predict_single_model_standard(model, loader, items):
    model.eval()
    all_preds = []
    for batch in loader:
        X = batch['X'].to(device)
        time = batch['time'].to(device)
        mask = batch['mask'].to(device)
        # print("mask valid", mask.sum().item() if hasattr(mask, "sum") else mask)  # 昔ここで落ちた
        mouse = batch['mouse'].to(device)
        pair = batch['pair'].to(device)
        reg_out, _, _, _ = model(X, time, mouse, pair, mask, use_causal=False)
        all_preds.append(reg_out.cpu().numpy())

    preds = np.concatenate(all_preds, axis=0)
    id_to_pred = {}
    for i, it in enumerate(items):
        for rid, val in zip(it['row_ids'], preds[i, :len(it['row_ids'])]):
            id_to_pred[rid] = float(val)
    return id_to_pred




In [ ]:
#
# fold loaders
#
# try pin_memory / persistent_workers only after confirming no deadlock
# train_loader = DataLoader(train_ds, batch_size=cfg.batch_size,
#                           shuffle=True, pin_memory=True, num_workers=2)
# print('first batch', next(iter(train_loader))['X'].shape)
# print('train/val lengths', len(train_items), len(val_items))
# assert set(train_indices).isdisjoint(set(val_indices))
# why does this fold have so few class 1 sequences????

def make_loaders_for_fold(train_indices, val_indices, seed):
    """
    Build train/val/test loaders for one fold.
    Preprocessing is fit transductively on ALL train_items_all + test_items.
    The fold split only determines which train items go to train vs val.
    """
    train_items = [train_items_all[i] for i in train_indices]
    # print("train_items count", len(train_items))
    val_items = [train_items_all[i] for i in val_indices]

    # FIT ON TRAIN + TEST COMBINED (transductive setting)
    all_items_combined = train_items_all + test_items
    # print("all_items_combined count", len(all_items_combined))
    mean_feat, std_feat, pca, C_feat = fit_preprocessing(
        all_items_combined, use_pca=cfg.use_pca, pca_components=cfg.pca_components, seed=seed
    )

    train_ds = LeverDataset(train_items, mean_feat, std_feat, pca,
                            tmax=cfg.tmax, with_target=True, augment=True)
    val_ds = LeverDataset(val_items, mean_feat, std_feat, pca,
                          tmax=cfg.tmax, with_target=True, augment=False)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size,
                              shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size,
                            shuffle=False, num_workers=2, pin_memory=True)
    test_ds = LeverDataset(test_items, mean_feat, std_feat, pca,
                           tmax=cfg.tmax, with_target=False, augment=False)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size,
                             shuffle=False, num_workers=2, pin_memory=True)
    return train_items, val_items, train_loader, val_loader, test_loader, mean_feat, std_feat, pca




In [ ]:
# =====
# one fold training
# =====
def train_one_fold(seed, fold_idx, train_indices, val_indices):
    """Train models for a single fold. Returns predictions and val MSE."""
    # print("passed here", fold_idx)  # 疑似ラベル部分ちょっと怖い
    seed_everything(seed + fold_idx)  # unique seed per fold
    train_items, val_items, train_loader, val_loader, test_loader, mean_feat, std_feat, pca = \
        make_loaders_for_fold(train_indices, val_indices, seed)

    run_label = f"Seed {seed} Fold {fold_idx+1}/{cfg.n_folds}"
    # print("run_label??", type(run_label), str(run_label)[:80])  # wtf

    model1 = new_transformer(pca, mean_feat)
    print(f"[{run_label}] Transformer params: {sum(p.numel() for p in model1.parameters()):,}")

    if cfg.use_dual_models:
        model2 = new_conv_lstm(pca, mean_feat)
        print(f"[{run_label}] Conv-LSTM params: {sum(p.numel() for p in model2.parameters()):,}")

    opt1 = torch.optim.AdamW(model1.parameters(), lr=cfg.lr,
                              weight_decay=cfg.weight_decay, betas=(0.9, 0.999))
    if cfg.use_dual_models:
        opt2 = torch.optim.AdamW(model2.parameters(), lr=cfg.lr,
                                  weight_decay=cfg.weight_decay, betas=(0.9, 0.999))

    total_steps = cfg.epochs * len(train_loader)
    warmup_steps = int(cfg.warmup_ratio * total_steps)
    # print("dbg warmup_steps", warmup_steps)

    def lr_lambda(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    sched1 = torch.optim.lr_scheduler.LambdaLR(opt1, lr_lambda)
    # print("sched1??", type(sched1), str(sched1)[:80])
    sched2 = None
    if cfg.use_dual_models:
        sched2 = torch.optim.lr_scheduler.LambdaLR(opt2, lr_lambda)

    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

    best_val_mse = float('inf')
    # print("best_val_mse?", best_val_mse.item() if hasattr(best_val_mse, "item") else best_val_mse)  # TODO?
    best_state1 = None
    best_state2 = None
    bad_epochs = 0
    patience = 12

    # tiny model debug runs used these (all disabled for full CV)
    # cfg.epochs = 3
    # cfg.use_pseudo = False
    # cfg.use_dual_models = False
    # torch.autograd.set_detect_anomaly(True)
    # print('params', sum(p.numel() for p in model1.parameters()))
    # print('steps/warmup', total_steps, warmup_steps)
    # print('first lr', opt1.param_groups[0]['lr'])
    # if device.type == 'cuda': print(torch.cuda.memory_allocated() / 1e9)
    # over half this notebook is waiting for training...

    for epoch in range(1, cfg.epochs + 1):
        model1.train()
        if cfg.use_dual_models:
            model2.train()

        train_tots, train_regs, train_diffs, train_diff2s, train_ces = [], [], [], [], []

        for batch in train_loader:
            X = batch['X'].to(device)
            time = batch['time'].to(device)
            mask = batch['mask'].to(device)
            # print("mask valid", mask.sum().item() if hasattr(mask, "sum") else mask)  # 昔ここで落ちた
            mouse = batch['mouse'].to(device)
            pair = batch['pair'].to(device)
            y = batch['y'].to(device)
            y_diff = batch['y_diff'].to(device)
            y_diff2 = batch['y_diff2'].to(device)

            # ---- Augmentation: Mixup or CutMix ----
            soft_targets = None
            # print("soft_targets", getattr(soft_targets, "shape", type(soft_targets).__name__))
            aug_batch = {
                'X': X, 'time': time, 'mask': mask,
                'mouse': mouse, 'pair': pair,
                'y': y, 'y_diff': y_diff,
                'y_diff2': y_diff2,
            }

            r = np.random.rand()
            if r < cfg.mixup_prob and cfg.mixup_alpha > 0:
                aug_batch, lam, perm = apply_mixup(aug_batch, alpha=cfg.mixup_alpha)
                soft_targets = make_soft_targets(aug_batch['y'], cfg.soft_ce_sigma)
            elif r < cfg.mixup_prob + cfg.cutmix_prob:
                aug_batch, lam, perm = apply_cutmix(aug_batch, cfg)
                soft_targets = make_soft_targets(aug_batch['y'], cfg.soft_ce_sigma)

            X = aug_batch['X']
            time = aug_batch['time']
            mask = aug_batch['mask']
            y = aug_batch['y']
            y_diff = aug_batch['y_diff']
            y_diff2 = aug_batch['y_diff2']

            # --- Model 1 ---
            opt1.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                reg1, diff1, diff2_1, cls1 = model1(X, time, mouse, pair, mask, use_causal=False)
                total1, l_reg1, l_diff1, l_diff2_1, l_cls1 = multitask_loss(
                    reg1, diff1, diff2_1, cls1, y, y_diff, y_diff2, mask, cfg,
                    soft_targets=soft_targets
                )

            scaler.scale(total1).backward()
            scaler.unscale_(opt1)
            torch.nn.utils.clip_grad_norm_(model1.parameters(), cfg.grad_clip)
            scaler.step(opt1)

            if cfg.use_dual_models:
                opt2.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                    reg2, diff2, diff2_2, cls2 = model2(X, time, mouse, pair, mask, use_causal=False)
                    total2, l_reg2, l_diff2, l_diff2_2, l_cls2 = multitask_loss(
                        reg2, diff2, diff2_2, cls2,
                        y, y_diff, y_diff2, mask, cfg,
                        soft_targets=soft_targets
                    )
                scaler.scale(total2).backward()
                scaler.unscale_(opt2)
                torch.nn.utils.clip_grad_norm_(model2.parameters(), cfg.grad_clip)
                scaler.step(opt2)

            scaler.update()
            sched1.step()
            if sched2:
                sched2.step()

            if cfg.use_dual_models:
                train_tots.append((total1.item() + total2.item()) / 2)
                train_regs.append((l_reg1.item() + l_reg2.item()) / 2)
                train_diffs.append((l_diff1.item() + l_diff2.item()) / 2)
                train_diff2s.append((l_diff2_1.item() + l_diff2_2.item()) / 2)
                train_ces.append((l_cls1.item() + l_cls2.item()) / 2)
            else:
                train_tots.append(total1.item())
                train_regs.append(l_reg1.item())
                train_diffs.append(l_diff1.item())
                train_diff2s.append(l_diff2_1.item())
                train_ces.append(l_cls1.item())

        avg_tot = np.mean(train_tots)
        avg_reg = np.mean(train_regs)
        avg_diff = np.mean(train_diffs)
        avg_diff2 = np.mean(train_diff2s)
        # print("avg_diff2??", type(avg_diff2), str(avg_diff2)[:80])
        avg_ce = np.mean(train_ces)

        # Validation
        if cfg.use_dual_models:
            val_mse = evaluate_dual(model1, model2, val_loader, weight=cfg.ensemble_weight)
        else:
            val_tot, val_reg, val_diff, val_diff2, val_ce = evaluate(model1, val_loader)
            val_mse = val_reg

        print(f"[{run_label}] Ep {epoch:03d} | "
              f"train: tot={avg_tot:.5f} reg={avg_reg:.6f} "
              f"diff1={avg_diff:.6f} diff2={avg_diff2:.6f} ce={avg_ce:.4f} | "
              f"val_MSE={val_mse:.6f} | best={best_val_mse:.6f}")

        if val_mse < best_val_mse - 1e-6:
            best_val_mse = val_mse
            best_state1 = {k: v.detach().cpu().clone() for k, v in model1.state_dict().items()}
            if cfg.use_dual_models:
                best_state2 = {k: v.detach().cpu().clone() for k, v in model2.state_dict().items()}
            bad_epochs = 0
            # print("dbg bad_epochs", bad_epochs)
        else:
            bad_epochs += 1
            # print("dbg bad_epochs", bad_epochs)  # wtf
            if bad_epochs >= patience:
                print(f"  Early stopping at epoch {epoch}")
                break

    model1.load_state_dict(best_state1)
    if cfg.use_dual_models:
        model2.load_state_dict(best_state2)

    # Final class-wise metrics
    print(f"\n[{run_label}] Final Validation (before pseudo):")
    if cfg.use_dual_models:
        overall, c0, c1, n0, n1 = evaluate_dual_with_class_metrics(
            model1, model2, val_loader, weight=cfg.ensemble_weight)
    else:
        overall, c0, c1, n0, n1 = evaluate_with_class_metrics(model1, val_loader)
    print(f"  Overall MSE: {overall:.6f} | C0: {c0:.6f} (n={n0}) | C1: {c1:.6f} (n={n1})")

    # ---- Multi-stage Pseudo-labeling ----
    # pseudo-label idea A: pick sequences using only prediction variance
    # std_per_seq = std_pred.mean(axis=1)
    # cutoff = np.quantile(std_per_seq, 0.25)
    # selected = std_per_seq < cutoff
    # print('pseudo selected', selected.sum(), len(selected))
    # pseudo-weight = 0.3   # this idea wasn't even implemented consistently
    # pseudo label confidence by variance + staged percentiles instead; watch leakage

    if cfg.use_pseudo:
        n_stages = cfg.pseudo_stages
        percentile_schedule = list(cfg.pseudo_percentile_schedule)
        # print("percentile_schedule??", type(percentile_schedule), str(percentile_schedule)[:80])  # 昔ここで落ちた
        while len(percentile_schedule) < n_stages:
            percentile_schedule.append(percentile_schedule[-1])

        for stage_idx in range(n_stages):
            stage_percentile = percentile_schedule[stage_idx]
            stage_lr_ratio = cfg.pseudo_lr_ratio * (cfg.pseudo_lr_decay_per_stage ** stage_idx)
            print(f"\n[{run_label}] === Pseudo Stage {stage_idx+1}/{n_stages} === "
                  f"percentile={stage_percentile}, lr_ratio={stage_lr_ratio:.5f}")

            model1.train()
            if cfg.use_dual_models:
                model2.train()

            all_predictions = []
            with torch.no_grad():
                for fwd_idx in range(cfg.n_ensemble):
                    preds_list = []
                    for batch in test_loader:
                        X = batch['X'].to(device)
                        if fwd_idx > 0:
                            X = X + torch.randn_like(X) * 0.015
                        time = batch['time'].to(device)
                        # print("time", getattr(time, "shape", type(time).__name__))
                        mask = batch['mask'].to(device)
                        mouse = batch['mouse'].to(device)
                        pair = batch['pair'].to(device)

                        reg1, _, _, _ = model1(X, time, mouse, pair, mask, use_causal=False)
                        if cfg.use_dual_models:
                            reg2, _, _, _ = model2(X, time, mouse, pair, mask, use_causal=False)
                            yhat = cfg.ensemble_weight * reg1 + (1 - cfg.ensemble_weight) * reg2
                        else:
                            yhat = reg1
                        preds_list.append(yhat.cpu().numpy())
                    all_predictions.append(np.concatenate(preds_list, axis=0))

            model1.eval()
            if cfg.use_dual_models:
                model2.eval()

            all_predictions = np.stack(all_predictions, axis=0)
            # print("all_predictions", getattr(all_predictions, "shape", type(all_predictions).__name__))
            mean_pred = all_predictions.mean(axis=0)
            std_pred = all_predictions.std(axis=0)
            # print("std_pred", getattr(std_pred, "shape", type(std_pred).__name__))  # 昔ここで落ちた

            seq_conf = []
            for si in range(len(test_items)):
                L = test_items[si]['seq_len']
                seq_conf.append((1.0 / (std_pred[si, :L] + 0.01)).mean())
            seq_conf = np.array(seq_conf)

            thr = np.percentile(seq_conf, stage_percentile)
            selected = seq_conf >= thr

            # pseudo debug - DO NOT let test targets enter this path
            # print('TTA predictions', all_predictions.shape)
            # print('uncertainties', np.quantile(uncertainty, [0, .5, .9, 1]))
            # print('selected percent', np.mean(selected))
            # print('one test item', test_items[0].keys())
            # assert all('y' not in it for it in test_items)
            # why is this XXXX???? some sequences have confidence but wild lever values

            pseudo_items = []
            for si in range(len(test_items)):
                if selected[si]:
                    it = test_items[si]
                    L = it['seq_len']
                    pseudo_y = mean_pred[si, :L].astype(np.float32)
                    pseudo_y_diff = np.zeros_like(pseudo_y)
                    # print("pseudo_y_diff??", type(pseudo_y_diff), str(pseudo_y_diff)[:80])
                    pseudo_y_diff[1:] = pseudo_y[1:] - pseudo_y[:-1]
                    pseudo_y_diff2 = np.zeros_like(pseudo_y)
                    pseudo_y_diff2[2:] = pseudo_y[2:] - 2*pseudo_y[1:-1] + pseudo_y[:-2]

                    pi = it.copy()
                    pi['y'] = pseudo_y
                    pi['y_diff'] = pseudo_y_diff
                    pi['y_diff2'] = pseudo_y_diff2
                    pi['y_bin'] = lever_to_bin(pseudo_y).astype(np.int64)
                    pi['seq_class'] = 1
                    pseudo_items.append(pi)

            print(f"[{run_label}] Pseudo stage {stage_idx+1}: {len(pseudo_items)} sequences selected "
                  f"(threshold conf={thr:.4f})")

            if len(pseudo_items) == 0:
                print(f"  No pseudo items selected, skipping stage.")
                continue

            combined = train_items + pseudo_items
            combined_ds = LeverDataset(combined, mean_feat, std_feat, pca,
                                       tmax=cfg.tmax, with_target=True, augment=True)
            combined_loader = DataLoader(combined_ds, batch_size=cfg.batch_size,
                                         shuffle=True, num_workers=2, pin_memory=True)

            opt_p1 = torch.optim.AdamW(model1.parameters(),
                                        lr=cfg.lr * stage_lr_ratio,
                                        weight_decay=cfg.weight_decay)
            opt_p2 = None
            # print("opt_p2", type(opt_p2).__name__)
            if cfg.use_dual_models:
                opt_p2 = torch.optim.AdamW(model2.parameters(),
                                            lr=cfg.lr * stage_lr_ratio,
                                            weight_decay=cfg.weight_decay)

            best_pseudo_mse = best_val_mse
            best_ps1, best_ps2 = None, None
            bad = 0

            # alternative pseudo fine-tune: freeze the backbone then unfreeze
            # for n, p in model1.named_parameters():
            #     p.requires_grad = ('head_' in n)
            # opt_p1 = torch.optim.AdamW(filter(lambda p: p.requires_grad, model1.parameters()), lr=1e-4)
            # print('trainable', sum(p.numel() for p in model1.parameters() if p.requires_grad))
            # then unfreeze all on stage 2 (never finalized this)
            # hard to tell whether pseudo helps or just follows own errors

            for ep in range(1, cfg.pseudo_epochs + 1):
                model1.train()
                if cfg.use_dual_models:
                    model2.train()

                ep_losses = []
                for batch in combined_loader:
                    X = batch['X'].to(device)
                    # print("X??", type(X), str(X)[:80])  # optional
                    time = batch['time'].to(device)
                    mask = batch['mask'].to(device)
                    mouse = batch['mouse'].to(device)
                    pair = batch['pair'].to(device)
                    y = batch['y'].to(device)
                    y_diff = batch['y_diff'].to(device)
                    y_diff2 = batch['y_diff2'].to(device)
                    # print("y_diff2??", type(y_diff2), str(y_diff2)[:80])
                    opt_p1.zero_grad(set_to_none=True)
                    with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                        r1, d1, d2_1, c1 = model1(X, time, mouse, pair, mask, use_causal=False)
                        t1, _, _, _, _ = multitask_loss(
                            r1, d1, d2_1, c1,
                            y, y_diff, y_diff2, mask, cfg
                        )
                    scaler.scale(t1).backward()
                    scaler.unscale_(opt_p1)
                    torch.nn.utils.clip_grad_norm_(model1.parameters(), cfg.grad_clip)
                    scaler.step(opt_p1)

                    if cfg.use_dual_models:
                        opt_p2.zero_grad(set_to_none=True)
                        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                            r2, d2, d2_2, c2 = model2(X, time, mouse, pair, mask, use_causal=False)
                            t2, _, _, _, _ = multitask_loss(
                                r2, d2, d2_2, c2,
                                y, y_diff, y_diff2, mask, cfg
                            )
                        scaler.scale(t2).backward()
                        scaler.unscale_(opt_p2)
                        torch.nn.utils.clip_grad_norm_(model2.parameters(), cfg.grad_clip)
                        scaler.step(opt_p2)

                    scaler.update()
                    ep_losses.append(t1.item() if not cfg.use_dual_models
                                     else (t1.item() + t2.item()) / 2)

                if cfg.use_dual_models:
                    val_mse = evaluate_dual(model1, model2, val_loader, weight=cfg.ensemble_weight)
                    # print("val_mse?", val_mse.item() if hasattr(val_mse, "item") else val_mse)
                else:
                    _, val_mse, _, _, _ = evaluate(model1, val_loader)

                print(f"[{run_label}] Pseudo S{stage_idx+1} Ep {ep:02d} | "
                      f"train={np.mean(ep_losses):.5f} | "
                      f"val_MSE={val_mse:.6f} | best={best_pseudo_mse:.6f}")

                if val_mse < best_pseudo_mse - 1e-6:
                    best_pseudo_mse = val_mse
                    best_ps1 = {k: v.detach().cpu().clone() for k, v in model1.state_dict().items()}
                    if cfg.use_dual_models:
                        best_ps2 = {k: v.detach().cpu().clone() for k, v in model2.state_dict().items()}
                    bad = 0
                else:
                    bad += 1
                    # print("bad??", type(bad), str(bad)[:80])
                    if bad >= cfg.pseudo_patience:
                        break

            if best_ps1 is not None and best_pseudo_mse < best_val_mse:
                model1.load_state_dict(best_ps1)
                if cfg.use_dual_models and best_ps2:
                    model2.load_state_dict(best_ps2)
                best_val_mse = best_pseudo_mse
                best_state1 = best_ps1
                if cfg.use_dual_models and best_ps2:
                    best_state2 = best_ps2
                print(f"  Pseudo stage {stage_idx+1} improved MSE to {best_val_mse:.6f}")
            else:
                model1.load_state_dict(best_state1)
                if cfg.use_dual_models:
                    model2.load_state_dict(best_state2)
                print(f"  Pseudo stage {stage_idx+1} did NOT improve, reverting.")

        # Final class-wise metrics after all pseudo stages
        print(f"\n[{run_label}] Final Validation (after all pseudo stages):")
        if cfg.use_dual_models:
            overall, c0, c1, n0, n1 = evaluate_dual_with_class_metrics(
                model1, model2, val_loader, weight=cfg.ensemble_weight)
        else:
            overall, c0, c1, n0, n1 = evaluate_with_class_metrics(model1, val_loader)
        print(f"  Overall MSE: {overall:.6f} | C0: {c0:.6f} (n={n0}) | C1: {c1:.6f} (n={n1})")

    # use reg output for inference, NOT bin head
    if cfg.use_tta:
        if cfg.use_dual_models:
            id_to_pred = predict_dual_with_tta(model1, model2, test_loader, test_items,
                                                n_tta=5, weight=cfg.ensemble_weight)
        else:
            id_to_pred = predict_with_tta(model1, test_loader, test_items, n_tta=5)
    else:
        if cfg.use_dual_models:
            id_to_pred = predict_dual_standard(model1, model2, test_loader, test_items,
                                                weight=cfg.ensemble_weight)
        else:
            id_to_pred = predict_standard(model1, test_loader, test_items)

    # Per-model predictions
    if cfg.use_tta:
        id_to_pred_m1 = predict_single_model_with_tta(model1, test_loader, test_items, n_tta=5)
        if cfg.use_dual_models:
            id_to_pred_m2 = predict_single_model_with_tta(model2, test_loader, test_items, n_tta=5)
        else:
            id_to_pred_m2 = None
    else:
        id_to_pred_m1 = predict_single_model_standard(model1, test_loader, test_items)
        if cfg.use_dual_models:
            id_to_pred_m2 = predict_single_model_standard(model2, test_loader, test_items)
            # print("id_to_pred_m2", getattr(id_to_pred_m2, "shape", type(id_to_pred_m2).__name__))
        else:
            id_to_pred_m2 = None

    # ---- Separate OOF predictions for Ridge meta-learner ----
    # Collect Transformer (model1) and LSTM (model2) val predictions SEPARATELY
    tf_val_id_to_pred = {}
    # print("tf_val_id_to_pred", getattr(tf_val_id_to_pred, "shape", type(tf_val_id_to_pred).__name__))
    lstm_val_id_to_pred = {}

    model1.eval()
    all_val_preds_m1 = []
    all_val_preds_m2 = []
    # print("all_val_preds_m2", getattr(all_val_preds_m2, "shape", type(all_val_preds_m2).__name__))
    for batch in val_loader:
        X = batch['X'].to(device)
        time_t = batch['time'].to(device)
        mask = batch['mask'].to(device)
        # print("mask valid", mask.sum().item() if hasattr(mask, "sum") else mask)  # TODO?
        mouse = batch['mouse'].to(device)
        # print("mouse??", type(mouse), str(mouse)[:80])
        pair = batch['pair'].to(device)
        with torch.no_grad():
            reg1, _, _, _ = model1(X, time_t, mouse, pair, mask, use_causal=False)
        all_val_preds_m1.append(reg1.cpu().numpy())
        if cfg.use_dual_models:
            with torch.no_grad():
                reg2, _, _, _ = model2(X, time_t, mouse, pair, mask, use_causal=False)
            all_val_preds_m2.append(reg2.cpu().numpy())

    all_val_preds_m1 = np.concatenate(all_val_preds_m1, axis=0)
    for i, it in enumerate(val_items):
        for rid, val in zip(it['row_ids'], all_val_preds_m1[i, :len(it['row_ids'])]):
            tf_val_id_to_pred[rid] = float(val)

    if cfg.use_dual_models:
        all_val_preds_m2 = np.concatenate(all_val_preds_m2, axis=0)
        for i, it in enumerate(val_items):
            for rid, val in zip(it['row_ids'], all_val_preds_m2[i, :len(it['row_ids'])]):
                lstm_val_id_to_pred[rid] = float(val)

    # optional checkpointing for Colab restarts (not part of final run)
    # os.makedirs('/content/checkpoints', exist_ok=True)
    # torch.save({'seed': seed, 'fold': fold_idx,
    #             'val_mse': best_val_mse, 'model1': model1.state_dict()},
    #            f'/content/checkpoints/fold_{seed}_{fold_idx}.pt')
    # if cfg.use_dual_models:
    #     torch.save(model2.state_dict(), 'last_lstm.pt')
    # print('GPU allocated', torch.cuda.memory_allocated() / 1024**3)
    # didn't want huge pt files for every fold

    # Free GPU memory
    del model1
    if cfg.use_dual_models:
        del model2
    gc.collect()
    torch.cuda.empty_cache()

    return id_to_pred, id_to_pred_m1, id_to_pred_m2, best_val_mse, tf_val_id_to_pred, lstm_val_id_to_pred




In [ ]:
# ------------------
# loop over the seeds/folds (slow)
# ------------------
# print(sample_sub.iloc[:3])  # idの順番確認
sample_sub = pd.read_csv(f"{cfg.data_path}/sample_submission.csv")
ids = sample_sub['id'].values
# print("ids count", len(ids))

all_run_preds = []       # list of id_to_pred dicts (one per split_seed×fold×nn_seed)
all_run_preds_m1 = []
all_run_preds_m2 = []
# print("all_run_preds_m2", getattr(all_run_preds_m2, "shape", type(all_run_preds_m2).__name__))
all_run_vals = []        # val MSE per run
all_run_labels = []      # human-readable labels

# NN seed平均をridgeに入れる方
all_tf_val_preds = []    # Transformer OOF val predictions per fold (averaged over nn_seeds)
all_lstm_val_preds = []  # LSTM OOF val predictions per fold (averaged over nn_seeds)
all_tf_test_preds = []   # Transformer test predictions per fold (averaged over nn_seeds)
all_lstm_test_preds = [] # LSTM test predictions per fold (averaged over nn_seeds)
# print("all_lstm_test_preds", getattr(all_lstm_test_preds, "shape", type(all_lstm_test_preds).__name__))
all_lgbm_val_preds = []  # LGBM OOF val predictions per fold
# print("all_lgbm_val_preds", getattr(all_lgbm_val_preds, "shape", type(all_lgbm_val_preds).__name__))
all_lgbm_test_preds = [] # LGBM test predictions per fold

# For detailed test CSV and incremental debug CSV
detailed_test_df = pd.DataFrame({'id': ids})
incremental_debug_df = pd.DataFrame({'id': ids})

base_seed = 42
run_count = 0
total_runs = cfg.seed_count * cfg.n_folds * cfg.nn_seed_count
# print("dbg total_runs", total_runs)

# Colab落ちないで... このループ結構長い
# cfg.seed_count = 1  # debug only; set this BEFORE starting if needed
# local quick test, never enable with the full export code below
# for seed_idx in range(1):
#     splits = get_kfold_splits(len(train_items_all), n_folds=2, seed=42)
#     tr, va = splits[0]
#     result = train_one_fold(42, 0, tr, va)
#     print('one fold result types', [type(x) for x in result])
#     break
# elapsed = time.time() - t0
# print('secs', elapsed)
# don't accidentally count nn_seeds twice again

for seed_idx in range(cfg.seed_count):
    split_seed = base_seed + seed_idx * 100

    # Generate fold splits for this split_seed
    fold_splits = get_kfold_splits(len(train_items_all), n_folds=cfg.n_folds, seed=split_seed)

    for fold_idx, (train_indices, val_indices) in enumerate(fold_splits):

        # --- Multiple NN seeds for this fold (same split, different weight init) ---
        fold_tf_val_preds = []   # list of tf_val_id_to_pred dicts, one per nn_seed
        fold_lstm_val_preds = [] # list of lstm_val_id_to_pred dicts, one per nn_seed
        fold_m1_test_preds = []  # list of id_to_pred_m1 dicts
        fold_m2_test_preds = []  # list of id_to_pred_m2 dicts
        fold_ens_test_preds = [] # list of id_to_pred (TF+LSTM ens) dicts
        fold_nn_vals = []

        for nn_seed_i in range(cfg.nn_seed_count):
            nn_seed = split_seed + nn_seed_i * 7  # different init seed, same fold
            # print("dbg nn_seed", nn_seed)
            run_count += 1
            run_label = f"s{split_seed}_f{fold_idx}_nn{nn_seed_i}"

            print("\n" + "="*60)
            print(f"RUN {run_count}/{total_runs} | split_seed={split_seed}, "
                  f"fold={fold_idx+1}/{cfg.n_folds}, nn_seed={nn_seed_i+1}/{cfg.nn_seed_count}")
            print(f"  Train: {len(train_indices)} sequences, Val: {len(val_indices)} sequences")
            print("="*60)

            id_to_pred, id_to_pred_m1, id_to_pred_m2, best_val, tf_val_id_to_pred, lstm_val_id_to_pred = \
                train_one_fold(nn_seed, fold_idx, train_indices, val_indices)

            all_run_preds.append(id_to_pred)
            all_run_preds_m1.append(id_to_pred_m1)
            all_run_preds_m2.append(id_to_pred_m2)
            all_run_vals.append(best_val)
            all_run_labels.append(run_label)

            fold_tf_val_preds.append(tf_val_id_to_pred)
            fold_lstm_val_preds.append(lstm_val_id_to_pred)
            fold_m1_test_preds.append(id_to_pred_m1)
            fold_m2_test_preds.append(id_to_pred_m2)
            fold_ens_test_preds.append(id_to_pred)
            fold_nn_vals.append(best_val)

            # --- Detailed test CSV: add per-run columns ---
            detailed_test_df[f"lever_{run_label}"] = [id_to_pred[rid] for rid in ids]
            detailed_test_df[f"lever_{run_label}_model1"] = [id_to_pred_m1[rid] for rid in ids]
            if id_to_pred_m2 is not None:
                detailed_test_df[f"lever_{run_label}_model2"] = [id_to_pred_m2[rid] for rid in ids]

            # --- Incremental debug CSV ---
            incremental_debug_df[f"lever_{run_label}"] = [id_to_pred[rid] for rid in ids]
            incremental_debug_df[f"lever_{run_label}_model1"] = [id_to_pred_m1[rid] for rid in ids]
            if id_to_pred_m2 is not None:
                incremental_debug_df[f"lever_{run_label}_model2"] = [id_to_pred_m2[rid] for rid in ids]

            # Save incrementally
            incremental_debug_df.to_csv("debug_incremental_predictions.csv", index=False)
            print(f"  [Incremental debug CSV updated: {run_count} runs completed]")

        # --- Average NN predictions across nn_seeds for this fold ---
        print(f"\n  [Fold {fold_idx+1}] Averaging {cfg.nn_seed_count} NN seeds "
              f"(val MSEs: {', '.join(f'{v:.6f}' for v in fold_nn_vals)})")

        # Average TF OOF val predictions
        avg_tf_val = {}
        for rid in fold_tf_val_preds[0]:
            avg_tf_val[rid] = float(np.mean([d[rid] for d in fold_tf_val_preds]))
        all_tf_val_preds.append(avg_tf_val)

        # Average LSTM OOF val predictions
        avg_lstm_val = {}
        if fold_lstm_val_preds[0]:  # non-empty
            for rid in fold_lstm_val_preds[0]:
                avg_lstm_val[rid] = float(np.mean([d[rid] for d in fold_lstm_val_preds if d]))
        all_lstm_val_preds.append(avg_lstm_val)

        # Average TF test predictions
        avg_tf_test = {}
        # print("avg_tf_test", getattr(avg_tf_test, "shape", type(avg_tf_test).__name__))
        for rid in ids:
            avg_tf_test[rid] = float(np.mean([d[rid] for d in fold_m1_test_preds]))
        all_tf_test_preds.append(avg_tf_test)

        # Average LSTM test predictions
        avg_lstm_test = {}
        if fold_m2_test_preds[0] is not None:
            for rid in ids:
                avg_lstm_test[rid] = float(np.mean([d[rid] for d in fold_m2_test_preds if d is not None]))
        all_lstm_test_preds.append(avg_lstm_test)

        # LGB is once per fold, not inside that nn seed loop!!!
        # LGB branch trained once per fold, not once per NN initialization
        # print('lgb dataset sizes', len(train_indices), len(val_indices))
        # print('lgbm fold', seed_idx, fold_idx)
        # assert not any('y' not in train_items_all[i] for i in train_indices)
        # print('lgb n models expected', cfg.seed_count * cfg.n_folds)
        # long loop -> this helps track where Colab died

        if cfg.use_lgbm:
            train_items_fold = [train_items_all[i] for i in train_indices]
            val_items_fold = [train_items_all[i] for i in val_indices]

            lgbm_val_preds, lgbm_test_preds, lgbm_val_mse = train_lgbm_fold(
                train_items_fold, val_items_fold, test_items, seed=split_seed + fold_idx * 1000
            )
            all_lgbm_val_preds.append(lgbm_val_preds)
            all_lgbm_test_preds.append(lgbm_test_preds)
            print(f"  [LGBM] Fold {fold_idx+1} val MSE: {lgbm_val_mse:.6f}")



In [ ]:
print("\n" + "="*60)
print("Per-run best val MSE:")
for label, v in zip(all_run_labels, all_run_vals):
    print(f"  {label}: {v:.6f}")
print(f"Mean val MSE (all runs): {np.mean(all_run_vals):.6f}")

# Per split-seed mean val MSE
# Colab落ちないで... このループ結構長い
# cfg.seed_count = 1  # debug only; set this BEFORE starting if needed
# quick aggregation sanity checks (off by default)
# print('expected runs', cfg.seed_count * cfg.n_folds * cfg.nn_seed_count)
# print('actual runs', len(all_run_vals), len(all_run_preds))
# print('labels', all_run_labels[:4])
# print('vals', np.array(all_run_vals)[:4])
# assert len(all_run_vals) == len(all_run_preds)
# シード平均のインデックス間違えたら全部終わる

for seed_idx in range(cfg.seed_count):
    split_seed = base_seed + seed_idx * 100
    # print("dbg split_seed", split_seed)
    start = seed_idx * cfg.n_folds * cfg.nn_seed_count
    end = start + cfg.n_folds * cfg.nn_seed_count
    seed_vals = all_run_vals[start:end]
    # print("seed_vals", getattr(seed_vals, "shape", type(seed_vals).__name__))
    print(f"  Split seed {split_seed} mean val MSE: {np.mean(seed_vals):.6f}")

# Save detailed test predictions CSV
detailed_test_df.to_csv("test_detailed_predictions.csv", index=False)
print(f"\nSaved: test_detailed_predictions.csv ({detailed_test_df.shape[1]} columns)")




In [ ]:
#
# ridge on TF / LSTM / LGBM
#
if cfg.use_lgbm:
    print("\n" + "="*60)
    print("RIDGE META-LEARNER: combining Transformer + LSTM + LightGBM")
    print("="*60)

    # Build OOF (out-of-fold) predictions for ALL train rows
    # For each fold, we have TF, LSTM, and LGBM val predictions separately

    # Collect all train row IDs and their true lever values
    all_train_row_ids = []
    all_train_y = []
    for it in train_items_all:
        all_train_row_ids.extend(it['row_ids'])
        all_train_y.extend(it['y'].tolist())
    all_train_row_ids = np.array(all_train_row_ids)
    # print("all_train_row_ids count", len(all_train_row_ids))
    all_train_y = np.array(all_train_y, dtype=np.float32)

    # Build OOF arrays
    oof_tf = np.full(len(all_train_row_ids), np.nan, dtype=np.float64)
    oof_lstm = np.full(len(all_train_row_ids), np.nan, dtype=np.float64)
    oof_lgbm = np.full(len(all_train_row_ids), np.nan, dtype=np.float64)

    rid_to_idx = {rid: i for i, rid in enumerate(all_train_row_ids)}

    for fold_idx in range(len(all_tf_val_preds)):
        # Transformer OOF
        for rid, pred in all_tf_val_preds[fold_idx].items():
            if rid in rid_to_idx:
                oof_tf[rid_to_idx[rid]] = pred
        # LSTM OOF
        if all_lstm_val_preds[fold_idx]:  # non-empty dict
            for rid, pred in all_lstm_val_preds[fold_idx].items():
                if rid in rid_to_idx:
                    oof_lstm[rid_to_idx[rid]] = pred
        # LGBM OOF
        for rid, pred in all_lgbm_val_preds[fold_idx].items():
            if rid in rid_to_idx:
                oof_lgbm[rid_to_idx[rid]] = pred

    # Check coverage
    tf_coverage = np.sum(~np.isnan(oof_tf)) / len(oof_tf)
    lstm_coverage = np.sum(~np.isnan(oof_lstm)) / len(oof_lstm)
    lgbm_coverage = np.sum(~np.isnan(oof_lgbm)) / len(oof_lgbm)
    print(f"  OOF coverage: TF={tf_coverage:.4f}, LSTM={lstm_coverage:.4f}, LGBM={lgbm_coverage:.4f}")

    # Determine which columns to use based on availability
    if cfg.use_dual_models and lstm_coverage > 0.9:
        # 3-column Ridge: TF, LSTM, LGBM
        valid_mask = ~np.isnan(oof_tf) & ~np.isnan(oof_lstm) & ~np.isnan(oof_lgbm)
        X_meta_train = np.column_stack([oof_tf[valid_mask], oof_lstm[valid_mask], oof_lgbm[valid_mask]])
        col_names = ['Transformer', 'LSTM', 'LightGBM']
    else:
        # 2-column Ridge: TF, LGBM (no LSTM)
        valid_mask = ~np.isnan(oof_tf) & ~np.isnan(oof_lgbm)
        X_meta_train = np.column_stack([oof_tf[valid_mask], oof_lgbm[valid_mask]])
        col_names = ['Transformer', 'LightGBM']

    y_meta_train = all_train_y[valid_mask]
    print(f"  Meta-learner training samples: {X_meta_train.shape[0]}, features: {col_names}")

    # Fit Ridge
    ridge = Ridge(alpha=cfg.ridge_alpha, fit_intercept=True)
    ridge.fit(X_meta_train, y_meta_train)

    oof_meta_pred = ridge.predict(X_meta_train)
    # ridge coefficients and meta signal debugging
    # print('Ridge coefs', ridge.coef_, 'intercept', ridge.intercept_)
    # print('X_meta_train', X_meta_train.shape, np.isnan(X_meta_train).sum())
    # print('OOF target range', y_meta_train.min(), y_meta_train.max())
    # print('OOF pred range', oof_meta_pred.min(), oof_meta_pred.max())
    # print('in-sample meta scores optimistic? ridge trained on same OOF array')

    oof_meta_mse = np.mean((oof_meta_pred - y_meta_train) ** 2)
    oof_tf_mse = np.mean((oof_tf[valid_mask] - y_meta_train) ** 2)
    # print("oof_tf_mse?", oof_tf_mse.item() if hasattr(oof_tf_mse, "item") else oof_tf_mse)
    oof_lgbm_mse = np.mean((oof_lgbm[valid_mask] - y_meta_train) ** 2)

    print(f"  OOF MSE: TF={oof_tf_mse:.6f}", end="")
    if cfg.use_dual_models and lstm_coverage > 0.9:
        oof_lstm_mse = np.mean((oof_lstm[valid_mask] - y_meta_train) ** 2)
        # print("oof_lstm_mse?", oof_lstm_mse.item() if hasattr(oof_lstm_mse, "item") else oof_lstm_mse)
        print(f", LSTM={oof_lstm_mse:.6f}", end="")
    print(f", LGBM={oof_lgbm_mse:.6f}, Ridge_Meta={oof_meta_mse:.6f}")

    coef_str = ", ".join(f"{name}={c:.4f}" for name, c in zip(col_names, ridge.coef_))
    print(f"  Ridge coefficients: {coef_str}, intercept={ridge.intercept_:.4f}")

    # Build test predictions for meta-learner
    # all_tf_test_preds / all_lstm_test_preds are already averaged across nn_seeds per fold
    # Average across folds (and split_seeds)
    tf_test_avg = {}
    for rid in ids:
        tf_test_avg[rid] = float(np.mean([d[rid] for d in all_tf_test_preds]))

    # Build meta test features
    if cfg.use_dual_models and lstm_coverage > 0.9:
        lstm_test_avg = {}
        for rid in ids:
            lstm_test_avg[rid] = float(np.mean([d[rid] for d in all_lstm_test_preds if d]))

        lgbm_test_avg = {}
        for rid in ids:
            lgbm_test_avg[rid] = float(np.mean([d[rid] for d in all_lgbm_test_preds]))

        X_meta_test = np.column_stack([
            np.array([tf_test_avg[rid] for rid in ids], dtype=np.float64),
            np.array([lstm_test_avg[rid] for rid in ids], dtype=np.float64),
            np.array([lgbm_test_avg[rid] for rid in ids], dtype=np.float64),
        ])
    else:
        lgbm_test_avg = {}
        for rid in ids:
            lgbm_test_avg[rid] = float(np.mean([d[rid] for d in all_lgbm_test_preds]))

        X_meta_test = np.column_stack([
            np.array([tf_test_avg[rid] for rid in ids], dtype=np.float64),
            np.array([lgbm_test_avg[rid] for rid in ids], dtype=np.float64),
        ])

    meta_test_pred = ridge.predict(X_meta_test)

    # Build final submission with Ridge meta-learner
    # experimented with a hand-weighted blend before the ridge fit
    # w_tf, w_lstm, w_lgb = 0.45, 0.20, 0.35
    # test_hand = w_tf * np.array([tf_test_avg[r] for r in ids])
    # test_hand += w_lstm * np.array([lstm_test_avg[r] for r in ids])
    # test_hand += w_lgb * np.array([lgbm_test_avg[r] for r in ids])
    # hand_sub = sample_sub.copy()
    # hand_sub['lever'] = test_hand
    # hand_sub.to_csv('submission_handblend.csv', index=False)
    # ridge is less arbitrary; keep that

    sub_meta = sample_sub.copy()
    sub_meta['lever'] = meta_test_pred.astype(np.float32)
    sub_meta.to_csv("submission_ridge_meta.csv", index=False)
    print(f"Saved: submission_ridge_meta.csv")

    # Also save individual model submissions for comparison
    sub_tf = sample_sub.copy()
    sub_tf['lever'] = np.array([tf_test_avg[rid] for rid in ids], dtype=np.float32)
    sub_tf.to_csv("submission_tf_only.csv", index=False)
    print(f"Saved: submission_tf_only.csv")

    if cfg.use_dual_models and lstm_coverage > 0.9:
        sub_lstm = sample_sub.copy()
        sub_lstm['lever'] = np.array([lstm_test_avg[rid] for rid in ids], dtype=np.float32)
        sub_lstm.to_csv("submission_lstm_only.csv", index=False)
        print(f"Saved: submission_lstm_only.csv")

    sub_lgbm = sample_sub.copy()
    # print("sub_lgbm??", type(sub_lgbm), str(sub_lgbm)[:80])
    sub_lgbm['lever'] = np.array([lgbm_test_avg[rid] for rid in ids], dtype=np.float32)
    sub_lgbm.to_csv("submission_lgbm_only.csv", index=False)
    print(f"Saved: submission_lgbm_only.csv")

    # Use Ridge meta as primary submission
    sub = sub_meta.copy()
    sub.to_csv("submission_multitask_v3.csv", index=False)

else:
    # No LGBM: just use NN ensemble as before
    # Drop worst runs
    n_total_runs = len(all_run_vals)
    # print("dbg n_total_runs", n_total_runs)  # TODO?
    worstk = int(n_total_runs * cfg.drop_worst_ratio)
    # print("worstk??", type(worstk), str(worstk)[:80])
    worst_idxs = np.argsort(all_run_vals)[-worstk:] if worstk > 0 else []
    worst_set = set(worst_idxs)
    kept_preds = [p for i, p in enumerate(all_run_preds) if i not in worst_set]

    bagged = {}
    # print("bagged??", type(bagged), str(bagged)[:80])
    for rid in ids:
        bagged[rid] = float(np.mean([d[rid] for d in kept_preds]))

    sub = sample_sub.copy()
    # print("sub??", type(sub), str(sub)[:80])  # sometimes NaN
    sub['lever'] = sub['id'].map(bagged).astype(np.float32)
    sub.to_csv("submission_multitask_v3.csv", index=False)

# Seed uncertainty


In [ ]:
pred_matrix = np.zeros((len(all_run_preds), len(ids)), dtype=np.float32)
# print("pred_matrix", getattr(pred_matrix, "shape", type(pred_matrix).__name__))
for s_idx, sp in enumerate(all_run_preds):
    pred_matrix[s_idx] = np.array([sp[rid] for rid in ids], dtype=np.float32)
row_std = pred_matrix.std(axis=0)
# print("row_std", getattr(row_std, "shape", type(row_std).__name__))
print(f"\nRow-wise STD: Q1={np.percentile(row_std,25):.6f} Med={np.percentile(row_std,50):.6f} "
      f"Q3={np.percentile(row_std,75):.6f} Mean={row_std.mean():.6f}")

# high-disagreement sequences from seed-bagging
# idx = np.argsort(-row_std)[:25]
# print('most unstable ids', ids[idx])
# print('std quantiles', np.quantile(row_std, [.01, .1, .5, .9, .99]))
# debug = pd.DataFrame(pred_matrix[:, idx].T, index=ids[idx])
# display(debug.head())
# debug.to_csv('seed_disagreement_top25.csv')
# if np.isnan(row_std).any(): raise RuntimeError('nan in ensemble')
# いちいちこれ全部printするとColabが重いのでオフ

std_df = pd.DataFrame({"id": ids, "std_across_runs": row_std})
std_df.to_csv("seed_prediction_std_per_row.csv", index=False)

print(f"\nSaved: submission_multitask_v3.csv")
print(f"Saved: test_detailed_predictions.csv")
print(f"Saved: debug_incremental_predictions.csv")
print(f"Multi-task v3 (5-fold CV + LightGBM + Ridge Meta): reg(w={cfg.w_regression}) + diff(w={cfg.w_diff}) + soft_cls(w={cfg.w_classification}, σ={cfg.soft_ce_sigma})")
print(f"Augmentation: mixup(α={cfg.mixup_alpha}, p={cfg.mixup_prob}) + cutmix(p={cfg.cutmix_prob})")
print(f"Pseudo-labeling: {cfg.pseudo_stages} stages, percentiles={cfg.pseudo_percentile_schedule}, lr_decay={cfg.pseudo_lr_decay_per_stage}")
print(f"CV: {cfg.n_folds}-fold × {cfg.seed_count} split_seeds × {cfg.nn_seed_count} nn_seeds = {cfg.seed_count * cfg.n_folds * cfg.nn_seed_count} total NN runs")
print(f"LightGBM: {cfg.lgbm_n_seeds} seeds per fold, Ridge alpha={cfg.ridge_alpha}")
print(f"Early stopping on val regression MSE. Inference uses regression head only.")